# *__Working on BTCUSD predictions with GRU model.__*

## __Check first before starting__

In [1]:
import os

# Change the working directory to the project root
Working_directory = os.path.normpath("C:/Users/james/OneDrive/文件/Continual_Learning")
os.chdir(Working_directory)
print(f"Working directory: {os.getcwd()}")  # Prints the current working directory

Working directory: C:\Users\james\OneDrive\文件\Continual_Learning


## **__All imports__**

In [2]:
import ipywidgets as widgets
from IPython.display import display
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from mpl_interactions import zoom_factory, panhandler
from sklearn.model_selection import train_test_split
import pickle
from ta import trend, momentum, volatility, volume
import math
from scipy.ndimage import gaussian_filter1d
from typing import Callable, Tuple
import shutil
import contextlib
import traceback
import gc
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

## __**All functions (For data processing)**__

In [3]:
def ensure_folder(folder_path: str) -> None:
    """Ensure the given folder exists, create it if not."""
    os.makedirs(folder_path, exist_ok=True)

def plot_with_matplotlib(data: pd.DataFrame, 
                         title: str, 
                         interactive: bool = False, 
                         save_path: str = None, 
                         show_plot: bool = True, 
                         save_matplotlib_object: str = None) -> None:
    """
    Plot time-series data using Matplotlib with optional trend-based coloring.

    Args:
        - data (pd.DataFrame): Data containing a 'close' column (required).
        - title (str): Plot title.
        - interactive (bool): Enable zoom & pan if True.
        - save_path (str, optional): Path to save the figure.
        - show_plot (bool): Whether to display the plot.
        - save_matplotlib_object (str, optional): Path to save the Matplotlib object.

    Returns:
        - None: Displays or saves the plot as specified.
    """
    # Check if 'close' column exists
    if 'close' not in data.columns:
        raise ValueError("DataFrame must contain a 'close' column.")

    # Set default color from Matplotlib cycle
    default_blue = plt.rcParams['axes.prop_cycle'].by_key()['color'][0]
    
    # Define colors for different trends
    trend_colors = {
        0: 'black',
        1: 'yellow',
        2: 'red',
        3: 'green',
        4: default_blue
    }

    # Create figure and axis for plotting
    fig, ax = plt.subplots(figsize=(12, 6))

    # Plot with trend-based coloring if 'trend' column exists
    if 'trend' in data.columns:
        legend_added = set()
        prev_idx = data.index[0]
        for idx, row in data.iterrows():
            if idx != prev_idx:
                trend_key = int(row['trend'])
                label = f'Trend {trend_key}' if trend_key not in legend_added else None
                ax.plot([prev_idx, idx], 
                        [data.loc[prev_idx, 'close'], row['close']],
                        color=trend_colors[trend_key], 
                        linestyle='-', 
                        linewidth=1,
                        label=label)
                legend_added.add(trend_key)
            prev_idx = idx
        ax.set_title(f"{title} (Connected, Colored by Trend)")
    else:
        # Plot default line if no 'trend' column
        ax.plot(data.index, data['close'], label='Closing Price', linestyle='-', marker='o', 
                markersize=2, linewidth=1, color=default_blue, markerfacecolor='green', markeredgecolor='black')
        ax.set_title(title)
    
    # Set axis labels and add legend/grid
    ax.set_xlabel('Date')
    ax.set_ylabel('Closing Price (USD)')
    ax.legend()
    ax.grid()
    
    # Enable interactive features if requested
    if interactive:
        zoom_factory(ax)
        panhandler(fig)

    # Save the plot if a path is provided
    if save_path:
        fig.tight_layout()
        fig.savefig(save_path, dpi=300, bbox_inches='tight')

    # Save the Matplotlib object if requested
    if save_matplotlib_object:
        with open(save_matplotlib_object, 'wb') as f:
            pickle.dump(fig, f)

    # Display the plot if requested
    if show_plot:
        plt.show()

def load_and_show_pickle(pickle_file_path: str):
    """
    Load a pickled Matplotlib figure object and display it.

    Args:
        - pickle_file_path (str): Path to the pickled Matplotlib figure file.

    Returns:
        - None: Displays the loaded figure.
    """
    # Load and display the pickled figure
    try:
        with open(pickle_file_path, "rb") as f:
            loaded_fig = pickle.load(f)

        print(f"Figure successfully loaded and displayed from: {pickle_file_path}")
        plt.show(block=True)

    except FileNotFoundError:
        print(f"Error: File not found at {pickle_file_path}.")
    except Exception as e:
        print(f"Error loading the pickled figure: {e}")

def save_to_csv(df: pd.DataFrame, file_path: str) -> None:
    """
    Save DataFrame to CSV.
    """
    df.to_csv(file_path)
    print(f"\nSuccessfully saved data with moving average to CSV: \n\t{file_path}\n")

def read_csv_file(file_path: str, preview_rows: int = 5, 
                  days_towards_end: int = None, 
                  days_from_start: int = None, description: str = ""):
    """
    Reads a CSV file and returns a pandas DataFrame filtered by date range.

    Args:
        - file_path (str): The path to the CSV file.
        - preview_rows (int): Number of rows to preview (default is 5).
        - days_towards_end (int, optional): Number of days from the most recent date.
        - days_from_start (int, optional): Number of days from the oldest date of filtered data.
        - description (str): A brief description of the dataset.
                           Explanation:
                           - To retrieve data from the **end**: Use `days_towards_end`.
                           - To retrieve data from the **start of the filtered range**: Use `days_from_start`.
                           - To retrieve data from the **middle**: Use both:
                             For example, if `days_towards_end=100` and `days_from_start=50`,
                             the function will first filter the last 100 days of the dataset,
                             and then filter the first 50 days from this range.
                             This results in data between the last 100th and the last 50th day.

    Returns:
        - pd.DataFrame: The loaded and filtered data from the CSV file.
    """
    try:
        if description:
            print(f"\nDescription: {description}")
        print(f"\nFile path: {file_path}")
        
        # Read the CSV file
        data = pd.read_csv(file_path, parse_dates=['date'], index_col='date')
        
        # Filter by days towards the end
        if days_towards_end is not None:
            # Get the most recent date in the dataset
            last_date = data.index.max()
            end_cutoff_date = last_date - pd.Timedelta(days=days_towards_end)
            data = data[data.index >= end_cutoff_date]
            print(f"\nRetrieving data from the past {days_towards_end} days (from {end_cutoff_date.date()} onwards):")
        
        # Filter by days from the start (from the filtered data)
        if days_from_start is not None:
            # Get the earliest date in the filtered dataset
            first_date = data.index.min()
            start_cutoff_date = first_date + pd.Timedelta(days=days_from_start)
            data = data[data.index <= start_cutoff_date]
            print(f"\nRetrieving the first {days_from_start} days from the filtered data (up to {start_cutoff_date.date()}):")

        if preview_rows:
            # Print a preview of the data
            print(f"\nPreview of the first {preview_rows} rows:")
            display(data.head(preview_rows))
            print()

            print(f"\nPreview of the last {preview_rows} rows:")
            display(data.tail(preview_rows))
            print()

        return data
    
    except FileNotFoundError:
        print("Error: File not found.")
    except pd.errors.EmptyDataError:
        print("Error: The file is empty.")
    except pd.errors.ParserError:
        print("Error: File parsing failed.")
    except Exception as e:
        print(f"Unexpected error: {e}")

def downsample_minute_data(data: pd.DataFrame, n: int) -> pd.DataFrame:
    """
    Downsample minute data into N-minute intervals by retaining every Nth row.

    Args:
        - data (pd.DataFrame): The original DataFrame with a datetime index.
        - n (int): The number of minutes for the downsampling interval.

    Returns:
        - pd.DataFrame: Downsampled DataFrame.
    """
    print("\n========---> Downsampling the data! \n")
    data = data.copy()

    # Ensure index is a DatetimeIndex
    if not isinstance(data.index, pd.DatetimeIndex):
        try:
            data.index = pd.to_datetime(data.index)
        except Exception as e:
            raise ValueError("DataFrame index conversion to DatetimeIndex failed.") from e

    # Downsample by selecting rows where minute % N == 0
    return data[data.index.minute % n == 0]

def calculate_log_returns_all_columns(data: pd.DataFrame, exclude_columns: list = [], dropna: bool = True) -> pd.DataFrame:
    """
    Calculate log returns for all numeric columns in a pandas DataFrame,
    excluding specified columns, and removing excluded columns from the returned DataFrame.

    Args:
        - data (pd.DataFrame): Input DataFrame containing numeric data.
        - exclude_columns (list): List of columns to exclude from log return calculations and the result.
        - dropna (bool): Whether to drop rows with NaN values resulting from the calculation.

    Returns:
        - pd.DataFrame: DataFrame with log returns for numeric columns, excluding specified columns.
    """
    # Copy data and remove excluded columns
    data = data.copy().drop(columns=exclude_columns)
    
    # Select numeric columns for transformation
    columns_to_transform = data.select_dtypes(include=[np.number]).columns
    print(f"columns_to_transform = \n{columns_to_transform}, \nlen(columns_to_transform) = {len(columns_to_transform)}")

    # Calculate log returns for each numeric column
    for col in columns_to_transform:
        if (data[col] <= 0).any():
            raise ValueError(f"Column '{col}' contains non-positive values. Log returns require strictly positive values.")
        data[col] = np.log(data[col] / data[col].shift(1))

    # Return data with or without NaN rows based on dropna
    return data.dropna() if dropna else data

def created_sequences_2(data: pd.DataFrame, sequence_length: int = 60, sliding_interval: int = 60) -> list:
    """
    Divide the dataset into sequences based on the sequence_length.
    Each sequence must fully cover the window size.

    Args:
    - data (pd.DataFrame): The input DataFrame.
    - sequence_length (int): The window size for sequences.

    Returns:
    - sequences (list): A list of sequences (as DataFrames).
    """
    sequences = []
    
    # Iterate over the data with a sliding window to create sequences
    for i in range(0, len(data) - sequence_length + 1, sliding_interval):
        # Extract a sequence of specified length from the DataFrame
        seq = data.iloc[i:i + sequence_length].copy()
        sequences.append(seq)

    return sequences

def gaussian_smoothing(data: pd.DataFrame, sigma=2) -> pd.DataFrame:
    """
    Applies Gaussian smoothing to numeric columns in a DataFrame.

    Args:
        - data (pd.DataFrame): Input DataFrame.
        - sigma (float): Standard deviation for the Gaussian kernel (default is 2).

    Returns:
        - pd.DataFrame: Smoothed DataFrame with sorted index.
    """
    # Sort data by index in ascending order and create a copy
    data = data.sort_index(ascending=True).copy()
    
    # Apply Gaussian smoothing to numeric columns
    for column in data.columns:
        if pd.api.types.is_numeric_dtype(data[column]):
            data[column] = gaussian_filter1d(data[column].values, sigma=sigma)
    
    return data

def detect_trends_4(
    dataframe: pd.DataFrame, 
    column: str = 'close', 
    lower_threshold: float = 0.001, 
    upper_threshold: float = 0.02,
    reverse_steps: int = 7,
    trends_to_keep: set = {0, 1, 2, 3, 4}  # Default keeps all trends
) -> pd.DataFrame:
    """
    Detects trends based on log return data provided in a specified column and categorizes them into different strength levels.

    This function analyzes time-series data by evaluating cumulative trends in log return values provided in the input DataFrame. 
    It uses three dictionaries (`dic1`, `dic2`, `dic3`) to track different phases of trends, handles multi-step reversals, and 
    classifies trends dynamically based on cumulative product thresholds and specified thresholds for trend strengths.

    Args:
        - dataframe (pd.DataFrame): Input DataFrame with log return data.
        - column (str): Column name for log returns (default is 'close').
        - lower_threshold (float): Threshold for moderate trends (default is 0.001).
        - upper_threshold (float): Threshold for strong trends (default is 0.02).
        - reverse_steps (int): Steps to confirm trend reversal (default is 7).
        - trends_to_keep (set): Trends to retain, others set to 0 (default is {0, 1, 2, 3, 4}).

    Returns:
        pd.DataFrame: DataFrame with 'trend' column:
                        - 0: No trend
                        - 1: Moderate negative trend
                        - 2: Very strong negative trend
                        - 3: Moderate positive trend
                        - 4: Very strong positive trend
                      Any trends not included in `trends_to_keep` will be reset to 0.

    Function Details:
    1. **Input Assumption**:
    - The input DataFrame already contains log return data in the specified column (`column`).

    2. **Trend Tracking**:
    - Uses dictionaries to monitor trends:
        - `dic1`: Tracks the first phase of the trend.
        - `dic2`: Tracks the second phase if a reversal occurs.
        - `dic3`: Tracks the third phase if another reversal occurs.

    3. **Cumulative Product**:
    - Calculates the cumulative product of `(1 + log_return)` from the specified column to evaluate the strength of trends.

    4. **Reversal Handling**:
    - If a trend reversal persists beyond `reverse_steps`, labels are assigned based on the cumulative product tracked in `dic1`.
    - Subsequent reversals are merged or labeled independently if conditions are met.

    5. **Label Assignment**:
    - Labels are dynamically assigned based on cumulative product thresholds for positive and negative trends:
        - Positive trends are categorized as moderate (3) or strong (4).
        - Negative trends are categorized as moderate (1) or strong (2).

    6. **Trend Filtering**:
    - After detecting trends, only those specified in `trends_to_keep` remain unchanged.
    - Any trend category not included in `trends_to_keep` is reset to 0 (No Trend).

    7. **Edge Cases**:
    - Properly handles scenarios where data points are insufficient for trend analysis or when trend phases overlap, ensuring all data points are labeled.
    """
    # Copy to avoid modifying the original DataFrame
    df = dataframe.copy()
    df['trend'] = None  # Default value 

    dic1, dic2, dic3 = None, None, None # Initialize trend tracking dictionaries
    
    def assign_label(dictio_, lower_threshold, upper_threshold):
        cumulative = dictio_['cumulative']
        if cumulative > (1 + upper_threshold):
            df.iloc[dictio_['ids'], df.columns.get_loc('trend')] = 4  # Very strong positive
        elif (1 + lower_threshold) < cumulative <= (1 + upper_threshold):
            df.iloc[dictio_['ids'], df.columns.get_loc('trend')] = 3  # Moderate positive
        elif (1 - upper_threshold) < cumulative <= (1 - lower_threshold):
            df.iloc[dictio_['ids'], df.columns.get_loc('trend')] = 1  # Moderate negative
        elif cumulative <= (1 - upper_threshold):
            df.iloc[dictio_['ids'], df.columns.get_loc('trend')] = 2  # Very strong negative
        else:
            df.iloc[dictio_['ids'], df.columns.get_loc('trend')] = 0  # No trend
    
    # Process each log return to detect trends
    for idx, log_ret in enumerate(df[column]):
        sign = 1 if log_ret > 0 else -1

        if dic1 is None:  # Initialize dic1
            dic1 = {'ids': [idx], 'last_sign': sign, 'cumulative': (1 + log_ret)}
            continue

        last_sign = dic1['last_sign']
        if sign == last_sign and dic2 is None:  # Continue same trend
            dic1['ids'].append(idx)
            dic1['last_sign'] = sign
            dic1['cumulative'] *= (1 + log_ret)
            continue

        # 1st Reversal occuring
        if dic2 is None:  # Start dic2
            dic2 = {'ids': [idx], 'last_sign': sign, 'cumulative': (1 + log_ret)}
            continue

        last_sign = dic2['last_sign']
        if sign == last_sign and dic3 is None:  # Continue same trend
            dic2['ids'].append(idx)
            dic2['last_sign'] = sign
            dic2['cumulative'] *= (1 + log_ret)
            if len(dic2['ids']) == reverse_steps:
                assign_label(dic1, lower_threshold, upper_threshold) # Assign labels in the 'trend' column for ids of dic1
                dic1, dic2 = dic2, None
            continue

        # 2nd Reversal occuring
        if dic3 is None:  # Start dic3
            dic3 = {'ids': [idx], 'last_sign': sign, 'cumulative': (1 + log_ret)}
            continue

        last_sign = dic3['last_sign']
        if sign == last_sign: # Continue same trend, there is no dic4 to check if is None
            dic3['ids'].append(idx)
            dic3['last_sign'] = sign
            dic3['cumulative'] *= (1 + log_ret)
            dic_prod = dic2['cumulative'] * dic3['cumulative']
            if (sign == 1 and dic_prod > 1) or (sign == -1 and dic_prod < 1):
                dic1['ids'] += dic2['ids'] + dic3['ids']
                dic1['last_sign'] = dic3['last_sign']
                dic1['cumulative'] *= dic2['cumulative'] * dic3['cumulative']
                dic2, dic3 = None, None
                continue

            if len(dic3['ids']) == reverse_steps:      
                assign_label(dic1, lower_threshold, upper_threshold) # Assign labels in the 'trend' column for ids of dic1
                assign_label(dic2, lower_threshold, upper_threshold) # Assign labels in the 'trend' column for ids of dic1
                dic1, dic2, dic3 = dic3, None, None
            continue
            
        # 3rd Reversal occuring
        assign_label(dic1, lower_threshold, upper_threshold) # Assign labels in the 'trend' column for ids of dic1
        dic1, dic2, dic3 = dic2, dic3, {'ids': [idx], 'last_sign': sign, 'cumulative': (1 + log_ret)}

    # Assign remaining labels
    if dic1:
        assign_label(dic1, lower_threshold, upper_threshold)
    if dic2:
        assign_label(dic2, lower_threshold, upper_threshold)
    if dic3:
        assign_label(dic3, lower_threshold, upper_threshold)
    
    # Apply filtering: Keep only selected trends, set others to 0
    df['trend'] = df['trend'].where(df['trend'].isin(trends_to_keep), 0)

    return df

def split_X_y(sequences: list[pd.DataFrame], 
              target_column: str = 'trend',
              detect_trends_function: Callable[[pd.DataFrame, str, float, float, int, set], pd.DataFrame] = detect_trends_4, 
              column: str = 'close', 
              lower_threshold: float = 0.0009, 
              upper_threshold: float = 0.015,
              reverse_steps: int = 7,
              trends_to_keep: set = {0, 1, 2, 3, 4}) -> Tuple[np.ndarray, np.ndarray]:
    """
    Process sequences to generate features (X) and labels (y) with trend detection.

    Args:
        - sequences (list[pd.DataFrame]): List of DataFrame sequences.
        - target_column (str): Column name for labels (default is 'trend').
        - detect_trends_function (Callable): Trend detection function (default is detect_trends_4).
        - column (str): Column for trend detection (default is 'close').
        - lower_threshold (float): Lower threshold for trends (default is 0.0009).
        - upper_threshold (float): Upper threshold for trends (default is 0.015).
        - reverse_steps (int): Steps for trend reversal (default is 7).
        - trends_to_keep (set): Trends to retain (default is {0, 1, 2, 3, 4}).

    Returns:
        - Tuple[np.ndarray, np.ndarray]: X (features), y (labels) as NumPy arrays.
    """
    # Initialize lists for features and labels
    X, y = [], []
    
    # Process each sequence
    for seq in sequences:
        # Apply trend detection
        seq = detect_trends_function(seq, column, lower_threshold, upper_threshold, reverse_steps, trends_to_keep)
        
        # Extract features and labels
        X.append(seq.drop(columns=[target_column]).values)
        y.append(seq[target_column].values)
    
    # Convert to NumPy arrays
    return np.array(X), np.array(y)

def process_and_return_splits(
    with_indicators_file_path: str,
    downsampled_data_minutes: int,
    exclude_columns: list[str],
    lower_threshold: float,
    upper_threshold: float,
    reverse_steps: int,
    sequence_length: int,
    sliding_interval: int,
    trends_to_keep: set = {0, 1, 2, 3, 4}  # Default keeps all trends
) -> tuple[
    list[list[float]],  # X_train: List of sequences, each containing a list of features
    list[list[int]],    # y_train: List of sequences, each containing a list of labels
    list[list[float]],  # X_val: List of sequences, each containing a list of features
    list[list[int]],    # y_val: List of sequences, each containing a list of labels
    list[list[float]],  # X_test: List of sequences, each containing a list of features
    list[list[int]]     # y_test: List of sequences, each containing a list of labels
]:
    """
    Processes time-series data from a CSV file and prepares it for machine learning.

    This function performs the following steps:
        1. Reads data from the specified CSV file and sorts it by date in descending order.
        2. Optionally downsamples the data to a lower frequency (e.g., 5-minute intervals).
        3. Applies Gaussian smoothing to reduce noise in the data.
        4. Calculates log returns for all numeric columns, excluding specified columns.
        5. Detects trends based on defined thresholds (`lower_threshold`, `upper_threshold`, and `reverse_steps`).
        6. Filters trends to keep only those specified in `trends_to_keep`, setting others to 0 (No Trend).
        7. Converts the processed data into sequences of a fixed length (`sequence_length`) with a sliding interval.
        8. Splits the sequences into training (80%), validation (10%), and test (10%) sets.
        9. Further splits the sequences into features (`X`) and labels (`y`) for supervised learning.

    Args:
        - with_indicators_file_path (str): Path to the CSV file with time-series data.
        - downsampled_data_minutes (int): Frequency for downsampling (e.g., 1 for no downsampling).
        - exclude_columns (list[str]): Columns to exclude from log return calculations.
        - lower_threshold (float): Lower threshold for trend detection.
        - upper_threshold (float): Upper threshold for trend detection.
        - reverse_steps (int): Steps for reversing trends in trend detection.
        - sequence_length (int): Length of sequences to create.
        - sliding_interval (int): Interval for sliding the window.
        - trends_to_keep (set): Trends to retain, others set to 0 (default is {0, 1, 2, 3, 4}).

    Returns:
        - tuple: X_train, y_train, X_val, y_val, X_test, y_test as lists of sequences.
    """
    def check_missing_timestamps(data: pd.DataFrame, stage: str):
        """
        Checks for missing timestamps and prints diagnostic info.
        """
        missing_timestamps = pd.date_range(
            start=data.index.min(),
            end=data.index.max(),
            freq='1min',  # Checking 1-minute frequency
            tz=data.index.tz,
        ).difference(data.index)

        print(f"\n{stage} - Missing timestamps: \n{missing_timestamps}")

        if not missing_timestamps.empty:
            for timestamp in missing_timestamps[:5]:  # Show only first 5 missing timestamps
                print(f"\nMissing timestamp: {timestamp}")

                before = data[data.index < timestamp].tail(5)  # 5 data points before
                after = data[data.index > timestamp].head(5)  # 5 data points after

                print("\nData before missing timestamp:")
                display(before) if not before.empty else print("No data available before.")

                print("\nData after missing timestamp:")
                display(after) if not after.empty else print("No data available after.")

    print("\n======== Processing Time-Series Data ========")

    # Step 1: Read & Sort Data
    data = read_csv_file(with_indicators_file_path, preview_rows=0).sort_index(ascending=False)

    # Step 2: Downsample Data
    if downsampled_data_minutes != 1:
        print("\n---> Downsampling Data")
        data = downsample_minute_data(data, downsampled_data_minutes)

    check_missing_timestamps(data, "Data Retrieved")

    # Step 3: Gaussian Smoothing
    data = gaussian_smoothing(data, sigma=7)
    check_missing_timestamps(data, "Gaussian Smoothed Data")

    # Step 4: Compute Log Returns
    data = calculate_log_returns_all_columns(data, exclude_columns=exclude_columns)
    check_missing_timestamps(data, "Log Returns Computed")

    # Step 5: Create Sequences
    sequences = created_sequences_2(data, sequence_length, sliding_interval)

    # Step 6: Train / Validation / Test Split
    train_size = int(len(sequences) * 0.8)
    val_size = int(len(sequences) * 0.1)

    train_sequences = sequences[:train_size]
    val_sequences = sequences[train_size:train_size + val_size]
    test_sequences = sequences[train_size + val_size:]

    print(f"\nNumber of sequences:\n"
          f"  - Total sequences: {len(sequences)}\n"
          f"  - Train: {len(train_sequences)}\n"
          f"  - Validation: {len(val_sequences)}\n"
          f"  - Test: {len(test_sequences)}\n")

    # Step 7: Convert Sequences to X, y
    def split_and_format_data(sequences):
        X, y = split_X_y(
            sequences, target_column='trend',
            detect_trends_function=detect_trends_4,
            column='close', lower_threshold=lower_threshold,
            upper_threshold=upper_threshold, reverse_steps=reverse_steps,
            trends_to_keep=trends_to_keep
        )
        return np.array(X), np.array(y)

    X_train, y_train = split_and_format_data(train_sequences)
    X_val, y_val = split_and_format_data(val_sequences)
    X_test, y_test = split_and_format_data(test_sequences)

    # Step 8: Data Integrity Check (Ensuring Proper Types)
    def check_data_types(X: np.ndarray, y: np.ndarray, label: str):
        """
        Checks if all values in X are float and y are integer.
        """
        unexpected_X = [(i, j, k, type(v)) for i, seq in enumerate(X)
                        for j, row in enumerate(seq)
                        for k, v in enumerate(row) if not isinstance(v, (float, np.float32))]
        unexpected_y = [(i, j, type(v)) for i, seq in enumerate(y)
                        for j, v in enumerate(seq) if not isinstance(v, (int, np.int64))]

        if unexpected_X:
            print(f"\n⚠️ Unexpected type in {label} X:")
            for i, j, k, t in unexpected_X[:5]:  # Show first 5 errors
                print(f"  Sequence {i}, Row {j}, Feature {k}: {t}")

        if unexpected_y:
            print(f"\n⚠️ Unexpected type in {label} y:")
            for i, j, t in unexpected_y[:5]:  # Show first 5 errors
                print(f"  Sequence {i}, Label {j}: {t}")

    check_data_types(X_train, y_train, "Train")
    check_data_types(X_val, y_val, "Validation")
    check_data_types(X_test, y_test, "Test")

    # Step 9: Convert y types if needed
    def convert_dtype(y: np.ndarray):
        return np.array(y, dtype=np.int64) if isinstance(y, np.ndarray) and y.dtype == np.object_ else y

    y_train, y_val, y_test = map(convert_dtype, [y_train, y_val, y_test])

    # Get feature info
    Number_features = X_train.shape[-1]
    close_col_index = data.columns.get_loc('close')
    
    print(f"\nFeature Info:\n  - close_col_index = {close_col_index}\n  - Number_features = {Number_features}")

    return X_train, y_train, X_val, y_val, X_test, y_test, Number_features

def print_class_distribution(y, var_name: str) -> None:
    """
    Prints the class distribution of a label array.

    Args:
        y: Tensor, array, or list of class labels.
        var_name: Name of the variable (for display).
    """
    if isinstance(y, torch.Tensor):
        y = y.cpu().numpy()
    flattened = np.array(y).flatten()

    unique_classes, counts = np.unique(flattened, return_counts=True)
    total = counts.sum()

    header = f"Class Distribution for '{var_name}':"
    line_parts = [
        f"Class {int(c):<3} Percent: {(count / total) * 100:>6.2f}%"
        for c, count in zip(unique_classes, counts)
    ]
    print(header.ljust(40) + " || ".join(line_parts))


## All(Initial) parameters here

In [4]:
ticker = 'BTC-USD'
downsampled_data_minutes = 1 # No downsampling

# Step 0 (Again): Identify parameters for trend settings of the loaded data with 1,000 data points
lower_threshold = 0.0009 
upper_threshold = 0.015  
reverse_steps = 13       

# Features not to be included in the analysis
exclude_columns= ['MACD', 'MACD_signal', 'ROC_10', 'OBV', 'AD_Line']

# Step 3, under ### Correlation Analysis
# Compute correlations with the 'trend' column
# corr = data_trends.corr()
# trend_corr = corr['trend'].sort_values(ascending=False)
strongly_correlated = ['close', 'open', 'SMA_5', 'high', 'low', 'EMA_10', 'SMA_10'] # Strongly correlated (correlation > 0.6)
moderately_correlated = ['BB_middle', 'BB_lower', 'BB_upper', 'RSI_14']             # Moderately correlated (correlation between 0.3 and 0.6)
weakly_correlated = ['SMA_50', 'volume', 'BBW', 'ATR_14']                           # Weakly correlated or negligible (correlation <~ 0.3)

# Add the weakly_correlated and moderately_correlated features to exclude_columns.
exclude_columns += weakly_correlated + moderately_correlated

sequence_length = 1000
sliding_interval = 60

## **_Build the GRU Model_**

### GPU details

In [5]:
# If not in Jupyter Notebook
# import subprocess
# result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
# print(result.stdout)

!nvidia-smi

Sun Apr 20 20:17:44 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 566.03                 Driver Version: 566.03         CUDA Version: 12.7     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4070      WDDM  |   00000000:01:00.0  On |                  N/A |
|  0%   42C    P8             11W /  200W |    2430MiB /  12282MiB |     17%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### Torch and CUDA check

In [6]:
# Check if GPU is available
gpu_available = torch.cuda.is_available()

print("torch.cuda.is_available()", gpu_available)

# If GPU is available, print additional information
if gpu_available:
    print("\nGPU Device Name:", torch.cuda.get_device_name(0))
    print("Number of GPUs:", torch.cuda.device_count())
    print("Total CUDA Cores:", torch.cuda.get_device_properties(0).multi_processor_count * 128)  # NVIDIA GPUs often have 128 cores/SM
    print("Current GPU Device:", torch.cuda.current_device())
else:
    print("No GPU detected.")


torch.cuda.is_available() True

GPU Device Name: NVIDIA GeForce RTX 4070
Number of GPUs: 1
Total CUDA Cores: 5888
Current GPU Device: 0


### Model building

#### Bi-Directional GRU with Attention

In [7]:
class BiGRUWithAttention(nn.Module):
    def __init__(self, input_size: int, hidden_size: int, output_size: int, num_layers: int, dropout: float = 0.0):
        super(BiGRUWithAttention, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        # Bi-Directional GRU Layer
        self.gru = nn.GRU(input_size, hidden_size, num_layers, batch_first=True, bidirectional=True, dropout=dropout)
        # Attention layer
        self.attention_fc = nn.Linear(hidden_size * 2, hidden_size * 2)  # Hidden size * 2 for bi-directional
        # Fully connected layer for classification
        self.fc = nn.Linear(hidden_size * 2, output_size)
        self.dropout = nn.Dropout(dropout)  # Apply dropout before the fully connected layer
        self.init_weights()

    def init_weights(self):
        for name, param in self.named_parameters():
            if 'weight' in name:
                nn.init.xavier_uniform_(param)  # Xavier initialization for weights
            elif 'bias' in name:
                nn.init.constant_(param, 0)  # Zero initialization for biases

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h0 = torch.zeros(self.num_layers * 2, x.size(0), self.hidden_size).to(x.device)  # Bi-directional: num_layers * 2
        # Bi-Directional GRU forward pass
        out, _ = self.gru(x, h0)  # Shape: (batch_size, seq_length, hidden_size * 2)

        # Attention mechanism
        attn_weights = torch.tanh(self.attention_fc(out))  # Shape: (batch_size, seq_length, hidden_size * 2)
        out = attn_weights * out  # Element-wise attention application
        out = self.dropout(out)  # Apply dropout

        # Fully connected layer (applied at each time step)
        out = self.fc(out)  # Shape: (batch_size, seq_length, output_size)
        return out


## Step 6: Train the Model

### Training and validation function

In [8]:
# Training and validation function for Period 1.
def train_and_validate(model, output_size, criterion, optimizer, 
                       X_train, y_train, X_val, y_val, scheduler, 
                       use_scheduler=None, num_epochs=10, batch_size=64, 
                       model_saving_folder=None, model_name=None, stop_signal_file=None):
    print("'train_and_validate' function started. \n")
    # Ensure model saving folder exists (deleting existing first if there is one)
    if model_saving_folder and os.path.exists(model_saving_folder):
        # os.rmdir(model_saving_folder) # Only works on empty folders 
        shutil.rmtree(model_saving_folder) # Safely remove all contents
        if not os.path.exists(model_saving_folder):
            print(f"Existing folder has been removed : {model_saving_folder}\n")
    if model_saving_folder and not os.path.exists(model_saving_folder):
        os.makedirs(model_saving_folder)
        
    if not model_saving_folder:
        model_saving_folder = './saved_models'
    if not model_name:
        model_name = 'model'

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)
    
    # Convert data to tensors # Returns a copy, original is safe
    X_train = torch.tensor(X_train, dtype=torch.float32).to(device)  # (seqs, seq_len, features)
    y_train = torch.tensor(y_train, dtype=torch.long).to(device)    # (seqs, seq_len)
    X_val = torch.tensor(X_val, dtype=torch.float32).to(device)
    y_val = torch.tensor(y_val, dtype=torch.long).to(device)

    # Create TensorDatasets
    train_dataset = TensorDataset(X_train, y_train)
    val_dataset = TensorDataset(X_val, y_val)

    # Create DataLoaders
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    print("y_train:")
    print(type(y_train))
    print(y_train.dtype)
    print(y_train.shape)
    print("X_train:")
    print(type(X_train))
    print(X_train.dtype)
    print(X_train.shape)
    print("\ny_val:")
    print(type(y_val))
    print(y_val.dtype)
    print(y_val.shape)
    print("X_val:")
    print(type(X_val))
    print(X_val.dtype)
    print(X_val.shape)

    # Debug prints for TensorDataset and DataLoader
    print("\nDataset Lengths:")
    print(f"Train Dataset Length: {len(train_dataset)}")
    print(f"Validation Dataset Length: {len(val_dataset)}")

    print("\nDataLoader Batch Sizes:")
    print(f"Number of Batches in Train DataLoader: {len(train_loader)}")
    print(f"Number of Batches in Validation DataLoader: {len(val_loader)}")

    # Additional details for y_train, y_val, and y_test
    print("\ny_train Unique Values and Stats:")
    print(f"Unique values in y_train: {y_train.unique()}")
    print(f"y_train Min: {y_train.min()}, Max: {y_train.max()}")

    print("\ny_val Unique Values and Stats:")
    print(f"Unique values in y_val: {y_val.unique()}")
    print(f"y_val Min: {y_val.min()}, Max: {y_val.max()}")

    # Device check
    print("\nDevice Info:")
    print(f"X_train Device: {X_train.device}")
    print(f"y_train Device: {y_train.device}")
    print(f"X_val Device: {X_val.device}")
    print(f"y_val Device: {y_val.device}\n")

    # Calculate number of batches
    # num_batches = (len(X_train) + batch_size - 1) // batch_size

    global best_results  # Ensure we can modify the external variable if defined outside.
    best_results = []    # Start empty each training run
    model.train()

    for epoch in range(num_epochs):
        epoch_loss = 0
        class_correct = {}  # Dictionary to store correct predictions per class
        class_total = {}  # Dictionary to store total samples per class
        if stop_signal_file and os.path.exists(stop_signal_file):
            print("\nStop signal detected. Exiting training loop safely.\n")
            break
        model.train()
        i=0
        for X_batch, y_batch in train_loader:
            # Reset gradients before forward pass
            optimizer.zero_grad()  # Best practice

            # Forward pass
            outputs = model(X_batch)
            outputs = outputs.view(-1, output_size)
            y_batch = y_batch.view(-1)

            if epoch == 1 and i < 3:
                i += 1
                print(f"\nUnique target values: {y_batch.unique()}")
                print(f"Target dtype: {y_batch.dtype}")
                print(f"Min target: {y_batch.min()}, Max target: {y_batch.max()}")
                print("Unique classes in y_train:", y_train.unique())
                print(f"Unique classes in y_val: {y_val.unique()}\n")
            
            # Compute loss
            loss = criterion(outputs, y_batch)

            # Compute class-wise accuracy (Accumulates values in dict)
            compute_classwise_accuracy(outputs, y_batch, class_correct, class_total)

            # Backward pass and optimization
            # No longer reset gradients here: optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item() * X_batch.size(0)  # Scale back to total loss
            
        train_loss = epoch_loss / len(train_loader.dataset)

        # Compute per-class training accuracy
        # train_classwise_accuracy = {int(c): (class_correct[c] / class_total[c]) * 100 if class_total[c] > 0 else 0 
        #                            for c in sorted(class_total.keys())}
        train_classwise_accuracy = {int(c): f"{(class_correct[c] / class_total[c]) * 100:.2f}%" if class_total[c] > 0 else "0.00%" 
                                    for c in sorted(class_total.keys())}
        
        # Perform validation at the end of each epoch
        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0
        val_class_correct = {}
        val_class_total = {}
        with torch.no_grad():
            for X_val_batch, y_val_batch in val_loader:
                val_outputs = model(X_val_batch).view(-1, output_size)
                val_labels = y_val_batch.view(-1)
                val_loss += criterion(val_outputs, val_labels).item() * X_val_batch.size(0)  # Scale to total loss
                val_predictions = torch.argmax(val_outputs, dim=-1)
                val_correct += (val_predictions == val_labels).sum().item()
                val_total += val_labels.size(0)
                # Compute per-class validation accuracy
                compute_classwise_accuracy(val_outputs, val_labels, val_class_correct, val_class_total)
        val_loss /= len(val_loader.dataset)
        val_accuracy = val_correct / val_total

        # Compute per-class validation accuracy
        # val_classwise_accuracy = {int(c): (val_class_correct[c] / val_class_total[c]) * 100 if val_class_total[c] > 0 else 0 
        #                          for c in sorted(val_class_total.keys())}
        val_classwise_accuracy = {int(c): f"{(val_class_correct[c] / val_class_total[c]) * 100:.2f}%" if val_class_total[c] > 0 else "0.00%" 
                                  for c in sorted(val_class_total.keys())}

        print(f"Epoch {epoch+1}/{num_epochs}, "
              f"Train Loss: {train_loss:.9f}, "
              f"Train-Class-Acc: {train_classwise_accuracy}, "
              f"Val Loss: {val_loss:.9f}, "
              f"Val Accuracy: {val_accuracy * 100:.2f}%, "
              f"Val-Class-Acc: {val_classwise_accuracy}, "
              f"LR: {optimizer.param_groups[0]['lr']:.9f}")

        # Save current model and update best results if applicable
        current_epoch_info = {
            "epoch": epoch+1,
            "train_loss": train_loss,
            "train_classwise_accuracy": train_classwise_accuracy,
            "val_loss": val_loss,
            "val_accuracy": val_accuracy,
            "val_classwise_accuracy": val_classwise_accuracy,
            'learning_rate': optimizer.param_groups[0]['lr'], # Optimizer state
            "model_path": os.path.join(model_saving_folder, f"{model_name}_epoch_{epoch+1}.pth")
        }

        # Insert this epoch if we have fewer than 5 results
        # or if it beats the lowest of the top 5
        if len(best_results) < 5 or val_accuracy > best_results[-1]["val_accuracy"]:
            if len(best_results) == 5:
                # Remove the worst model from the list, the last (lowest accuracy)
                worst = best_results.pop() 
                if os.path.exists(worst["model_path"]):
                    os.remove(worst["model_path"])
                    print(f"Removed old model with accuracy {worst['val_accuracy']*100:.2f}%, and file was at {worst['model_path']}")
            # Just insert and sort by val_accuracy descending
            best_results.append(current_epoch_info) 
            best_results.sort(key=lambda x: x["val_accuracy"], reverse=True)
            torch.save({ # Save this model
                'epoch': epoch+1,  # Save the current epoch
                'train_loss': train_loss,
                'val_loss': val_loss,
                'model_state_dict': model.state_dict(),  # Model weights
                'optimizer_state_dict': optimizer.state_dict(),  # Optimizer state
                'learning_rate': optimizer.param_groups[0]['lr'] # Optimizer state
            }, current_epoch_info["model_path"])
            print(f"Model saved after epoch {epoch+1} to {current_epoch_info['model_path']} \n")

        if use_scheduler == True:
            # Scheduler step should follow after considering the results (placed after otallher losses)
            scheduler.step(val_loss)


    # Save the final model
    if current_epoch_info:
        final_model_path = os.path.join(model_saving_folder, f"{model_name}_final.pth")
        torch.save({ # Save this model
            'epoch': epoch+1,  # Save the current epoch
            'train_loss': train_loss,
            'val_loss': val_loss,
            'model_state_dict': model.state_dict(),  # Model weights
            'optimizer_state_dict': optimizer.state_dict(),  # Optimizer state
            'learning_rate': optimizer.param_groups[0]['lr'] # Optimizer state
        }, final_model_path)
        print(f"\nFinal model saved to {final_model_path}")

    print("\nTraining complete. \n\nTop 5 Models Sorted by Validation Accuracy: ")
    for res in best_results:        
        print(f"Epoch {res['epoch']}/{num_epochs}, "
              f"Train Loss: {res['train_loss']:.9f}, "
              f"Train-Class-Acc: {train_classwise_accuracy}, " 
              f"Val Loss: {res['val_loss']:.9f}, "
              f"Val Accuracy: {res['val_accuracy'] * 100:.2f}%, "
              f"Val-Class-Acc: {val_classwise_accuracy}, "
              f"Model Path: {res['model_path']}")
    print('\n')
    
    del X_train, y_train, X_val, y_val, train_dataset, val_dataset, train_loader, val_loader
    torch.cuda.empty_cache()

    # Load the checkpoint
    # checkpoint = torch.load("path/to/model_checkpoint.pth")
    # # Restore model state
    # model.load_state_dict(checkpoint['model_state_dict'])
    # # Restore optimizer state
    # optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    # # Restore scheduler state (if used)
    # scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    # # Restore epoch and other metadata
    # start_epoch = checkpoint['epoch'] + 1  # Resume from the next epoch
    # loss = checkpoint['loss']  # Optional
    # print(f"Checkpoint loaded. Resuming from epoch {start_epoch}")


def compute_classwise_accuracy(student_logits_flat, y_batch, class_correct, class_total):
    """
    Computes per-class accuracy by accumulating correct and total samples for each class.
    
    student_logits_flat: Model predictions (logits) in shape [batch_size * seq_len, output_size]
    y_batch: True labels in shape [batch_size * seq_len]
    class_correct: Dictionary to store correct predictions per class
    class_total: Dictionary to store total samples per class
    """
    # Convert logits to predicted class indices
    predictions = torch.argmax(student_logits_flat, dim=-1)  # Shape: [batch_size * seq_len]

    # Loop through batch elements to track correct/total per class
    for label, pred in zip(y_batch.cpu().numpy(), predictions.cpu().numpy()):
        if label not in class_total:
            class_total[label] = 0
            class_correct[label] = 0
        class_total[label] += 1
        if label == pred:
            class_correct[label] += 1


# Training and validation function for Period 2 and beyond 
def train_and_validate_MSE(student_model, teacher_model, stable_classes, output_size, criterion, optimizer, 
                           X_train, y_train, X_val, y_val, scheduler, 
                           use_scheduler=None, num_epochs=10, batch_size=64, alpha=0.5,
                           model_saving_folder=None, model_name=None, stop_signal_file=None):
    """
    student_model: The new LoRA-based student model (with output size 3).
    teacher_model: Frozen teacher model from period 1 (with output size 2).
    criterion: CrossEntropyLoss function.
    optimizer: Optimizer for student model.
    X_train, y_train, X_val, y_val: Training/validation data (as NumPy arrays or similar).
    num_epochs: Number of epochs to train.
    batch_size: Batch size for DataLoader.
    alpha: Weighting factor for distillation loss (alpha * distill_loss + (1-alpha) * ce_loss).
    """
    print("'train_and_validate' function started. \n")
    # Ensure model saving folder exists (deleting existing first if there is one)
    if model_saving_folder and os.path.exists(model_saving_folder):
        # os.rmdir(model_saving_folder) # Only works on empty folders 
        shutil.rmtree(model_saving_folder) # Safely remove all contents
        if not os.path.exists(model_saving_folder):
            print(f"Existing folder has been removed : {model_saving_folder}\n")
    if model_saving_folder and not os.path.exists(model_saving_folder):
        os.makedirs(model_saving_folder)
        
    if not model_saving_folder:
        model_saving_folder = './saved_models'
    if not model_name:
        model_name = 'model'

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    student_model.to(device)
    teacher_model.to(device)

    # Convert data to tensors # Returns a copy, original is safe
    X_train = torch.tensor(X_train, dtype=torch.float32).to(device)  # (seqs, seq_len, features)
    y_train = torch.tensor(y_train, dtype=torch.long).to(device)    # (seqs, seq_len)
    X_val = torch.tensor(X_val, dtype=torch.float32).to(device)
    y_val = torch.tensor(y_val, dtype=torch.long).to(device)

    # Create TensorDatasets
    train_dataset = TensorDataset(X_train, y_train)
    val_dataset = TensorDataset(X_val, y_val)

    # Create DataLoaders
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    print("y_train:")
    print(type(y_train))
    print(y_train.dtype)
    print(y_train.shape)
    print("X_train:")
    print(type(X_train))
    print(X_train.dtype)
    print(X_train.shape)
    print("\ny_val:")
    print(type(y_val))
    print(y_val.dtype)
    print(y_val.shape)
    print("X_val:")
    print(type(X_val))
    print(X_val.dtype)
    print(X_val.shape)

    # Debug prints for TensorDataset and DataLoader
    print("\nDataset Lengths:")
    print(f"Train Dataset Length: {len(train_dataset)}")
    print(f"Validation Dataset Length: {len(val_dataset)}")

    print("\nDataLoader Batch Sizes:")
    print(f"Number of Batches in Train DataLoader: {len(train_loader)}")
    print(f"Number of Batches in Validation DataLoader: {len(val_loader)}")

    # Additional details for y_train, y_val, and y_test
    print("\ny_train Unique Values and Stats:")
    print(f"Unique values in y_train: {y_train.unique()}")
    print(f"y_train Min: {y_train.min()}, Max: {y_train.max()}")

    print("\ny_val Unique Values and Stats:")
    print(f"Unique values in y_val: {y_val.unique()}")
    print(f"y_val Min: {y_val.min()}, Max: {y_val.max()}")

    # Device check
    print("\nDevice Info:")
    print(f"X_train Device: {X_train.device}")
    print(f"y_train Device: {y_train.device}")
    print(f"X_val Device: {X_val.device}")
    print(f"y_val Device: {y_val.device}\n")

    # Calculate number of batches
    # num_batches = (len(X_train) + batch_size - 1) // batch_size

    global best_results  # Ensure we can modify the external variable if defined outside.
    best_results = []    # Start empty each training run

    teacher_model.eval()  # Ensure teacher is frozen
    for epoch in range(num_epochs):
        epoch_loss = 0.0
        class_correct = {}  # Dictionary to store correct predictions per class
        class_total = {}  # Dictionary to store total samples per class
        if stop_signal_file and os.path.exists(stop_signal_file):
            print("\nStop signal detected. Exiting training loop safely.\n")
            break
        student_model.train()
        i=0
        for X_batch, y_batch in train_loader:
            # Reset gradients before forward pass
            optimizer.zero_grad()  # Best practice

            # Forward pass: student model produces logits for output_size classes.
            student_logits = student_model(X_batch)  # Shape: [batch, seq_len, output_size]
            
            # Reshape for CE loss computation.
            student_logits_flat = student_logits.view(-1, output_size)
            y_batch = y_batch.view(-1)

            # Compute Cross-Entropy loss
            ce_loss = criterion(student_logits_flat, y_batch)

            # Compute class-wise accuracy (Accumulates values in dict)
            compute_classwise_accuracy(student_logits_flat, y_batch, class_correct, class_total)

            if epoch == 1 and i < 3:
                i += 1
                print(f"\nUnique target values: {y_batch.unique()}")
                print(f"Target dtype: {y_batch.dtype}")
                print(f"Min target: {y_batch.min()}, Max target: {y_batch.max()}")
                print("Unique classes in y_train:", y_train.unique())
                print(f"Unique classes in y_val: {y_val.unique()}\n")

            # Knowledge Distillation:
            # Forward pass through teacher (pre-trained on previous period data).
            with torch.no_grad():
                teacher_logits = teacher_model(X_batch)  # Shape: [batch, seq_len, teacher_output_size]
            # Use stable_classes (a list of indices) to extract the relevant logits.
            # We distill only the stable classes (class 1 if teacher is from period 1).
            # Teacher's class index 1 corresponds to student's class index 1.
            # It's safer to use index_select to ensure the operation works on GPU.
            stable_indices = torch.tensor(stable_classes, device=teacher_logits.device)
            teacher_stable = teacher_logits.index_select(dim=2, index=stable_indices)
            student_stable = student_logits.index_select(dim=2, index=stable_indices)

            # Compute the distillation loss over the selected stable classes.
            distill_loss = F.mse_loss(student_stable, teacher_stable)
            
            # Total loss: balance between cross-entropy and distillation loss.
            total_loss = alpha * distill_loss + (1 - alpha) * ce_loss
            total_loss.backward()
            optimizer.step()
            epoch_loss += total_loss.item() * X_batch.size(0)
            
        train_loss = epoch_loss / len(train_loader.dataset)

        # Compute per-class training accuracy
        # train_classwise_accuracy = {int(c): (class_correct[c] / class_total[c]) * 100 if class_total[c] > 0 else 0 
        #                            for c in sorted(class_total.keys())}
        train_classwise_accuracy = {int(c): f"{(class_correct[c] / class_total[c]) * 100:.2f}%" if class_total[c] > 0 else "0.00%" 
                                    for c in sorted(class_total.keys())}

        # Perform validation at the end of each epoch (only CE loss and accuracy)
        student_model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0
        val_class_correct = {}
        val_class_total = {}
        with torch.no_grad():
            for X_val_batch, y_val_batch in val_loader:
                val_outputs = student_model(X_val_batch).view(-1, output_size)
                val_labels = y_val_batch.view(-1)
                val_loss += criterion(val_outputs, val_labels).item() * X_val_batch.size(0)  # Scale to total loss
                val_predictions = torch.argmax(val_outputs, dim=-1)
                val_correct += (val_predictions == val_labels).sum().item()
                val_total += val_labels.size(0)
                # Compute per-class validation accuracy
                compute_classwise_accuracy(val_outputs, val_labels, val_class_correct, val_class_total)
        val_loss /= len(val_loader.dataset)
        val_accuracy = val_correct / val_total

        # Compute per-class validation accuracy
        # val_classwise_accuracy = {int(c): (val_class_correct[c] / val_class_total[c]) * 100 if val_class_total[c] > 0 else 0 
        #                          for c in sorted(val_class_total.keys())}
        val_classwise_accuracy = {int(c): f"{(val_class_correct[c] / val_class_total[c]) * 100:.2f}%" if val_class_total[c] > 0 else "0.00%" 
                                  for c in sorted(val_class_total.keys())}

        print(f"Epoch {epoch+1}/{num_epochs}, "
              f"Train Loss: {train_loss:.9f}, "
              f"Train-Class-Acc: {train_classwise_accuracy}, "
              f"Val Loss: {val_loss:.9f}, "
              f"Val Accuracy: {val_accuracy * 100:.2f}%, "
              f"Val-Class-Acc: {val_classwise_accuracy}, "
              f"LR: {optimizer.param_groups[0]['lr']:.9f}")

        # Save current model and update best results if applicable
        current_epoch_info = {
            "epoch": epoch+1,
            "train_loss": train_loss,
            "train_classwise_accuracy": train_classwise_accuracy,
            "val_loss": val_loss,
            "val_accuracy": val_accuracy,
            "val_classwise_accuracy": val_classwise_accuracy,
            'learning_rate': optimizer.param_groups[0]['lr'], # Optimizer state
            "model_path": os.path.join(model_saving_folder, f"{model_name}_epoch_{epoch+1}.pth")
        }

        # Insert this epoch if we have fewer than 5 results
        # or if it beats the lowest of the top 5
        if len(best_results) < 5 or val_accuracy > best_results[-1]["val_accuracy"]:
            if len(best_results) == 5:
                # Remove the worst model from the list, the last (lowest accuracy)
                worst = best_results.pop() 
                if os.path.exists(worst["model_path"]):
                    os.remove(worst["model_path"])
                    print(f"Removed old model with accuracy {worst['val_accuracy']*100:.2f}%, and file was at {worst['model_path']}")
            # Just insert and sort by val_accuracy descending
            best_results.append(current_epoch_info) 
            best_results.sort(key=lambda x: x["val_accuracy"], reverse=True)
            torch.save({ # Save this model
                'epoch': epoch+1,  # Save the current epoch
                'train_loss': train_loss,
                'val_loss': val_loss,
                'model_state_dict': student_model.state_dict(),  # Model weights
                'optimizer_state_dict': optimizer.state_dict(),  # Optimizer state
                'learning_rate': optimizer.param_groups[0]['lr'] # Optimizer state
            }, current_epoch_info["model_path"])
            print(f"Model saved after epoch {epoch+1} to {current_epoch_info['model_path']} \n")

        if use_scheduler == True:
            # Scheduler step should follow after considering the results (placed after otallher losses)
            scheduler.step(val_loss)

    # Save the final model
    if current_epoch_info:
        final_model_path = os.path.join(model_saving_folder, f"{model_name}_final.pth")
        torch.save({ # Save this model
            'epoch': epoch+1,  # Save the current epoch
            'train_loss': train_loss,
            'val_loss': val_loss,
            'model_state_dict': student_model.state_dict(),  # Model weights
            'optimizer_state_dict': optimizer.state_dict(),  # Optimizer state
            'learning_rate': optimizer.param_groups[0]['lr'] # Optimizer state
        }, final_model_path)
        print(f"\nFinal model saved to {final_model_path}")

    print("\nTraining complete. \n\nTop 5 Models Sorted by Validation Accuracy: ")
    for res in best_results:        
        print(f"Epoch {res['epoch']}/{num_epochs}, "
              f"Train Loss: {res['train_loss']:.9f}, "
              f"Train-Class-Acc: {train_classwise_accuracy}, " 
              f"Val Loss: {res['val_loss']:.9f}, "
              f"Val Accuracy: {res['val_accuracy'] * 100:.2f}%, "
              f"Val-Class-Acc: {val_classwise_accuracy}, "
              f"Model Path: {res['model_path']}")
    print('\n')
    
    del X_train, y_train, X_val, y_val, train_dataset, val_dataset, train_loader, val_loader
    torch.cuda.empty_cache()


### Set parameters and initialize model

In [9]:
print("PyTorch compiled CUDA version:", torch.version.cuda)

PyTorch compiled CUDA version: 12.4


In [10]:
print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
print("Num GPUs:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))

torch.manual_seed(42)
print("Seeding successful!")


PyTorch Version: 2.4.1+cu124
CUDA Available: True
Num GPUs: 1
GPU Name: NVIDIA GeForce RTX 4070
Seeding successful!


### list_period_files_full_path

In [11]:
pair = 'BTCUSD'
file_name = f'Polygon_{pair}_4Y_1min'  # File name for saving data
# BASE_FOLDER_PATH = f"{Working_directory}/Data/{file_name}"
BASE_FOLDER_PATH = f"Data/{file_name}"
# folder_path=f'{BASE_FOLDER_PATH}/polygon_io/12_USD_Crypto_Pairs/{file_name}'
folder_path=f'{BASE_FOLDER_PATH}'
if not os.path.isdir(folder_path):
    raise FileNotFoundError(f"Directory '{folder_path}' does not exist.")
file_path=f'{folder_path}/{file_name}.csv'
number_days = 190
with_indicators_file_path = os.path.normpath(f'{folder_path}/_{file_name}_{number_days}_days_with_indicators.csv')

# LABORATORY\_Global_Pytorch\Continual_Learning\Data\Polygon_BTCUSD_4Y_1min

list_period_files_full_path = [
    # Period 1
    with_indicators_file_path,
    # Period 2
    os.path.normpath(f"{folder_path}/Polygon_BTCUSD_4Y_1min_190_days__2020-11-11__2021-05-20__with_indicators.csv"),
    # Period 3
    os.path.normpath(f"{folder_path}/Polygon_BTCUSD_4Y_1min_190_days__2021-05-20__2021-11-26__with_indicators.csv"),
    # Period 4
    os.path.normpath(f"{folder_path}/Polygon_BTCUSD_4Y_1min_190_days__2021-11-26__2022-06-04__with_indicators.csv"),
    # Period 5
    os.path.normpath(f"{folder_path}/Polygon_BTCUSD_4Y_1min_190_days__2022-06-04__2022-12-11__with_indicators.csv")
]

with_indicators_file_path

'Data\\Polygon_BTCUSD_4Y_1min\\_Polygon_BTCUSD_4Y_1min_190_days_with_indicators.csv'

In [12]:
for file in os.listdir(folder_path):
    print(f"Found file: {file}")

# Polygon_BTCUSD_4Y_1min_190_days_with_indicators.csv

Found file: Polygon_BTCUSD_4Y_1min.png
Found file: Polygon_BTCUSD_4Y_1min_190_days__2020-11-11__2021-05-20.png
Found file: Polygon_BTCUSD_4Y_1min_190_days__2020-11-11__2021-05-20__with_indicators.csv
Found file: Polygon_BTCUSD_4Y_1min_190_days__2021-05-20__2021-11-26.png
Found file: Polygon_BTCUSD_4Y_1min_190_days__2021-05-20__2021-11-26__with_indicators.csv
Found file: Polygon_BTCUSD_4Y_1min_190_days__2021-11-26__2022-06-04.png
Found file: Polygon_BTCUSD_4Y_1min_190_days__2021-11-26__2022-06-04__with_indicators.csv
Found file: Polygon_BTCUSD_4Y_1min_190_days__2022-06-04__2022-12-11.png
Found file: Polygon_BTCUSD_4Y_1min_190_days__2022-06-04__2022-12-11__with_indicators.csv
Found file: Polygon_BTCUSD_4Y_1min_525_days.png
Found file: _Polygon_BTCUSD_4Y_1min_190_days.png
Found file: _Polygon_BTCUSD_4Y_1min_190_days_with_indicators.csv


In [13]:
X_train, y_train, X_val, y_val, X_test, y_test, Number_features = process_and_return_splits(
    with_indicators_file_path = list_period_files_full_path[0],
    downsampled_data_minutes = downsampled_data_minutes,
    exclude_columns = exclude_columns,
    lower_threshold = lower_threshold,
    upper_threshold = upper_threshold,
    reverse_steps = reverse_steps,
    sequence_length = sequence_length,
    sliding_interval = sliding_interval,
    trends_to_keep = {0, 1, 2, 3, 4}  # Default keeps all trends
    # trends_to_keep = {0, 1, 2, 3, 4}  # Default keeps all trends
)

unique_classes = np.unique(y_val)
num_classes = len(unique_classes)
print(f"unique_classes = {unique_classes}")
print(f"num_classes = {num_classes}")

del X_train, y_train, X_val, y_val, X_test, y_test, Number_features
del unique_classes, num_classes



======== Processing Time-Series Data ========

File path: Data\Polygon_BTCUSD_4Y_1min\_Polygon_BTCUSD_4Y_1min_190_days_with_indicators.csv

Data Retrieved - Missing timestamps: 
DatetimeIndex([], dtype='datetime64[ns, UTC]', freq='min')

Gaussian Smoothed Data - Missing timestamps: 
DatetimeIndex([], dtype='datetime64[ns, UTC]', freq='min')
columns_to_transform = 
Index(['open', 'high', 'low', 'close', 'SMA_5', 'SMA_10', 'EMA_10'], dtype='object'), 
len(columns_to_transform) = 7

Log Returns Computed - Missing timestamps: 
DatetimeIndex([], dtype='datetime64[ns, UTC]', freq='min')

Number of sequences:
  - Total sequences: 4543
  - Train: 3634
  - Validation: 454
  - Test: 455


Feature Info:
  - close_col_index = 3
  - Number_features = 7
unique_classes = [0 1 2 3 4]
num_classes = 5


### __*All periods data*__

In [15]:
"""
- 'trend': Categorized trend values based on the detected phases:
    - 0: No trend
    - 1: Moderate negative trend
    - 2: Very strong negative trend
    - 3: Moderate positive trend
    - 4: Very strong positive trend
"""

# Initialize empty lists to combine validation elements
# X_train_all = []
# y_train_all = []
# X_val_all = []
# y_val_all = []
# X_test_all = []
# y_test_all = []

# for path_ in list_period_files_full_path:
#     with contextlib.redirect_stdout(open(os.devnull, 'w')):
#         X_train_, y_train_, X_val_, y_val_, X_test_, y_test_, Number_features = process_and_return_splits(
#             with_indicators_file_path = path_,
#             downsampled_data_minutes = downsampled_data_minutes,
#             exclude_columns = exclude_columns,
#             lower_threshold = lower_threshold,
#             upper_threshold = upper_threshold,
#             reverse_steps = reverse_steps,
#             sequence_length = sequence_length,
#             sliding_interval = sliding_interval,
#             trends_to_keep = {0, 1, 2, 3, 4}  # Default keeps all trends
#         )
#     # Combine validation elements
#     X_train_all.extend(X_train_)
#     y_train_all.extend(y_train_)
#     X_val_all.extend(X_val_)
#     y_val_all.extend(y_val_)
#     X_test_all.extend(X_test_)
#     y_test_all.extend(y_test_)
#     # Delete unused variables to save memory
#     del X_train_, y_train_, X_val_, y_val_, X_test_, y_test_

# X_train_all = np.array(X_train_all)
# y_train_all = np.array(y_train_all)
# X_val_all = np.array(X_val_all)
# y_val_all = np.array(y_val_all)
# X_test_all = np.array(X_test_all)
# y_test_all = np.array(y_test_all)


# with contextlib.redirect_stdout(open(os.devnull, 'w')):
#     X_train, y_train, X_val, y_val, X_test, y_test, Number_features = process_and_return_splits(
#         with_indicators_file_path = list_period_files_full_path[0],
#         downsampled_data_minutes = downsampled_data_minutes,
#         exclude_columns = exclude_columns,
#         lower_threshold = lower_threshold,
#         upper_threshold = upper_threshold,
#         reverse_steps = reverse_steps,
#         sequence_length = sequence_length,
#         sliding_interval = sliding_interval,
#         trends_to_keep = {0, 1}  # Default keeps all trends : {0, 1, 2, 3, 4}
#         # trends_to_keep = {0, 1, 2, 3, 4}  # Default keeps all trends
#     )
    
# print(f"\nNumber_features = {Number_features}")

# unique_classes = np.unique(y_val)
# num_classes = len(unique_classes)
# print(f"unique_classes = {unique_classes}")
# print(f"num_classes = {num_classes}")


"\n- 'trend': Categorized trend values based on the detected phases:\n    - 0: No trend\n    - 1: Moderate negative trend\n    - 2: Very strong negative trend\n    - 3: Moderate positive trend\n    - 4: Very strong positive trend\n"

### Period 1 --> Training and saving in __*'1st_try'*__ (BiGRUWithAttention, num_layers = 4) ---> Val acc = 98.35 %
### Val-Class-Acc: {0: '98.63%', 1: '97.92%'}

In [58]:
"""
- 'trend': Categorized trend values based on the detected phases:
    - 0: No trend
    - 1: Moderate negative trend
    - 2: Very strong negative trend
    - 3: Moderate positive trend
    - 4: Very strong positive trend
"""
with contextlib.redirect_stdout(open(os.devnull, 'w')):
    X_train, y_train, X_val, y_val, X_test, y_test, Number_features = process_and_return_splits(
        with_indicators_file_path = list_period_files_full_path[0], # Change 
        downsampled_data_minutes = downsampled_data_minutes,
        exclude_columns = exclude_columns,
        lower_threshold = lower_threshold,
        upper_threshold = upper_threshold,
        reverse_steps = reverse_steps,
        sequence_length = sequence_length,
        sliding_interval = sliding_interval,
        trends_to_keep = {0, 1}  # Default keeps all trends : {0, 1, 2, 3, 4}
        # trends_to_keep = {0, 1, 2, 3, 4}  # Default keeps all trends
    )

print(f"\nNumber_features = {Number_features}")

unique_classes = np.unique(y_val)
num_classes = len(unique_classes)
print(f"unique_classes = {unique_classes}")
print(f"num_classes = {num_classes}")



Number_features = 7
unique_classes = [0 1]
num_classes = 2


In [61]:
#----------------------------------------------------------------------
# Initialize the list to store results across runs
track_across_runs = []
#----------------------------------------------------------------------

# Model parameters
input_size = Number_features  # Number of features
hidden_size = 64  # Number of GRU units
output_size = num_classes # Must be dynamic, up to 5  # Number of trend classes (0, 15, 25, -15, -25)
num_layers = 4  # Number of GRU layers
num_epochs= 2000 # Number of epochs/ go through entire data
batch_size= 64 # How many sequences passed at once to the model
model_name = 'BiGRUWithAttention' # Name of the model to use for saving
best_results = [] # Initialize this outside the training function or at the beginning of training
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# Define a global stop signal
stop_signal_file = os.path.normpath(os.path.join('Class_Incremental_CL', 'Classif_Bi_Dir_GRU_Model/stop_training.txt'))  # Create this file to stop training
model_saving_folder = os.path.normpath(os.path.join('Class_Incremental_CL', "Classif_Bi_Dir_GRU_Model/Trained_models/Baseline/Period_1/1st_try"))
ensure_folder(model_saving_folder)

# Instantiate the model
class_gru_model = BiGRUWithAttention(input_size, hidden_size, output_size, num_layers).to(device)

# Define the loss function, optimizer and scheduler
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(class_gru_model.parameters(), lr=0.0001) # lr=0.00005
# optimizer = optim.Adam(class_gru_model.parameters(), lr=0.001, weight_decay=1e-5)
# scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', factor=0.9, patience=10)

# train_and_validate(class_gru_model, output_size, criterion, optimizer, X_train_all, y_train_all, X_val_all, y_val_all, scheduler, 
train_and_validate(class_gru_model, output_size, criterion, optimizer, X_train, y_train, X_val, y_val, scheduler, 
                   False, num_epochs, batch_size, model_saving_folder, model_name, stop_signal_file)

#----------------------------------------------------------------------
# Append only the best result (already at index 0)
track_across_runs.append(best_results[0])
#----------------------------------------------------------------------

for res in best_results:        
    print(f"Epoch {res['epoch']}/{num_epochs}, "
            f"Train Loss: {res['train_loss']:.4f}, " 
            f"Val Loss: {res['val_loss']:.4f}, "
            f"Val Accuracy: {res['val_accuracy'] * 100:.2f}%, "
            f"Model Path: {res['model_path']}")      
print(f"\nclass_gru_model: \n{class_gru_model}")

print(f"\nunique_classes = {unique_classes}")
print(f"num_classes = {num_classes}")
# del unique_classes, num_classes
for var in ["X_train", "y_train", "X_val", "y_val", "X_test", "y_test", "Number_features", "unique_classes", "num_classes"]:
    if var in locals():
        del locals()[var]


'train_and_validate' function started. 

Existing folder has been removed : Class_Incremental_CL\Classif_Bi_Dir_GRU_Model\Trained_models\Baseline\Period_1\1st_try

y_train:
<class 'torch.Tensor'>
torch.int64
torch.Size([3634, 1000])
X_train:
<class 'torch.Tensor'>
torch.float32
torch.Size([3634, 1000, 7])

y_val:
<class 'torch.Tensor'>
torch.int64
torch.Size([454, 1000])
X_val:
<class 'torch.Tensor'>
torch.float32
torch.Size([454, 1000, 7])

Dataset Lengths:
Train Dataset Length: 3634
Validation Dataset Length: 454

DataLoader Batch Sizes:
Number of Batches in Train DataLoader: 57
Number of Batches in Validation DataLoader: 8

y_train Unique Values and Stats:
Unique values in y_train: tensor([0, 1], device='cuda:0')
y_train Min: 0, Max: 1

y_val Unique Values and Stats:
Unique values in y_val: tensor([0, 1], device='cuda:0')
y_val Min: 0, Max: 1

Device Info:
X_train Device: cuda:0
y_train Device: cuda:0
X_val Device: cuda:0
y_val Device: cuda:0

Epoch 1/2000, Train Loss: 0.681195149, 

In [15]:
import torch
from torch.utils.data import TensorDataset, DataLoader

def compute_classwise_accuracy(student_logits_flat, y_batch, class_correct, class_total):
    predictions = torch.argmax(student_logits_flat, dim=-1)
    correct_mask = (predictions == y_batch)
    unique_labels = torch.unique(y_batch)

    for label in unique_labels:
        label = label.item()
        if label not in class_total:
            class_total[label] = 0
            class_correct[label] = 0
        label_mask = (y_batch == label)
        class_total[label] += label_mask.sum().item()
        class_correct[label] += (label_mask & correct_mask).sum().item()

def add_val_class_acc(model_copy, output_size, criterion, X_val, y_val, file_path, model_checkpoint, batch_size=64):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model_copy.to(device)
    model_copy.eval()

    X_val = torch.tensor(X_val, dtype=torch.float32).to(device)
    y_val = torch.tensor(y_val, dtype=torch.long).to(device)
    val_dataset = TensorDataset(X_val, y_val)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    val_loss = 0.0
    val_correct = 0
    val_total = 0
    val_class_correct = {}
    val_class_total = {}

    with torch.no_grad():
        for X_val_batch, y_val_batch in val_loader:
            val_outputs = model_copy(X_val_batch).view(-1, output_size)
            val_labels = y_val_batch.view(-1)
            val_loss += criterion(val_outputs, val_labels).item() * X_val_batch.size(0)
            val_predictions = torch.argmax(val_outputs, dim=-1)
            val_correct += (val_predictions == val_labels).sum().item()
            val_total += val_labels.size(0)
            compute_classwise_accuracy(val_outputs, val_labels, val_class_correct, val_class_total)

    val_accuracy = val_correct / val_total
    val_classwise_accuracy = {
        int(c): f"{(val_class_correct[c] / val_class_total[c]) * 100:.2f}%" if val_class_total[c] > 0 else "0.00%"
        for c in sorted(val_class_total.keys())
    }

    model_checkpoint.update({
        "val_accuracy": val_accuracy,
        "Val-Class-Acc": val_classwise_accuracy
    })
    torch.save(model_checkpoint, file_path)
    print(f"✅ Updated: {os.path.basename(file_path)} | Acc: {val_accuracy:.4f} | Saved to: {file_path}")

def show_val_accuracy_from_checkpoints(model_folder):
    import torch
    import glob
    import os

    ckpt_paths = sorted(glob.glob(os.path.join(model_folder, "*.pth")))

    print(f"\n📝 Summary of val_accuracy and Val-Class-Acc:")
    for path in ckpt_paths:
        checkpoint = torch.load(path, map_location="cpu")
        filename = os.path.basename(path)

        val_acc = checkpoint.get("val_accuracy", None)
        val_class_acc = checkpoint.get("Val-Class-Acc", None)

        if val_acc is not None and val_class_acc is not None:
            print(f"{filename}: val_accuracy = {val_acc:.4f}, Val-Class-Acc: {val_class_acc}")
        else:
            print(f"{filename}: ❌ val_accuracy or Val-Class-Acc not found")


In [ ]:
import os
import glob
import copy

# Reconstruct model with same config
model_template = BiGRUWithAttention(
    input_size=Number_features,
    hidden_size=64,
    output_size=num_classes,
    num_layers=4
)

# Prepare validation data
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
X_val_tensor = torch.tensor(X_val, dtype=torch.float32).to(device)
y_val_tensor = torch.tensor(y_val, dtype=torch.long).to(device)
criterion = nn.CrossEntropyLoss()

# Your model folder
model_folder = 'Class_Incremental_CL/Classif_Bi_Dir_GRU_Model/Trained_models/Baseline/Period_1/1st_try'
ckpt_paths = sorted(glob.glob(os.path.join(model_folder, "*.pth")))

# Loop through and update each model
for file_path in ckpt_paths:
    model_copy = copy.deepcopy(model_template).to(device)
    checkpoint = torch.load(file_path, map_location=device)
    model_copy.load_state_dict(checkpoint['model_state_dict'])

    add_val_class_acc(
        model_copy=model_copy,
        output_size=num_classes,
        criterion=criterion,
        X_val=X_val_tensor,
        y_val=y_val_tensor,
        file_path=file_path,
        model_checkpoint=checkpoint,
        batch_size=64
    )

show_val_accuracy_from_checkpoints(
    "Class_Incremental_CL/Classif_Bi_Dir_GRU_Model/Trained_models/Baseline/Period_1/1st_try"
)


C:\Users\james\AppData\Local\Temp\ipykernel_149096\3997419134.py:26: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(file_path, map_location=device)
C:

✅ Updated: BiGRUWithAttention_epoch_1550.pth | Acc: 0.9833 | Saved to: Class_Incremental_CL/Classif_Bi_Dir_GRU_Model/Trained_models/Baseline/Period_1/1st_try\BiGRUWithAttention_epoch_1550.pth
✅ Updated: BiGRUWithAttention_epoch_1986.pth | Acc: 0.9835 | Saved to: Class_Incremental_CL/Classif_Bi_Dir_GRU_Model/Trained_models/Baseline/Period_1/1st_try\BiGRUWithAttention_epoch_1986.pth
✅ Updated: BiGRUWithAttention_epoch_1987.pth | Acc: 0.9835 | Saved to: Class_Incremental_CL/Classif_Bi_Dir_GRU_Model/Trained_models/Baseline/Period_1/1st_try\BiGRUWithAttention_epoch_1987.pth
✅ Updated: BiGRUWithAttention_epoch_1989.pth | Acc: 0.9833 | Saved to: Class_Incremental_CL/Classif_Bi_Dir_GRU_Model/Trained_models/Baseline/Period_1/1st_try\BiGRUWithAttention_epoch_1989.pth
✅ Updated: BiGRUWithAttention_epoch_1999.pth | Acc: 0.9833 | Saved to: Class_Incremental_CL/Classif_Bi_Dir_GRU_Model/Trained_models/Baseline/Period_1/1st_try\BiGRUWithAttention_epoch_1999.pth
✅ Updated: BiGRUWithAttention_final.pth 

C:\Users\james\AppData\Local\Temp\ipykernel_149096\2472753690.py:66: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(path, map_location="cpu")


### Period 1 --> Training and saving in __*'2nd_try'*__ (BiGRUWithAttention, num_layers = 3) ---> Val acc = 98.31 %

In [ ]:
"""
- 'trend': Categorized trend values based on the detected phases:
    - 0: No trend
    - 1: Moderate negative trend
    - 2: Very strong negative trend
    - 3: Moderate positive trend
    - 4: Very strong positive trend
"""
with contextlib.redirect_stdout(open(os.devnull, 'w')):
    X_train, y_train, X_val, y_val, X_test, y_test, Number_features = process_and_return_splits(
        with_indicators_file_path = list_period_files_full_path[0], # Change 
        downsampled_data_minutes = downsampled_data_minutes,
        exclude_columns = exclude_columns,
        lower_threshold = lower_threshold,
        upper_threshold = upper_threshold,
        reverse_steps = reverse_steps,
        sequence_length = sequence_length,
        sliding_interval = sliding_interval,
        trends_to_keep = {0, 1}  # Default keeps all trends : {0, 1, 2, 3, 4}
        # trends_to_keep = {0, 1, 2, 3, 4}  # Default keeps all trends
    )

print(f"\nNumber_features = {Number_features}")

unique_classes = np.unique(y_val)
num_classes = len(unique_classes)
print(f"unique_classes = {unique_classes}")
print(f"num_classes = {num_classes}")


In [ ]:
# Model parameters
input_size = Number_features  # Number of features
hidden_size = 64  # Number of GRU units
output_size = num_classes # Must be dynamic, up to 5  # Number of trend classes (0, 15, 25, -15, -25)
num_layers = 3  # Number of GRU layers
num_epochs= 2000 # Number of epochs/ go through entire data
batch_size= 64 # How many sequences passed at once to the model
model_name = 'BiGRUWithAttention' # Name of the model to use for saving
best_results = [] # Initialize this outside the training function or at the beginning of training
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# Define a global stop signal
stop_signal_file = os.path.normpath(os.path.join('Class_Incremental_CL', 'Classif_Bi_Dir_GRU_Model/stop_training.txt'))  # Create this file to stop training
model_saving_folder = os.path.normpath(os.path.join('Class_Incremental_CL', "Classif_Bi_Dir_GRU_Model/Trained_models/Baseline/Period_1/2nd_try"))
ensure_folder(model_saving_folder)

# Instantiate the model
class_gru_model = BiGRUWithAttention(input_size, hidden_size, output_size, num_layers).to(device)

# Define the loss function, optimizer and scheduler
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(class_gru_model.parameters(), lr=0.0001) # lr=0.00005
# optimizer = optim.Adam(class_gru_model.parameters(), lr=0.001, weight_decay=1e-5)
# scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', factor=0.9, patience=10)

# train_and_validate(class_gru_model, output_size, criterion, optimizer, X_train_all, y_train_all, X_val_all, y_val_all, scheduler, 
train_and_validate(class_gru_model, output_size, criterion, optimizer, X_train, y_train, X_val, y_val, scheduler, 
                   False, num_epochs, batch_size, model_saving_folder, model_name, stop_signal_file)

for res in best_results:        
    print(f"Epoch {res['epoch']}/{num_epochs}, "
            f"Train Loss: {res['train_loss']:.4f}, " 
            f"Val Loss: {res['val_loss']:.4f}, "
            f"Val Accuracy: {res['val_accuracy'] * 100:.2f}%, "
            f"Model Path: {res['model_path']}")      
print(f"\nclass_gru_model: \n{class_gru_model}")

print(f"\nunique_classes = {unique_classes}")
print(f"num_classes = {num_classes}")
# del unique_classes, num_classes
for var in ["X_train", "y_train", "X_val", "y_val", "X_test", "y_test", "Number_features", "unique_classes", "num_classes"]:
    if var in locals():
        del locals()[var]


'train_and_validate' function started. 

Existing folder has been removed : Class_Incremental_CL\Classif_Bi_Dir_GRU_Model\Trained_models\2nd_try

y_train:
<class 'torch.Tensor'>
torch.int64
torch.Size([3634, 1000])
X_train:
<class 'torch.Tensor'>
torch.float32
torch.Size([3634, 1000, 7])

y_val:
<class 'torch.Tensor'>
torch.int64
torch.Size([454, 1000])
X_val:
<class 'torch.Tensor'>
torch.float32
torch.Size([454, 1000, 7])

Dataset Lengths:
Train Dataset Length: 3634
Validation Dataset Length: 454

DataLoader Batch Sizes:
Number of Batches in Train DataLoader: 57
Number of Batches in Validation DataLoader: 8

y_train Unique Values and Stats:
Unique values in y_train: tensor([0, 1], device='cuda:0')
y_train Min: 0, Max: 1

y_val Unique Values and Stats:
Unique values in y_val: tensor([0, 1], device='cuda:0')
y_val Min: 0, Max: 1

Device Info:
X_train Device: cuda:0
y_train Device: cuda:0
X_val Device: cuda:0
y_val Device: cuda:0

Epoch 1/2000, Train Loss: 0.010724814, Val Loss: 0.011819

### Period 2 --> Training and saving in __*'1st_try'*__ (BiGRUWithAttention, num_layers = 4, alpha = 0.5) ---> Val acc = 97.19 %
### __Val-Class-Acc = {0: '99.39%', 1: '97.61%', 2: '77.85%'} %__


In [16]:
"""
- 'trend': Categorized trend values based on the detected phases:
    - 0: No trend
    - 1: Moderate negative trend
    - 2: Very strong negative trend
    - 3: Moderate positive trend
    - 4: Very strong positive trend
"""
with contextlib.redirect_stdout(open(os.devnull, 'w')):
    X_train, y_train, X_val, y_val, X_test, y_test, Number_features = process_and_return_splits(
        with_indicators_file_path = list_period_files_full_path[1], # Change 
        downsampled_data_minutes = downsampled_data_minutes,
        exclude_columns = exclude_columns,
        lower_threshold = lower_threshold,
        upper_threshold = upper_threshold,
        reverse_steps = reverse_steps,
        sequence_length = sequence_length,
        sliding_interval = sliding_interval,
        trends_to_keep = {0, 1, 2}  # Default keeps all trends : {0, 1, 2, 3, 4}
        # trends_to_keep = {0, 1, 2, 3, 4}  # Default keeps all trends
    )

print(f"\nNumber_features = {Number_features}")

unique_classes = np.unique(y_val)
num_classes = len(unique_classes)
print(f"unique_classes = {unique_classes}")
print(f"num_classes = {num_classes}")



Number_features = 7
unique_classes = [0 1 2]
num_classes = 3


In [17]:
# Model parameters
stable_classes = [1] # <----------------------------<<< From period 1: I exclude class 0 because it will change in period 2
input_size = Number_features  # Number of features
hidden_size = 64  # Number of GRU units
output_size = num_classes # Must be dynamic, up to 5  # Number of trend classes (0, 15, 25, -15, -25)
num_layers = 4  # Number of GRU layers
dropout = 0.0
learning_rate = 0.0001
alpha = 0.5          # Weight for distillation loss
num_epochs= 2000 # Number of epochs/ go through entire data
batch_size= 64 # How many sequences passed at once to the model
model_name = 'BiGRUWithAttention' # Name of the model to use for saving
best_results = [] # Initialize this outside the training function or at the beginning of training
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# Define a global stop signal
stop_signal_file = os.path.normpath(os.path.join('Class_Incremental_CL', 'Classif_Bi_Dir_GRU_Model/stop_training.txt'))  # Create this file to stop training
model_saving_folder = os.path.normpath(os.path.join('Class_Incremental_CL', "Classif_Bi_Dir_GRU_Model/Trained_models/MSE_Distillation_CIL/Period_2/1st_try"))
ensure_folder(model_saving_folder)

# Instantiate the student model
student_model = BiGRUWithAttention(input_size, hidden_size, output_size, num_layers, dropout).to(device)

# Instantiate and load the teacher model
teacher_model = BiGRUWithAttention(input_size, hidden_size, output_size - 1, num_layers, dropout).to(device)
#-------------------------------------------------------------------------
best_model_path = r"Class_Incremental_CL\Classif_Bi_Dir_GRU_Model\Trained_models\Baseline\Period_1\1st_try\BiGRUWithAttention_epoch_1987.pth"
#----------------------------------------------------------------------
# Initialize the list to store results across runs
track_across_runs = []
#----------------------------------------------------------------------
#-------------------------------------------------------------------------
teacher_checkpoint_path = os.path.normpath(best_model_path)
# teacher_checkpoint = torch.load("teacher_model.pth", map_location=device)
teacher_checkpoint = torch.load(teacher_checkpoint_path, map_location=device, weights_only=True)
# print(f"\n{teacher_checkpoint}\n")
teacher_model.load_state_dict(teacher_checkpoint['model_state_dict'])
del teacher_checkpoint
gc.collect()
print(f"Loaded teacher model from: \n\t{teacher_checkpoint_path}")
print(f"\n{teacher_model}\n")

# Define the loss function, optimizer and scheduler
criterion = nn.CrossEntropyLoss()
# Optimizer: only train parameters of the student model (LoRA parameters and any unfrozen ones).
optimizer = optim.Adam(student_model.parameters(), lr=learning_rate) # lr=0.00005
# optimizer = optim.Adam(class_gru_model.parameters(), lr=0.001, weight_decay=1e-5)
# scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', factor=0.9, patience=10)

train_and_validate_MSE(student_model, teacher_model, stable_classes, output_size, criterion, optimizer, 
                       X_train, y_train, X_val, y_val, scheduler, False, num_epochs, batch_size, 
                       alpha, model_saving_folder, model_name, stop_signal_file)

#----------------------------------------------------------------------
# Append only the best result (already at index 0)
track_across_runs.append(best_results[0])
#----------------------------------------------------------------------

for res in best_results:        
    print(f"Epoch {res['epoch']}/{num_epochs}, "
            f"Train Loss: {res['train_loss']:.4f}, " 
            f"Val Loss: {res['val_loss']:.4f}, "
            f"Val Accuracy: {res['val_accuracy'] * 100:.2f}%, "
            f"Model Path: {res['model_path']}")      
print(f"\nclass_gru_model (student_model): \n{student_model}")

print(f"\nunique_classes = {unique_classes}")
print(f"num_classes = {num_classes}")
# del unique_classes, num_classes
for var in ["X_train", "y_train", "X_val", "y_val", "X_test", "y_test", "Number_features", "unique_classes", "num_classes"]:
    if var in locals():
        del locals()[var]


Loaded teacher model from: 
	Class_Incremental_CL\Classif_Bi_Dir_GRU_Model\Trained_models\Baseline\Period_1\1st_try\BiGRUWithAttention_epoch_1987.pth

BiGRUWithAttention(
  (gru): GRU(7, 64, num_layers=4, batch_first=True, bidirectional=True)
  (attention_fc): Linear(in_features=128, out_features=128, bias=True)
  (fc): Linear(in_features=128, out_features=2, bias=True)
  (dropout): Dropout(p=0.0, inplace=False)
)

'train_and_validate' function started. 

Existing folder has been removed : Class_Incremental_CL\Classif_Bi_Dir_GRU_Model\Trained_models\MSE_Distillation_CIL\Period_2\1st_try

y_train:
<class 'torch.Tensor'>
torch.int64
torch.Size([3634, 1000])
X_train:
<class 'torch.Tensor'>
torch.float32
torch.Size([3634, 1000, 7])

y_val:
<class 'torch.Tensor'>
torch.int64
torch.Size([454, 1000])
X_val:
<class 'torch.Tensor'>
torch.float32
torch.Size([454, 1000, 7])

Dataset Lengths:
Train Dataset Length: 3634
Validation Dataset Length: 454

DataLoader Batch Sizes:
Number of Batches in Tr

In [18]:
print(best_model_path)

Class_Incremental_CL\Classif_Bi_Dir_GRU_Model\Trained_models\Baseline\Period_1\1st_try\BiGRUWithAttention_epoch_1987.pth


### Period 2 --> Training and saving in __*'2nd_try'*__ (BiGRUWithAttention, num_layers = 4, alpha = 0.4) ---> Val acc = 96.76 %
### __Val-Class-Acc = {0: '99.81%', 1: '96.10%', 2: '83.35%'} %__

In [23]:
"""
- 'trend': Categorized trend values based on the detected phases:
    - 0: No trend
    - 1: Moderate negative trend
    - 2: Very strong negative trend
    - 3: Moderate positive trend
    - 4: Very strong positive trend
"""
with contextlib.redirect_stdout(open(os.devnull, 'w')):
    X_train, y_train, X_val, y_val, X_test, y_test, Number_features = process_and_return_splits(
        with_indicators_file_path = list_period_files_full_path[1], # Change 
        downsampled_data_minutes = downsampled_data_minutes,
        exclude_columns = exclude_columns,
        lower_threshold = lower_threshold,
        upper_threshold = upper_threshold,
        reverse_steps = reverse_steps,
        sequence_length = sequence_length,
        sliding_interval = sliding_interval,
        trends_to_keep = {0, 1, 2}  # Default keeps all trends : {0, 1, 2, 3, 4}
        # trends_to_keep = {0, 1, 2, 3, 4}  # Default keeps all trends
    )

print(f"\nNumber_features = {Number_features}")

unique_classes = np.unique(y_val)
num_classes = len(unique_classes)
print(f"unique_classes = {unique_classes}")
print(f"num_classes = {num_classes}")



Number_features = 7
unique_classes = [0 1 2]
num_classes = 3


In [26]:
# Model parameters
stable_classes = [1] # <----------------------------<<< From period 1: I exclude class 0 because it will change in period 2
input_size = Number_features  # Number of features
hidden_size = 64  # Number of GRU units
output_size = num_classes # Must be dynamic, up to 5  # Number of trend classes (0, 15, 25, -15, -25)
num_layers = 4  # Number of GRU layers
dropout = 0.0
learning_rate = 0.0001
alpha = 0.4          # Weight for distillation loss
num_epochs= 2000 # Number of epochs/ go through entire data
batch_size= 64 # How many sequences passed at once to the model
model_name = 'BiGRUWithAttention' # Name of the model to use for saving
best_results = [] # Initialize this outside the training function or at the beginning of training
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# Define a global stop signal
stop_signal_file = os.path.normpath(os.path.join('Class_Incremental_CL', 'Classif_Bi_Dir_GRU_Model/stop_training.txt'))  # Create this file to stop training
model_saving_folder = os.path.normpath(os.path.join('Class_Incremental_CL', "Classif_Bi_Dir_GRU_Model/Trained_models/MSE_Distillation_CIL/Period_2/2nd_try"))
ensure_folder(model_saving_folder)

# Instantiate the student model
student_model = BiGRUWithAttention(input_size, hidden_size, output_size, num_layers, dropout).to(device)

# Instantiate and load the teacher model
teacher_model = BiGRUWithAttention(input_size, hidden_size, output_size - 1, num_layers, dropout).to(device)
#-------------------------------------------------------------------------
best_model_path = r"Class_Incremental_CL\Classif_Bi_Dir_GRU_Model\Trained_models\Baseline\Period_1\1st_try\BiGRUWithAttention_epoch_1987.pth"
#----------------------------------------------------------------------
# Initialize the list to store results across runs
track_across_runs = []
#----------------------------------------------------------------------
#-------------------------------------------------------------------------
teacher_checkpoint_path = os.path.normpath(best_model_path)
# teacher_checkpoint = torch.load("teacher_model.pth", map_location=device)
teacher_checkpoint = torch.load(teacher_checkpoint_path, map_location=device, weights_only=True)
# print(f"\n{teacher_checkpoint}\n")
teacher_model.load_state_dict(teacher_checkpoint['model_state_dict'])
del teacher_checkpoint
gc.collect()
print(f"Loaded teacher model from: \n\t{teacher_checkpoint_path}")
print(f"\n{teacher_model}\n")

# Define the loss function, optimizer and scheduler
criterion = nn.CrossEntropyLoss()
# Optimizer: only train parameters of the student model (LoRA parameters and any unfrozen ones).
optimizer = optim.Adam(student_model.parameters(), lr=learning_rate) # lr=0.00005
# optimizer = optim.Adam(class_gru_model.parameters(), lr=0.001, weight_decay=1e-5)
# scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', factor=0.9, patience=10)

train_and_validate_MSE(student_model, teacher_model, stable_classes, output_size, criterion, optimizer, 
                       X_train, y_train, X_val, y_val, scheduler, False, num_epochs, batch_size, 
                       alpha, model_saving_folder, model_name, stop_signal_file)

#----------------------------------------------------------------------
# Append only the best result (already at index 0)
track_across_runs.append(best_results[0])
#----------------------------------------------------------------------

for res in best_results:        
    print(f"Epoch {res['epoch']}/{num_epochs}, "
            f"Train Loss: {res['train_loss']:.4f}, " 
            f"Val Loss: {res['val_loss']:.4f}, "
            f"Val Accuracy: {res['val_accuracy'] * 100:.2f}%, "
            f"Model Path: {res['model_path']}")      
print(f"\nclass_gru_model (student_model): \n{student_model}")

print(f"\nunique_classes = {unique_classes}")
print(f"num_classes = {num_classes}")
# del unique_classes, num_classes
for var in ["X_train", "y_train", "X_val", "y_val", "X_test", "y_test", "Number_features", "unique_classes", "num_classes"]:
    if var in locals():
        del locals()[var]


Loaded teacher model from: 
	Class_Incremental_CL\Classif_Bi_Dir_GRU_Model\Trained_models\Baseline\Period_1\1st_try\BiGRUWithAttention_epoch_1987.pth

BiGRUWithAttention(
  (gru): GRU(7, 64, num_layers=4, batch_first=True, bidirectional=True)
  (attention_fc): Linear(in_features=128, out_features=128, bias=True)
  (fc): Linear(in_features=128, out_features=2, bias=True)
  (dropout): Dropout(p=0.0, inplace=False)
)

'train_and_validate' function started. 

Existing folder has been removed : Class_Incremental_CL\Classif_Bi_Dir_GRU_Model\Trained_models\MSE_Distillation_CIL\Period_2\2nd_try

y_train:
<class 'torch.Tensor'>
torch.int64
torch.Size([3634, 1000])
X_train:
<class 'torch.Tensor'>
torch.float32
torch.Size([3634, 1000, 7])

y_val:
<class 'torch.Tensor'>
torch.int64
torch.Size([454, 1000])
X_val:
<class 'torch.Tensor'>
torch.float32
torch.Size([454, 1000, 7])

Dataset Lengths:
Train Dataset Length: 3634
Validation Dataset Length: 454

DataLoader Batch Sizes:
Number of Batches in Tr

### Period 2 --> Training and saving in __*'3rd_try'*__ (BiGRUWithAttention, num_layers = 4, alpha = 0.3) ---> Val acc = 96.98 %
### __Val-Class-Acc = {0: '99.49%', 1: '96.16%', 2: '84.82%'} %__

In [31]:
"""
- 'trend': Categorized trend values based on the detected phases:
    - 0: No trend
    - 1: Moderate negative trend
    - 2: Very strong negative trend
    - 3: Moderate positive trend
    - 4: Very strong positive trend
"""
with contextlib.redirect_stdout(open(os.devnull, 'w')):
    X_train, y_train, X_val, y_val, X_test, y_test, Number_features = process_and_return_splits(
        with_indicators_file_path = list_period_files_full_path[1], # Change 
        downsampled_data_minutes = downsampled_data_minutes,
        exclude_columns = exclude_columns,
        lower_threshold = lower_threshold,
        upper_threshold = upper_threshold,
        reverse_steps = reverse_steps,
        sequence_length = sequence_length,
        sliding_interval = sliding_interval,
        trends_to_keep = {0, 1, 2}  # Default keeps all trends : {0, 1, 2, 3, 4}
        # trends_to_keep = {0, 1, 2, 3, 4}  # Default keeps all trends
    )

print(f"\nNumber_features = {Number_features}")

unique_classes = np.unique(y_val)
num_classes = len(unique_classes)
print(f"unique_classes = {unique_classes}")
print(f"num_classes = {num_classes}")



Number_features = 7
unique_classes = [0 1 2]
num_classes = 3


In [32]:
# Model parameters
stable_classes = [1] # <----------------------------<<< From period 1: I exclude class 0 because it will change in period 2
input_size = Number_features  # Number of features
hidden_size = 64  # Number of GRU units
output_size = num_classes # Must be dynamic, up to 5  # Number of trend classes (0, 15, 25, -15, -25)
num_layers = 4  # Number of GRU layers
dropout = 0.0
learning_rate = 0.0001
alpha = 0.3          # Weight for distillation loss
num_epochs= 2000 # Number of epochs/ go through entire data
batch_size= 64 # How many sequences passed at once to the model
model_name = 'BiGRUWithAttention' # Name of the model to use for saving
best_results = [] # Initialize this outside the training function or at the beginning of training
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# Define a global stop signal
stop_signal_file = os.path.normpath(os.path.join('Class_Incremental_CL', 'Classif_Bi_Dir_GRU_Model/stop_training.txt'))  # Create this file to stop training
model_saving_folder = os.path.normpath(os.path.join('Class_Incremental_CL', "Classif_Bi_Dir_GRU_Model/Trained_models/MSE_Distillation_CIL/Period_2/3rd_try"))
ensure_folder(model_saving_folder)

# Instantiate the student model
student_model = BiGRUWithAttention(input_size, hidden_size, output_size, num_layers, dropout).to(device)

# Instantiate and load the teacher model
teacher_model = BiGRUWithAttention(input_size, hidden_size, output_size - 1, num_layers, dropout).to(device)
#-------------------------------------------------------------------------
best_model_path = r"Class_Incremental_CL\Classif_Bi_Dir_GRU_Model\Trained_models\Baseline\Period_1\1st_try\BiGRUWithAttention_epoch_1987.pth"
#----------------------------------------------------------------------
# Initialize the list to store results across runs
track_across_runs = []
#----------------------------------------------------------------------
#-------------------------------------------------------------------------
teacher_checkpoint_path = os.path.normpath(best_model_path)
# teacher_checkpoint = torch.load("teacher_model.pth", map_location=device)
teacher_checkpoint = torch.load(teacher_checkpoint_path, map_location=device, weights_only=True)
# print(f"\n{teacher_checkpoint}\n")
teacher_model.load_state_dict(teacher_checkpoint['model_state_dict'])
del teacher_checkpoint
gc.collect()
print(f"Loaded teacher model from: \n\t{teacher_checkpoint_path}")
print(f"\n{teacher_model}\n")

# Define the loss function, optimizer and scheduler
criterion = nn.CrossEntropyLoss()
# Optimizer: only train parameters of the student model (LoRA parameters and any unfrozen ones).
optimizer = optim.Adam(student_model.parameters(), lr=learning_rate) # lr=0.00005
# optimizer = optim.Adam(class_gru_model.parameters(), lr=0.001, weight_decay=1e-5)
# scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', factor=0.9, patience=10)

train_and_validate_MSE(student_model, teacher_model, stable_classes, output_size, criterion, optimizer, 
                       X_train, y_train, X_val, y_val, scheduler, False, num_epochs, batch_size, 
                       alpha, model_saving_folder, model_name, stop_signal_file)

#----------------------------------------------------------------------
# Append only the best result (already at index 0)
track_across_runs.append(best_results[0])
#----------------------------------------------------------------------

for res in best_results:        
    print(f"Epoch {res['epoch']}/{num_epochs}, "
            f"Train Loss: {res['train_loss']:.4f}, " 
            f"Val Loss: {res['val_loss']:.4f}, "
            f"Val Accuracy: {res['val_accuracy'] * 100:.2f}%, "
            f"Model Path: {res['model_path']}")      
print(f"\nclass_gru_model (student_model): \n{student_model}")

print(f"\nunique_classes = {unique_classes}")
print(f"num_classes = {num_classes}")
# del unique_classes, num_classes
for var in ["X_train", "y_train", "X_val", "y_val", "X_test", "y_test", "Number_features", "unique_classes", "num_classes"]:
    if var in locals():
        del locals()[var]


Loaded teacher model from: 
	Class_Incremental_CL\Classif_Bi_Dir_GRU_Model\Trained_models\Baseline\Period_1\1st_try\BiGRUWithAttention_epoch_1987.pth

BiGRUWithAttention(
  (gru): GRU(7, 64, num_layers=4, batch_first=True, bidirectional=True)
  (attention_fc): Linear(in_features=128, out_features=128, bias=True)
  (fc): Linear(in_features=128, out_features=2, bias=True)
  (dropout): Dropout(p=0.0, inplace=False)
)

'train_and_validate' function started. 

Existing folder has been removed : Class_Incremental_CL\Classif_Bi_Dir_GRU_Model\Trained_models\MSE_Distillation_CIL\Period_2\3rd_try

y_train:
<class 'torch.Tensor'>
torch.int64
torch.Size([3634, 1000])
X_train:
<class 'torch.Tensor'>
torch.float32
torch.Size([3634, 1000, 7])

y_val:
<class 'torch.Tensor'>
torch.int64
torch.Size([454, 1000])
X_val:
<class 'torch.Tensor'>
torch.float32
torch.Size([454, 1000, 7])

Dataset Lengths:
Train Dataset Length: 3634
Validation Dataset Length: 454

DataLoader Batch Sizes:
Number of Batches in Tr

### Period 2 --> Training and saving in __*'4th_try'*__ (BiGRUWithAttention, num_layers = 4, alpha = 0.6) ---> Val acc = 97.13 %
### __Val-Class-Acc = {0: '99.53%', 1: '95.21%', 2: '81.89%'} %__

In [37]:
"""
- 'trend': Categorized trend values based on the detected phases:
    - 0: No trend
    - 1: Moderate negative trend
    - 2: Very strong negative trend
    - 3: Moderate positive trend
    - 4: Very strong positive trend
"""
with contextlib.redirect_stdout(open(os.devnull, 'w')):
    X_train, y_train, X_val, y_val, X_test, y_test, Number_features = process_and_return_splits(
        with_indicators_file_path = list_period_files_full_path[1], # Change 
        downsampled_data_minutes = downsampled_data_minutes,
        exclude_columns = exclude_columns,
        lower_threshold = lower_threshold,
        upper_threshold = upper_threshold,
        reverse_steps = reverse_steps,
        sequence_length = sequence_length,
        sliding_interval = sliding_interval,
        trends_to_keep = {0, 1, 2}  # Default keeps all trends : {0, 1, 2, 3, 4}
        # trends_to_keep = {0, 1, 2, 3, 4}  # Default keeps all trends
    )

print(f"\nNumber_features = {Number_features}")

unique_classes = np.unique(y_val)
num_classes = len(unique_classes)
print(f"unique_classes = {unique_classes}")
print(f"num_classes = {num_classes}")



Number_features = 7
unique_classes = [0 1 2]
num_classes = 3


In [38]:
# Model parameters
stable_classes = [1] # <----------------------------<<< From period 1: I exclude class 0 because it will change in period 2
input_size = Number_features  # Number of features
hidden_size = 64  # Number of GRU units
output_size = num_classes # Must be dynamic, up to 5  # Number of trend classes (0, 15, 25, -15, -25)
num_layers = 4  # Number of GRU layers
dropout = 0.0
learning_rate = 0.0001
alpha = 0.6          # Weight for distillation loss
num_epochs= 2000 # Number of epochs/ go through entire data
batch_size= 64 # How many sequences passed at once to the model
model_name = 'BiGRUWithAttention' # Name of the model to use for saving
best_results = [] # Initialize this outside the training function or at the beginning of training
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# Define a global stop signal
stop_signal_file = os.path.normpath(os.path.join('Class_Incremental_CL', 'Classif_Bi_Dir_GRU_Model/stop_training.txt'))  # Create this file to stop training
model_saving_folder = os.path.normpath(os.path.join('Class_Incremental_CL', "Classif_Bi_Dir_GRU_Model/Trained_models/MSE_Distillation_CIL/Period_2/4th_try"))
ensure_folder(model_saving_folder)

# Instantiate the student model
student_model = BiGRUWithAttention(input_size, hidden_size, output_size, num_layers, dropout).to(device)

# Instantiate and load the teacher model
teacher_model = BiGRUWithAttention(input_size, hidden_size, output_size - 1, num_layers, dropout).to(device)
#-------------------------------------------------------------------------
best_model_path = r"Class_Incremental_CL\Classif_Bi_Dir_GRU_Model\Trained_models\Baseline\Period_1\1st_try\BiGRUWithAttention_epoch_1987.pth"
#----------------------------------------------------------------------
# Initialize the list to store results across runs
track_across_runs = []
#----------------------------------------------------------------------
#-------------------------------------------------------------------------
teacher_checkpoint_path = os.path.normpath(best_model_path)
# teacher_checkpoint = torch.load("teacher_model.pth", map_location=device)
teacher_checkpoint = torch.load(teacher_checkpoint_path, map_location=device, weights_only=True)
# print(f"\n{teacher_checkpoint}\n")
teacher_model.load_state_dict(teacher_checkpoint['model_state_dict'])
del teacher_checkpoint
gc.collect()
print(f"Loaded teacher model from: \n\t{teacher_checkpoint_path}")
print(f"\n{teacher_model}\n")

# Define the loss function, optimizer and scheduler
criterion = nn.CrossEntropyLoss()
# Optimizer: only train parameters of the student model (LoRA parameters and any unfrozen ones).
optimizer = optim.Adam(student_model.parameters(), lr=learning_rate) # lr=0.00005
# optimizer = optim.Adam(class_gru_model.parameters(), lr=0.001, weight_decay=1e-5)
# scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', factor=0.9, patience=10)

train_and_validate_MSE(student_model, teacher_model, stable_classes, output_size, criterion, optimizer, 
                       X_train, y_train, X_val, y_val, scheduler, False, num_epochs, batch_size, 
                       alpha, model_saving_folder, model_name, stop_signal_file)

#----------------------------------------------------------------------
# Append only the best result (already at index 0)
track_across_runs.append(best_results[0])
#----------------------------------------------------------------------

for res in best_results:        
    print(f"Epoch {res['epoch']}/{num_epochs}, "
            f"Train Loss: {res['train_loss']:.4f}, " 
            f"Val Loss: {res['val_loss']:.4f}, "
            f"Val Accuracy: {res['val_accuracy'] * 100:.2f}%, "
            f"Model Path: {res['model_path']}")      
print(f"\nclass_gru_model (student_model): \n{student_model}")

print(f"\nunique_classes = {unique_classes}")
print(f"num_classes = {num_classes}")
# del unique_classes, num_classes
for var in ["X_train", "y_train", "X_val", "y_val", "X_test", "y_test", "Number_features", "unique_classes", "num_classes"]:
    if var in locals():
        del locals()[var]


Loaded teacher model from: 
	Class_Incremental_CL\Classif_Bi_Dir_GRU_Model\Trained_models\Baseline\Period_1\1st_try\BiGRUWithAttention_epoch_1987.pth

BiGRUWithAttention(
  (gru): GRU(7, 64, num_layers=4, batch_first=True, bidirectional=True)
  (attention_fc): Linear(in_features=128, out_features=128, bias=True)
  (fc): Linear(in_features=128, out_features=2, bias=True)
  (dropout): Dropout(p=0.0, inplace=False)
)

'train_and_validate' function started. 

Existing folder has been removed : Class_Incremental_CL\Classif_Bi_Dir_GRU_Model\Trained_models\MSE_Distillation_CIL\Period_2\4th_try

y_train:
<class 'torch.Tensor'>
torch.int64
torch.Size([3634, 1000])
X_train:
<class 'torch.Tensor'>
torch.float32
torch.Size([3634, 1000, 7])

y_val:
<class 'torch.Tensor'>
torch.int64
torch.Size([454, 1000])
X_val:
<class 'torch.Tensor'>
torch.float32
torch.Size([454, 1000, 7])

Dataset Lengths:
Train Dataset Length: 3634
Validation Dataset Length: 454

DataLoader Batch Sizes:
Number of Batches in Tr

### Period 2 --> Training and saving in __*'5th_try'*__ (BiGRUWithAttention, num_layers = 4, alpha = 0.1) ---> Val acc = 98.17 %
### __Val-Class-Acc = {0: '99.88%', 1: '96.31%', 2: '82.99%'} %__

In [43]:
"""
- 'trend': Categorized trend values based on the detected phases:
    - 0: No trend
    - 1: Moderate negative trend
    - 2: Very strong negative trend
    - 3: Moderate positive trend
    - 4: Very strong positive trend
"""
with contextlib.redirect_stdout(open(os.devnull, 'w')):
    X_train, y_train, X_val, y_val, X_test, y_test, Number_features = process_and_return_splits(
        with_indicators_file_path = list_period_files_full_path[1], # Change 
        downsampled_data_minutes = downsampled_data_minutes,
        exclude_columns = exclude_columns,
        lower_threshold = lower_threshold,
        upper_threshold = upper_threshold,
        reverse_steps = reverse_steps,
        sequence_length = sequence_length,
        sliding_interval = sliding_interval,
        trends_to_keep = {0, 1, 2}  # Default keeps all trends : {0, 1, 2, 3, 4}
        # trends_to_keep = {0, 1, 2, 3, 4}  # Default keeps all trends
    )

print(f"\nNumber_features = {Number_features}")

unique_classes = np.unique(y_val)
num_classes = len(unique_classes)
print(f"unique_classes = {unique_classes}")
print(f"num_classes = {num_classes}")



Number_features = 7
unique_classes = [0 1 2]
num_classes = 3


In [44]:
# Model parameters
stable_classes = [1] # <----------------------------<<< From period 1: I exclude class 0 because it will change in period 2
input_size = Number_features  # Number of features
hidden_size = 64  # Number of GRU units
output_size = num_classes # Must be dynamic, up to 5  # Number of trend classes (0, 15, 25, -15, -25)
num_layers = 4  # Number of GRU layers
dropout = 0.0
learning_rate = 0.0001
alpha = 0.1         # Weight for distillation loss
num_epochs= 2000 # Number of epochs/ go through entire data
batch_size= 64 # How many sequences passed at once to the model
model_name = 'BiGRUWithAttention' # Name of the model to use for saving
best_results = [] # Initialize this outside the training function or at the beginning of training
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# Define a global stop signal
stop_signal_file = os.path.normpath(os.path.join('Class_Incremental_CL', 'Classif_Bi_Dir_GRU_Model/stop_training.txt'))  # Create this file to stop training
model_saving_folder = os.path.normpath(os.path.join('Class_Incremental_CL', "Classif_Bi_Dir_GRU_Model/Trained_models/MSE_Distillation_CIL/Period_2/5th_try"))
ensure_folder(model_saving_folder)

# Instantiate the student model
student_model = BiGRUWithAttention(input_size, hidden_size, output_size, num_layers, dropout).to(device)

# Instantiate and load the teacher model
teacher_model = BiGRUWithAttention(input_size, hidden_size, output_size - 1, num_layers, dropout).to(device)
#-------------------------------------------------------------------------
best_model_path = r"Class_Incremental_CL\Classif_Bi_Dir_GRU_Model\Trained_models\Baseline\Period_1\1st_try\BiGRUWithAttention_epoch_1987.pth"
#----------------------------------------------------------------------
# Initialize the list to store results across runs
track_across_runs = []
#----------------------------------------------------------------------
#-------------------------------------------------------------------------
teacher_checkpoint_path = os.path.normpath(best_model_path)
# teacher_checkpoint = torch.load("teacher_model.pth", map_location=device)
teacher_checkpoint = torch.load(teacher_checkpoint_path, map_location=device, weights_only=True)
# print(f"\n{teacher_checkpoint}\n")
teacher_model.load_state_dict(teacher_checkpoint['model_state_dict'])
del teacher_checkpoint
gc.collect()
print(f"Loaded teacher model from: \n\t{teacher_checkpoint_path}")
print(f"\n{teacher_model}\n")

# Define the loss function, optimizer and scheduler
criterion = nn.CrossEntropyLoss()
# Optimizer: only train parameters of the student model (LoRA parameters and any unfrozen ones).
optimizer = optim.Adam(student_model.parameters(), lr=learning_rate) # lr=0.00005
# optimizer = optim.Adam(class_gru_model.parameters(), lr=0.001, weight_decay=1e-5)
# scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', factor=0.9, patience=10)

train_and_validate_MSE(student_model, teacher_model, stable_classes, output_size, criterion, optimizer, 
                       X_train, y_train, X_val, y_val, scheduler, False, num_epochs, batch_size, 
                       alpha, model_saving_folder, model_name, stop_signal_file)

#----------------------------------------------------------------------
# Append only the best result (already at index 0)
track_across_runs.append(best_results[0])
#----------------------------------------------------------------------

for res in best_results:        
    print(f"Epoch {res['epoch']}/{num_epochs}, "
            f"Train Loss: {res['train_loss']:.4f}, " 
            f"Val Loss: {res['val_loss']:.4f}, "
            f"Val Accuracy: {res['val_accuracy'] * 100:.2f}%, "
            f"Model Path: {res['model_path']}")      
print(f"\nclass_gru_model (student_model): \n{student_model}")

print(f"\nunique_classes = {unique_classes}")
print(f"num_classes = {num_classes}")
# del unique_classes, num_classes
for var in ["X_train", "y_train", "X_val", "y_val", "X_test", "y_test", "Number_features", "unique_classes", "num_classes"]:
    if var in locals():
        del locals()[var]


Loaded teacher model from: 
	Class_Incremental_CL\Classif_Bi_Dir_GRU_Model\Trained_models\Baseline\Period_1\1st_try\BiGRUWithAttention_epoch_1987.pth

BiGRUWithAttention(
  (gru): GRU(7, 64, num_layers=4, batch_first=True, bidirectional=True)
  (attention_fc): Linear(in_features=128, out_features=128, bias=True)
  (fc): Linear(in_features=128, out_features=2, bias=True)
  (dropout): Dropout(p=0.0, inplace=False)
)

'train_and_validate' function started. 

Existing folder has been removed : Class_Incremental_CL\Classif_Bi_Dir_GRU_Model\Trained_models\MSE_Distillation_CIL\Period_2\5th_try

y_train:
<class 'torch.Tensor'>
torch.int64
torch.Size([3634, 1000])
X_train:
<class 'torch.Tensor'>
torch.float32
torch.Size([3634, 1000, 7])

y_val:
<class 'torch.Tensor'>
torch.int64
torch.Size([454, 1000])
X_val:
<class 'torch.Tensor'>
torch.float32
torch.Size([454, 1000, 7])

Dataset Lengths:
Train Dataset Length: 3634
Validation Dataset Length: 454

DataLoader Batch Sizes:
Number of Batches in Tr

### Period 2 --> Training and saving in __*'6th_try'*__ (BiGRUWithAttention, num_layers = 4, alpha = 0.2) ---> Val acc = 98.04 %
### Val-Class-Acc = {0: '99.33%', 1: '97.66%', 2: '93.01%'}

In [16]:
"""
- 'trend': Categorized trend values based on the detected phases:
    - 0: No trend
    - 1: Moderate negative trend
    - 2: Very strong negative trend
    - 3: Moderate positive trend
    - 4: Very strong positive trend
"""
with contextlib.redirect_stdout(open(os.devnull, 'w')):
    X_train, y_train, X_val, y_val, X_test, y_test, Number_features = process_and_return_splits(
        with_indicators_file_path = list_period_files_full_path[1], # Change 
        downsampled_data_minutes = downsampled_data_minutes,
        exclude_columns = exclude_columns,
        lower_threshold = lower_threshold,
        upper_threshold = upper_threshold,
        reverse_steps = reverse_steps,
        sequence_length = sequence_length,
        sliding_interval = sliding_interval,
        trends_to_keep = {0, 1, 2}  # Default keeps all trends : {0, 1, 2, 3, 4}
        # trends_to_keep = {0, 1, 2, 3, 4}  # Default keeps all trends
    )

print(f"\nNumber_features = {Number_features}")

unique_classes = np.unique(y_val)
num_classes = len(unique_classes)
print(f"unique_classes = {unique_classes}")
print(f"num_classes = {num_classes}")



Number_features = 7
unique_classes = [0 1 2]
num_classes = 3


In [17]:
# Model parameters
stable_classes = [1] # <----------------------------<<< From period 1: I exclude class 0 because it will change in period 2
input_size = Number_features  # Number of features
hidden_size = 64  # Number of GRU units
output_size = num_classes # Must be dynamic, up to 5  # Number of trend classes (0, 15, 25, -15, -25)
num_layers = 4  # Number of GRU layers
dropout = 0.0
learning_rate = 0.0001
alpha = 0.2         # Weight for distillation loss
num_epochs= 2000 # Number of epochs/ go through entire data
batch_size= 64 # How many sequences passed at once to the model
model_name = 'BiGRUWithAttention' # Name of the model to use for saving
best_results = [] # Initialize this outside the training function or at the beginning of training
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# Define a global stop signal
stop_signal_file = os.path.normpath(os.path.join('Class_Incremental_CL', 'Classif_Bi_Dir_GRU_Model/stop_training.txt'))  # Create this file to stop training
model_saving_folder = os.path.normpath(os.path.join('Class_Incremental_CL', "Classif_Bi_Dir_GRU_Model/Trained_models/MSE_Distillation_CIL/Period_2/6th_try"))
ensure_folder(model_saving_folder)

# Instantiate the student model
student_model = BiGRUWithAttention(input_size, hidden_size, output_size, num_layers, dropout).to(device)

# Instantiate and load the teacher model
teacher_model = BiGRUWithAttention(input_size, hidden_size, output_size - 1, num_layers, dropout).to(device)
#-------------------------------------------------------------------------
best_model_path = r"Class_Incremental_CL\Classif_Bi_Dir_GRU_Model\Trained_models\Baseline\Period_1\1st_try\BiGRUWithAttention_epoch_1987.pth"
#----------------------------------------------------------------------
# Initialize the list to store results across runs
track_across_runs = []
#----------------------------------------------------------------------
#-------------------------------------------------------------------------
teacher_checkpoint_path = os.path.normpath(best_model_path)
# teacher_checkpoint = torch.load("teacher_model.pth", map_location=device)
teacher_checkpoint = torch.load(teacher_checkpoint_path, map_location=device, weights_only=True)
# print(f"\n{teacher_checkpoint}\n")
teacher_model.load_state_dict(teacher_checkpoint['model_state_dict'])
del teacher_checkpoint
gc.collect()
print(f"Loaded teacher model from: \n\t{teacher_checkpoint_path}")
print(f"\n{teacher_model}\n")

# Define the loss function, optimizer and scheduler
criterion = nn.CrossEntropyLoss()
# Optimizer: only train parameters of the student model (LoRA parameters and any unfrozen ones).
optimizer = optim.Adam(student_model.parameters(), lr=learning_rate) # lr=0.00005
# optimizer = optim.Adam(class_gru_model.parameters(), lr=0.001, weight_decay=1e-5)
# scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', factor=0.9, patience=10)

train_and_validate_MSE(student_model, teacher_model, stable_classes, output_size, criterion, optimizer, 
                       X_train, y_train, X_val, y_val, scheduler, False, num_epochs, batch_size, 
                       alpha, model_saving_folder, model_name, stop_signal_file)

#----------------------------------------------------------------------
# Append only the best result (already at index 0)
track_across_runs.append(best_results[0])
#----------------------------------------------------------------------

for res in best_results:        
    print(f"Epoch {res['epoch']}/{num_epochs}, "
            f"Train Loss: {res['train_loss']:.4f}, " 
            f"Val Loss: {res['val_loss']:.4f}, "
            f"Val Accuracy: {res['val_accuracy'] * 100:.2f}%, "
            f"Model Path: {res['model_path']}")      
print(f"\nclass_gru_model (student_model): \n{student_model}")

print(f"\nunique_classes = {unique_classes}")
print(f"num_classes = {num_classes}")
# del unique_classes, num_classes
for var in ["X_train", "y_train", "X_val", "y_val", "X_test", "y_test", "Number_features", "unique_classes", "num_classes"]:
    if var in locals():
        del locals()[var]


Loaded teacher model from: 
	Class_Incremental_CL\Classif_Bi_Dir_GRU_Model\Trained_models\Baseline\Period_1\1st_try\BiGRUWithAttention_epoch_1987.pth

BiGRUWithAttention(
  (gru): GRU(7, 64, num_layers=4, batch_first=True, bidirectional=True)
  (attention_fc): Linear(in_features=128, out_features=128, bias=True)
  (fc): Linear(in_features=128, out_features=2, bias=True)
  (dropout): Dropout(p=0.0, inplace=False)
)

'train_and_validate' function started. 

Existing folder has been removed : Class_Incremental_CL\Classif_Bi_Dir_GRU_Model\Trained_models\MSE_Distillation_CIL\Period_2\6th_try

y_train:
<class 'torch.Tensor'>
torch.int64
torch.Size([3634, 1000])
X_train:
<class 'torch.Tensor'>
torch.float32
torch.Size([3634, 1000, 7])

y_val:
<class 'torch.Tensor'>
torch.int64
torch.Size([454, 1000])
X_val:
<class 'torch.Tensor'>
torch.float32
torch.Size([454, 1000, 7])

Dataset Lengths:
Train Dataset Length: 3634
Validation Dataset Length: 454

DataLoader Batch Sizes:
Number of Batches in Tr

In [17]:
# Rebuild student model (same as above)
student_model_eval = BiGRUWithAttention(
    input_size=Number_features,
    hidden_size=64,
    output_size=num_classes,
    num_layers=4,
    dropout=0.0
)

# Prepare validation data
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
X_val_tensor = torch.tensor(X_val, dtype=torch.float32).to(device)
y_val_tensor = torch.tensor(y_val, dtype=torch.long).to(device)
criterion = nn.CrossEntropyLoss()

# Your model folder
model_folder = 'Class_Incremental_CL/Classif_Bi_Dir_GRU_Model/Trained_models/MSE_Distillation_CIL/Period_2/6th_try'
ckpt_paths = sorted(glob.glob(os.path.join(model_folder, "*.pth")))

# Loop through and update each model
for file_path in ckpt_paths:
    model_copy = copy.deepcopy(student_model_eval).to(device)
    checkpoint = torch.load(file_path, map_location=device)
    model_copy.load_state_dict(checkpoint['model_state_dict'])

    add_val_class_acc(
        model_copy=model_copy,
        output_size=num_classes,
        criterion=criterion,
        X_val=X_val_tensor,
        y_val=y_val_tensor,
        file_path=file_path,
        model_checkpoint=checkpoint,
        batch_size=64
    )

# 顯示 summary
show_val_accuracy_from_checkpoints(model_folder)


C:\Users\james\AppData\Local\Temp\ipykernel_81344\3542141606.py:23: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(file_path, map_location=device)
C:\

✅ Updated: BiGRUWithAttention_epoch_1007.pth | Acc: 0.9801 | Saved to: Class_Incremental_CL/Classif_Bi_Dir_GRU_Model/Trained_models/MSE_Distillation_CIL/Period_2/6th_try\BiGRUWithAttention_epoch_1007.pth
✅ Updated: BiGRUWithAttention_epoch_1030.pth | Acc: 0.9804 | Saved to: Class_Incremental_CL/Classif_Bi_Dir_GRU_Model/Trained_models/MSE_Distillation_CIL/Period_2/6th_try\BiGRUWithAttention_epoch_1030.pth
✅ Updated: BiGRUWithAttention_epoch_938.pth | Acc: 0.9801 | Saved to: Class_Incremental_CL/Classif_Bi_Dir_GRU_Model/Trained_models/MSE_Distillation_CIL/Period_2/6th_try\BiGRUWithAttention_epoch_938.pth
✅ Updated: BiGRUWithAttention_epoch_963.pth | Acc: 0.9803 | Saved to: Class_Incremental_CL/Classif_Bi_Dir_GRU_Model/Trained_models/MSE_Distillation_CIL/Period_2/6th_try\BiGRUWithAttention_epoch_963.pth
✅ Updated: BiGRUWithAttention_epoch_971.pth | Acc: 0.9801 | Saved to: Class_Incremental_CL/Classif_Bi_Dir_GRU_Model/Trained_models/MSE_Distillation_CIL/Period_2/6th_try\BiGRUWithAttention_

C:\Users\james\AppData\Local\Temp\ipykernel_81344\1933887850.py:66: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(path, map_location="cpu")


### Period 3 --> Training and saving in __*'1st_try'*__ (BiGRUWithAttention, num_layers = 4, alpha = 0.5) ---> Val acc = 97.49 %
### __Val-Class-Acc = {0: '89.69%', 1: '97.89%', 2: '93.78%', 3: '98.94%'} %__

In [19]:
"""
- 'trend': Categorized trend values based on the detected phases:
    - 0: No trend
    - 1: Moderate negative trend
    - 2: Very strong negative trend
    - 3: Moderate positive trend
    - 4: Very strong positive trend
"""
with contextlib.redirect_stdout(open(os.devnull, 'w')):
    X_train, y_train, X_val, y_val, X_test, y_test, Number_features = process_and_return_splits(
        with_indicators_file_path = list_period_files_full_path[2], # Change 
        downsampled_data_minutes = downsampled_data_minutes,
        exclude_columns = exclude_columns,
        lower_threshold = lower_threshold,
        upper_threshold = upper_threshold,
        reverse_steps = reverse_steps,
        sequence_length = sequence_length,
        sliding_interval = sliding_interval,
        trends_to_keep = {0, 1, 2, 3}  # Default keeps all trends : {0, 1, 2, 3, 4}
        # trends_to_keep = {0, 1, 2, 3, 4}  # Default keeps all trends
    )

print(f"\nNumber_features = {Number_features}")

unique_classes = np.unique(y_val)
num_classes = len(unique_classes)
print(f"unique_classes = {unique_classes}")
print(f"num_classes = {num_classes}")



Number_features = 7
unique_classes = [0 1 2 3]
num_classes = 4


In [20]:
# Model parameters
stable_classes = [1, 2] # <----------------------------<<< From period 2: I exclude class 0 because it will change in period 3
input_size = Number_features  # Number of features
hidden_size = 64  # Number of GRU units
output_size = num_classes # Must be dynamic, up to 5  # Number of trend classes (0, 15, 25, -15, -25)
num_layers = 4  # Number of GRU layers
dropout = 0.0
learning_rate = 0.0001
alpha = 0.5          # Weight for distillation loss
num_epochs= 2000 # Number of epochs/ go through entire data
batch_size= 64 # How many sequences passed at once to the model
model_name = 'BiGRUWithAttention' # Name of the model to use for saving
best_results = [] # Initialize this outside the training function or at the beginning of training
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# Define a global stop signal
stop_signal_file = os.path.normpath(os.path.join('Class_Incremental_CL', 'Classif_Bi_Dir_GRU_Model/stop_training.txt'))  # Create this file to stop training
model_saving_folder = os.path.normpath(os.path.join('Class_Incremental_CL', "Classif_Bi_Dir_GRU_Model/Trained_models/MSE_Distillation_CIL/Period_3/1st_try"))
ensure_folder(model_saving_folder)

# Instantiate the student model
student_model = BiGRUWithAttention(input_size, hidden_size, output_size, num_layers, dropout).to(device)

# Instantiate and load the teacher model
teacher_model = BiGRUWithAttention(input_size, hidden_size, output_size - 1, num_layers, dropout).to(device)
#-------------------------------------------------------------------------
best_overall = max(track_across_runs, key=lambda res: res['val_accuracy'])
best_model_path = best_overall['model_path']
#----------------------------------------------------------------------
# Initialize the list to store results across runs
track_across_runs = []
#----------------------------------------------------------------------
#-------------------------------------------------------------------------
teacher_checkpoint_path = os.path.normpath(best_model_path)
teacher_checkpoint = torch.load(teacher_checkpoint_path, map_location=device, weights_only=True)
# print(f"\n{teacher_checkpoint}\n")
teacher_model.load_state_dict(teacher_checkpoint['model_state_dict'])
del teacher_checkpoint
gc.collect()
print(f"Loaded teacher model from: \n\t{teacher_checkpoint_path}")
print(f"\n{teacher_model}\n")

# Define the loss function, optimizer and scheduler
criterion = nn.CrossEntropyLoss()
# Optimizer: only train parameters of the student model (LoRA parameters and any unfrozen ones).
optimizer = optim.Adam(student_model.parameters(), lr=learning_rate) # lr=0.00005
# optimizer = optim.Adam(class_gru_model.parameters(), lr=0.001, weight_decay=1e-5)
# scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', factor=0.9, patience=10)

train_and_validate_MSE(student_model, teacher_model, stable_classes, output_size, criterion, optimizer, 
                       X_train, y_train, X_val, y_val, scheduler, False, num_epochs, batch_size, 
                       alpha, model_saving_folder, model_name, stop_signal_file)

#----------------------------------------------------------------------
# Append only the best result (already at index 0)
track_across_runs.append(best_results[0])
#----------------------------------------------------------------------

for res in best_results:        
    print(f"Epoch {res['epoch']}/{num_epochs}, "
            f"Train Loss: {res['train_loss']:.4f}, " 
            f"Val Loss: {res['val_loss']:.4f}, "
            f"Val Accuracy: {res['val_accuracy'] * 100:.2f}%, "
            f"Model Path: {res['model_path']}")      
print(f"\nclass_gru_model (student_model): \n{student_model}")

print(f"\nunique_classes = {unique_classes}")
print(f"num_classes = {num_classes}")
# del unique_classes, num_classes
for var in ["X_train", "y_train", "X_val", "y_val", "X_test", "y_test", "Number_features", "unique_classes", "num_classes"]:
    if var in locals():
        del locals()[var]


Loaded teacher model from: 
	Class_Incremental_CL\Classif_Bi_Dir_GRU_Model\Trained_models\MSE_Distillation_CIL\Period_2\1st_try\BiGRUWithAttention_epoch_183.pth

BiGRUWithAttention(
  (gru): GRU(7, 64, num_layers=4, batch_first=True, bidirectional=True)
  (attention_fc): Linear(in_features=128, out_features=128, bias=True)
  (fc): Linear(in_features=128, out_features=3, bias=True)
  (dropout): Dropout(p=0.0, inplace=False)
)

'train_and_validate' function started. 

Existing folder has been removed : Class_Incremental_CL\Classif_Bi_Dir_GRU_Model\Trained_models\MSE_Distillation_CIL\Period_3\1st_try

y_train:
<class 'torch.Tensor'>
torch.int64
torch.Size([3634, 1000])
X_train:
<class 'torch.Tensor'>
torch.float32
torch.Size([3634, 1000, 7])

y_val:
<class 'torch.Tensor'>
torch.int64
torch.Size([454, 1000])
X_val:
<class 'torch.Tensor'>
torch.float32
torch.Size([454, 1000, 7])

Dataset Lengths:
Train Dataset Length: 3634
Validation Dataset Length: 454

DataLoader Batch Sizes:
Number of Ba

### Period 3 --> Training and saving in __*'2nd_try'*__ (BiGRUWithAttention, num_layers = 4, alpha = 0.4) ---> Val acc = 97.53 %97.53
### __Val-Class-Acc = {0: '83.44%', 1: '99.01%', 2: '97.40%', 3: '96.69%'} %__

In [27]:
"""
- 'trend': Categorized trend values based on the detected phases:
    - 0: No trend
    - 1: Moderate negative trend
    - 2: Very strong negative trend
    - 3: Moderate positive trend
    - 4: Very strong positive trend
"""
with contextlib.redirect_stdout(open(os.devnull, 'w')):
    X_train, y_train, X_val, y_val, X_test, y_test, Number_features = process_and_return_splits(
        with_indicators_file_path = list_period_files_full_path[2], # Change 
        downsampled_data_minutes = downsampled_data_minutes,
        exclude_columns = exclude_columns,
        lower_threshold = lower_threshold,
        upper_threshold = upper_threshold,
        reverse_steps = reverse_steps,
        sequence_length = sequence_length,
        sliding_interval = sliding_interval,
        trends_to_keep = {0, 1, 2, 3}  # Default keeps all trends : {0, 1, 2, 3, 4}
        # trends_to_keep = {0, 1, 2, 3, 4}  # Default keeps all trends
    )

print(f"\nNumber_features = {Number_features}")

unique_classes = np.unique(y_val)
num_classes = len(unique_classes)
print(f"unique_classes = {unique_classes}")
print(f"num_classes = {num_classes}")



Number_features = 7
unique_classes = [0 1 2 3]
num_classes = 4


In [28]:
# Model parameters
stable_classes = [1, 2] # <----------------------------<<< From period 2: I exclude class 0 because it will change in period 3
input_size = Number_features  # Number of features
hidden_size = 64  # Number of GRU units
output_size = num_classes # Must be dynamic, up to 5  # Number of trend classes (0, 15, 25, -15, -25)
num_layers = 4  # Number of GRU layers
dropout = 0.0
learning_rate = 0.0001
alpha = 0.4          # Weight for distillation loss
num_epochs= 2000 # Number of epochs/ go through entire data
batch_size= 64 # How many sequences passed at once to the model
model_name = 'BiGRUWithAttention' # Name of the model to use for saving
best_results = [] # Initialize this outside the training function or at the beginning of training
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# Define a global stop signal
stop_signal_file = os.path.normpath(os.path.join('Class_Incremental_CL', 'Classif_Bi_Dir_GRU_Model/stop_training.txt'))  # Create this file to stop training
model_saving_folder = os.path.normpath(os.path.join('Class_Incremental_CL', "Classif_Bi_Dir_GRU_Model/Trained_models/MSE_Distillation_CIL/Period_3/2nd_try"))
ensure_folder(model_saving_folder)

# Instantiate the student model
student_model = BiGRUWithAttention(input_size, hidden_size, output_size, num_layers, dropout).to(device)

# Instantiate and load the teacher model
teacher_model = BiGRUWithAttention(input_size, hidden_size, output_size - 1, num_layers, dropout).to(device)
#-------------------------------------------------------------------------
best_overall = max(track_across_runs, key=lambda res: res['val_accuracy'])
best_model_path = best_overall['model_path']
#----------------------------------------------------------------------
# Initialize the list to store results across runs
track_across_runs = []
#----------------------------------------------------------------------
#-------------------------------------------------------------------------
teacher_checkpoint_path = os.path.normpath(best_model_path)
teacher_checkpoint = torch.load(teacher_checkpoint_path, map_location=device, weights_only=True)
# print(f"\n{teacher_checkpoint}\n")
teacher_model.load_state_dict(teacher_checkpoint['model_state_dict'])
del teacher_checkpoint
gc.collect()
print(f"Loaded teacher model from: \n\t{teacher_checkpoint_path}")
print(f"\n{teacher_model}\n")

# Define the loss function, optimizer and scheduler
criterion = nn.CrossEntropyLoss()
# Optimizer: only train parameters of the student model (LoRA parameters and any unfrozen ones).
optimizer = optim.Adam(student_model.parameters(), lr=learning_rate) # lr=0.00005
# optimizer = optim.Adam(class_gru_model.parameters(), lr=0.001, weight_decay=1e-5)
# scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', factor=0.9, patience=10)

train_and_validate_MSE(student_model, teacher_model, stable_classes, output_size, criterion, optimizer, 
                       X_train, y_train, X_val, y_val, scheduler, False, num_epochs, batch_size, 
                       alpha, model_saving_folder, model_name, stop_signal_file)

#----------------------------------------------------------------------
# Append only the best result (already at index 0)
track_across_runs.append(best_results[0])
#----------------------------------------------------------------------

for res in best_results:        
    print(f"Epoch {res['epoch']}/{num_epochs}, "
            f"Train Loss: {res['train_loss']:.4f}, " 
            f"Val Loss: {res['val_loss']:.4f}, "
            f"Val Accuracy: {res['val_accuracy'] * 100:.2f}%, "
            f"Model Path: {res['model_path']}")      
print(f"\nclass_gru_model (student_model): \n{student_model}")

print(f"\nunique_classes = {unique_classes}")
print(f"num_classes = {num_classes}")
# del unique_classes, num_classes
for var in ["X_train", "y_train", "X_val", "y_val", "X_test", "y_test", "Number_features", "unique_classes", "num_classes"]:
    if var in locals():
        del locals()[var]


Loaded teacher model from: 
	Class_Incremental_CL\Classif_Bi_Dir_GRU_Model\Trained_models\MSE_Distillation_CIL\Period_2\2nd_try\BiGRUWithAttention_epoch_1938.pth

BiGRUWithAttention(
  (gru): GRU(7, 64, num_layers=4, batch_first=True, bidirectional=True)
  (attention_fc): Linear(in_features=128, out_features=128, bias=True)
  (fc): Linear(in_features=128, out_features=3, bias=True)
  (dropout): Dropout(p=0.0, inplace=False)
)

'train_and_validate' function started. 

Existing folder has been removed : Class_Incremental_CL\Classif_Bi_Dir_GRU_Model\Trained_models\MSE_Distillation_CIL\Period_3\2nd_try

y_train:
<class 'torch.Tensor'>
torch.int64
torch.Size([3634, 1000])
X_train:
<class 'torch.Tensor'>
torch.float32
torch.Size([3634, 1000, 7])

y_val:
<class 'torch.Tensor'>
torch.int64
torch.Size([454, 1000])
X_val:
<class 'torch.Tensor'>
torch.float32
torch.Size([454, 1000, 7])

Dataset Lengths:
Train Dataset Length: 3634
Validation Dataset Length: 454

DataLoader Batch Sizes:
Number of B

### Period 3 --> Training and saving in __*'3rd_try'*__ (BiGRUWithAttention, num_layers = 4, alpha = 0.3) ---> Val acc = 97.29 %
### __Val-Class-Acc = {0: '91.26%', 1: '97.68%', 2: '96.68%', 3: '97.88%'} %__

In [33]:
"""
- 'trend': Categorized trend values based on the detected phases:
    - 0: No trend
    - 1: Moderate negative trend
    - 2: Very strong negative trend
    - 3: Moderate positive trend
    - 4: Very strong positive trend
"""
with contextlib.redirect_stdout(open(os.devnull, 'w')):
    X_train, y_train, X_val, y_val, X_test, y_test, Number_features = process_and_return_splits(
        with_indicators_file_path = list_period_files_full_path[2], # Change 
        downsampled_data_minutes = downsampled_data_minutes,
        exclude_columns = exclude_columns,
        lower_threshold = lower_threshold,
        upper_threshold = upper_threshold,
        reverse_steps = reverse_steps,
        sequence_length = sequence_length,
        sliding_interval = sliding_interval,
        trends_to_keep = {0, 1, 2, 3}  # Default keeps all trends : {0, 1, 2, 3, 4}
        # trends_to_keep = {0, 1, 2, 3, 4}  # Default keeps all trends
    )

print(f"\nNumber_features = {Number_features}")

unique_classes = np.unique(y_val)
num_classes = len(unique_classes)
print(f"unique_classes = {unique_classes}")
print(f"num_classes = {num_classes}")



Number_features = 7
unique_classes = [0 1 2 3]
num_classes = 4


In [34]:
# Model parameters
stable_classes = [1, 2] # <----------------------------<<< From period 2: I exclude class 0 because it will change in period 3
input_size = Number_features  # Number of features
hidden_size = 64  # Number of GRU units
output_size = num_classes # Must be dynamic, up to 5  # Number of trend classes (0, 15, 25, -15, -25)
num_layers = 4  # Number of GRU layers
dropout = 0.0
learning_rate = 0.0001
alpha = 0.3          # Weight for distillation loss
num_epochs= 2000 # Number of epochs/ go through entire data
batch_size= 64 # How many sequences passed at once to the model
model_name = 'BiGRUWithAttention' # Name of the model to use for saving
best_results = [] # Initialize this outside the training function or at the beginning of training
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# Define a global stop signal
stop_signal_file = os.path.normpath(os.path.join('Class_Incremental_CL', 'Classif_Bi_Dir_GRU_Model/stop_training.txt'))  # Create this file to stop training
model_saving_folder = os.path.normpath(os.path.join('Class_Incremental_CL', "Classif_Bi_Dir_GRU_Model/Trained_models/MSE_Distillation_CIL/Period_3/3rd_try"))
ensure_folder(model_saving_folder)

# Instantiate the student model
student_model = BiGRUWithAttention(input_size, hidden_size, output_size, num_layers, dropout).to(device)

# Instantiate and load the teacher model
teacher_model = BiGRUWithAttention(input_size, hidden_size, output_size - 1, num_layers, dropout).to(device)
#-------------------------------------------------------------------------
best_overall = max(track_across_runs, key=lambda res: res['val_accuracy'])
best_model_path = best_overall['model_path']
#----------------------------------------------------------------------
# Initialize the list to store results across runs
track_across_runs = []
#----------------------------------------------------------------------
#-------------------------------------------------------------------------
teacher_checkpoint_path = os.path.normpath(best_model_path)
teacher_checkpoint = torch.load(teacher_checkpoint_path, map_location=device, weights_only=True)
# print(f"\n{teacher_checkpoint}\n")
teacher_model.load_state_dict(teacher_checkpoint['model_state_dict'])
del teacher_checkpoint
gc.collect()
print(f"Loaded teacher model from: \n\t{teacher_checkpoint_path}")
print(f"\n{teacher_model}\n")

# Define the loss function, optimizer and scheduler
criterion = nn.CrossEntropyLoss()
# Optimizer: only train parameters of the student model (LoRA parameters and any unfrozen ones).
optimizer = optim.Adam(student_model.parameters(), lr=learning_rate) # lr=0.00005
# optimizer = optim.Adam(class_gru_model.parameters(), lr=0.001, weight_decay=1e-5)
# scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', factor=0.9, patience=10)

train_and_validate_MSE(student_model, teacher_model, stable_classes, output_size, criterion, optimizer, 
                       X_train, y_train, X_val, y_val, scheduler, False, num_epochs, batch_size, 
                       alpha, model_saving_folder, model_name, stop_signal_file)

#----------------------------------------------------------------------
# Append only the best result (already at index 0)
track_across_runs.append(best_results[0])
#----------------------------------------------------------------------

for res in best_results:        
    print(f"Epoch {res['epoch']}/{num_epochs}, "
            f"Train Loss: {res['train_loss']:.4f}, " 
            f"Val Loss: {res['val_loss']:.4f}, "
            f"Val Accuracy: {res['val_accuracy'] * 100:.2f}%, "
            f"Model Path: {res['model_path']}")      
print(f"\nclass_gru_model (student_model): \n{student_model}")

print(f"\nunique_classes = {unique_classes}")
print(f"num_classes = {num_classes}")
# del unique_classes, num_classes
for var in ["X_train", "y_train", "X_val", "y_val", "X_test", "y_test", "Number_features", "unique_classes", "num_classes"]:
    if var in locals():
        del locals()[var]


Loaded teacher model from: 
	Class_Incremental_CL\Classif_Bi_Dir_GRU_Model\Trained_models\MSE_Distillation_CIL\Period_2\3rd_try\BiGRUWithAttention_epoch_998.pth

BiGRUWithAttention(
  (gru): GRU(7, 64, num_layers=4, batch_first=True, bidirectional=True)
  (attention_fc): Linear(in_features=128, out_features=128, bias=True)
  (fc): Linear(in_features=128, out_features=3, bias=True)
  (dropout): Dropout(p=0.0, inplace=False)
)

'train_and_validate' function started. 

Existing folder has been removed : Class_Incremental_CL\Classif_Bi_Dir_GRU_Model\Trained_models\MSE_Distillation_CIL\Period_3\3rd_try

y_train:
<class 'torch.Tensor'>
torch.int64
torch.Size([3634, 1000])
X_train:
<class 'torch.Tensor'>
torch.float32
torch.Size([3634, 1000, 7])

y_val:
<class 'torch.Tensor'>
torch.int64
torch.Size([454, 1000])
X_val:
<class 'torch.Tensor'>
torch.float32
torch.Size([454, 1000, 7])

Dataset Lengths:
Train Dataset Length: 3634
Validation Dataset Length: 454

DataLoader Batch Sizes:
Number of Ba

### Period 3 --> Training and saving in __*'4th_try'*__ (BiGRUWithAttention, num_layers = 4, alpha = 0.6) ---> Val acc = 97.01 %
### __Val-Class-Acc = {0: '86.76%', 1: '98.11%', 2: '95.41%', 3: '98.19%'} %__

In [39]:
"""
- 'trend': Categorized trend values based on the detected phases:
    - 0: No trend
    - 1: Moderate negative trend
    - 2: Very strong negative trend
    - 3: Moderate positive trend
    - 4: Very strong positive trend
"""
with contextlib.redirect_stdout(open(os.devnull, 'w')):
    X_train, y_train, X_val, y_val, X_test, y_test, Number_features = process_and_return_splits(
        with_indicators_file_path = list_period_files_full_path[2], # Change 
        downsampled_data_minutes = downsampled_data_minutes,
        exclude_columns = exclude_columns,
        lower_threshold = lower_threshold,
        upper_threshold = upper_threshold,
        reverse_steps = reverse_steps,
        sequence_length = sequence_length,
        sliding_interval = sliding_interval,
        trends_to_keep = {0, 1, 2, 3}  # Default keeps all trends : {0, 1, 2, 3, 4}
        # trends_to_keep = {0, 1, 2, 3, 4}  # Default keeps all trends
    )

print(f"\nNumber_features = {Number_features}")

unique_classes = np.unique(y_val)
num_classes = len(unique_classes)
print(f"unique_classes = {unique_classes}")
print(f"num_classes = {num_classes}")



Number_features = 7
unique_classes = [0 1 2 3]
num_classes = 4


In [40]:
# Model parameters
stable_classes = [1, 2] # <----------------------------<<< From period 2: I exclude class 0 because it will change in period 3
input_size = Number_features  # Number of features
hidden_size = 64  # Number of GRU units
output_size = num_classes # Must be dynamic, up to 5  # Number of trend classes (0, 15, 25, -15, -25)
num_layers = 4  # Number of GRU layers
dropout = 0.0
learning_rate = 0.0001
alpha = 0.6          # Weight for distillation loss
num_epochs= 2000 # Number of epochs/ go through entire data
batch_size= 64 # How many sequences passed at once to the model
model_name = 'BiGRUWithAttention' # Name of the model to use for saving
best_results = [] # Initialize this outside the training function or at the beginning of training
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# Define a global stop signal
stop_signal_file = os.path.normpath(os.path.join('Class_Incremental_CL', 'Classif_Bi_Dir_GRU_Model/stop_training.txt'))  # Create this file to stop training
model_saving_folder = os.path.normpath(os.path.join('Class_Incremental_CL', "Classif_Bi_Dir_GRU_Model/Trained_models/MSE_Distillation_CIL/Period_3/4th_try"))
ensure_folder(model_saving_folder)

# Instantiate the student model
student_model = BiGRUWithAttention(input_size, hidden_size, output_size, num_layers, dropout).to(device)

# Instantiate and load the teacher model
teacher_model = BiGRUWithAttention(input_size, hidden_size, output_size - 1, num_layers, dropout).to(device)
#-------------------------------------------------------------------------
best_overall = max(track_across_runs, key=lambda res: res['val_accuracy'])
best_model_path = best_overall['model_path']
#----------------------------------------------------------------------
# Initialize the list to store results across runs
track_across_runs = []
#----------------------------------------------------------------------
#-------------------------------------------------------------------------
teacher_checkpoint_path = os.path.normpath(best_model_path)
teacher_checkpoint = torch.load(teacher_checkpoint_path, map_location=device, weights_only=True)
# print(f"\n{teacher_checkpoint}\n")
teacher_model.load_state_dict(teacher_checkpoint['model_state_dict'])
del teacher_checkpoint
gc.collect()
print(f"Loaded teacher model from: \n\t{teacher_checkpoint_path}")
print(f"\n{teacher_model}\n")

# Define the loss function, optimizer and scheduler
criterion = nn.CrossEntropyLoss()
# Optimizer: only train parameters of the student model (LoRA parameters and any unfrozen ones).
optimizer = optim.Adam(student_model.parameters(), lr=learning_rate) # lr=0.00005
# optimizer = optim.Adam(class_gru_model.parameters(), lr=0.001, weight_decay=1e-5)
# scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', factor=0.9, patience=10)

train_and_validate_MSE(student_model, teacher_model, stable_classes, output_size, criterion, optimizer, 
                       X_train, y_train, X_val, y_val, scheduler, False, num_epochs, batch_size, 
                       alpha, model_saving_folder, model_name, stop_signal_file)

#----------------------------------------------------------------------
# Append only the best result (already at index 0)
track_across_runs.append(best_results[0])
#----------------------------------------------------------------------

for res in best_results:        
    print(f"Epoch {res['epoch']}/{num_epochs}, "
            f"Train Loss: {res['train_loss']:.4f}, " 
            f"Val Loss: {res['val_loss']:.4f}, "
            f"Val Accuracy: {res['val_accuracy'] * 100:.2f}%, "
            f"Model Path: {res['model_path']}")      
print(f"\nclass_gru_model (student_model): \n{student_model}")

print(f"\nunique_classes = {unique_classes}")
print(f"num_classes = {num_classes}")
# del unique_classes, num_classes
for var in ["X_train", "y_train", "X_val", "y_val", "X_test", "y_test", "Number_features", "unique_classes", "num_classes"]:
    if var in locals():
        del locals()[var]


Loaded teacher model from: 
	Class_Incremental_CL\Classif_Bi_Dir_GRU_Model\Trained_models\MSE_Distillation_CIL\Period_2\4th_try\BiGRUWithAttention_epoch_273.pth

BiGRUWithAttention(
  (gru): GRU(7, 64, num_layers=4, batch_first=True, bidirectional=True)
  (attention_fc): Linear(in_features=128, out_features=128, bias=True)
  (fc): Linear(in_features=128, out_features=3, bias=True)
  (dropout): Dropout(p=0.0, inplace=False)
)

'train_and_validate' function started. 

Existing folder has been removed : Class_Incremental_CL\Classif_Bi_Dir_GRU_Model\Trained_models\MSE_Distillation_CIL\Period_3\4th_try

y_train:
<class 'torch.Tensor'>
torch.int64
torch.Size([3634, 1000])
X_train:
<class 'torch.Tensor'>
torch.float32
torch.Size([3634, 1000, 7])

y_val:
<class 'torch.Tensor'>
torch.int64
torch.Size([454, 1000])
X_val:
<class 'torch.Tensor'>
torch.float32
torch.Size([454, 1000, 7])

Dataset Lengths:
Train Dataset Length: 3634
Validation Dataset Length: 454

DataLoader Batch Sizes:
Number of Ba

### Period 3 --> Training and saving in __*'5th_try'*__ (BiGRUWithAttention, num_layers = 4, alpha = 0.1) ---> Val acc = 98.31 %
### Val-Class-Acc = {0: '95.21%', 1: '98.55%', 2: '97.44%', 3: '98.92%'}

In [18]:
"""
- 'trend': Categorized trend values based on the detected phases:
    - 0: No trend
    - 1: Moderate negative trend
    - 2: Very strong negative trend
    - 3: Moderate positive trend
    - 4: Very strong positive trend
"""
with contextlib.redirect_stdout(open(os.devnull, 'w')):
    X_train, y_train, X_val, y_val, X_test, y_test, Number_features = process_and_return_splits(
        with_indicators_file_path = list_period_files_full_path[2], # Change 
        downsampled_data_minutes = downsampled_data_minutes,
        exclude_columns = exclude_columns,
        lower_threshold = lower_threshold,
        upper_threshold = upper_threshold,
        reverse_steps = reverse_steps,
        sequence_length = sequence_length,
        sliding_interval = sliding_interval,
        trends_to_keep = {0, 1, 2, 3}  # Default keeps all trends : {0, 1, 2, 3, 4}
        # trends_to_keep = {0, 1, 2, 3, 4}  # Default keeps all trends
    )

print(f"\nNumber_features = {Number_features}")

unique_classes = np.unique(y_val)
num_classes = len(unique_classes)
print(f"unique_classes = {unique_classes}")
print(f"num_classes = {num_classes}")



Number_features = 7
unique_classes = [0 1 2 3]
num_classes = 4


In [46]:
# Model parameters
stable_classes = [1, 2] # <----------------------------<<< From period 2: I exclude class 0 because it will change in period 3
input_size = Number_features  # Number of features
hidden_size = 64  # Number of GRU units
output_size = num_classes # Must be dynamic, up to 5  # Number of trend classes (0, 15, 25, -15, -25)
num_layers = 4  # Number of GRU layers
dropout = 0.0
learning_rate = 0.0001
alpha = 0.1          # Weight for distillation loss
num_epochs= 2000 # Number of epochs/ go through entire data
batch_size= 64 # How many sequences passed at once to the model
model_name = 'BiGRUWithAttention' # Name of the model to use for saving
best_results = [] # Initialize this outside the training function or at the beginning of training
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# Define a global stop signal
stop_signal_file = os.path.normpath(os.path.join('Class_Incremental_CL', 'Classif_Bi_Dir_GRU_Model/stop_training.txt'))  # Create this file to stop training
model_saving_folder = os.path.normpath(os.path.join('Class_Incremental_CL', "Classif_Bi_Dir_GRU_Model/Trained_models/MSE_Distillation_CIL/Period_3/5th_try"))
ensure_folder(model_saving_folder)

# Instantiate the student model
student_model = BiGRUWithAttention(input_size, hidden_size, output_size, num_layers, dropout).to(device)

# Instantiate and load the teacher model
teacher_model = BiGRUWithAttention(input_size, hidden_size, output_size - 1, num_layers, dropout).to(device)
#-------------------------------------------------------------------------
best_overall = max(track_across_runs, key=lambda res: res['val_accuracy'])
best_model_path = best_overall['model_path']
#----------------------------------------------------------------------
# Initialize the list to store results across runs
track_across_runs = []
#----------------------------------------------------------------------
#-------------------------------------------------------------------------
teacher_checkpoint_path = os.path.normpath(best_model_path)
teacher_checkpoint = torch.load(teacher_checkpoint_path, map_location=device, weights_only=True)
# print(f"\n{teacher_checkpoint}\n")
teacher_model.load_state_dict(teacher_checkpoint['model_state_dict'])
del teacher_checkpoint
gc.collect()
print(f"Loaded teacher model from: \n\t{teacher_checkpoint_path}")
print(f"\n{teacher_model}\n")

# Define the loss function, optimizer and scheduler
criterion = nn.CrossEntropyLoss()
# Optimizer: only train parameters of the student model (LoRA parameters and any unfrozen ones).
optimizer = optim.Adam(student_model.parameters(), lr=learning_rate) # lr=0.00005
# optimizer = optim.Adam(class_gru_model.parameters(), lr=0.001, weight_decay=1e-5)
# scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', factor=0.9, patience=10)

train_and_validate_MSE(student_model, teacher_model, stable_classes, output_size, criterion, optimizer, 
                       X_train, y_train, X_val, y_val, scheduler, False, num_epochs, batch_size, 
                       alpha, model_saving_folder, model_name, stop_signal_file)

#----------------------------------------------------------------------
# Append only the best result (already at index 0)
track_across_runs.append(best_results[0])
#----------------------------------------------------------------------

for res in best_results:        
    print(f"Epoch {res['epoch']}/{num_epochs}, "
            f"Train Loss: {res['train_loss']:.4f}, " 
            f"Val Loss: {res['val_loss']:.4f}, "
            f"Val Accuracy: {res['val_accuracy'] * 100:.2f}%, "
            f"Model Path: {res['model_path']}")      
print(f"\nclass_gru_model (student_model): \n{student_model}")

print(f"\nunique_classes = {unique_classes}")
print(f"num_classes = {num_classes}")
# del unique_classes, num_classes
for var in ["X_train", "y_train", "X_val", "y_val", "X_test", "y_test", "Number_features", "unique_classes", "num_classes"]:
    if var in locals():
        del locals()[var]


Loaded teacher model from: 
	Class_Incremental_CL\Classif_Bi_Dir_GRU_Model\Trained_models\MSE_Distillation_CIL\Period_2\5th_try\BiGRUWithAttention_epoch_318.pth

BiGRUWithAttention(
  (gru): GRU(7, 64, num_layers=4, batch_first=True, bidirectional=True)
  (attention_fc): Linear(in_features=128, out_features=128, bias=True)
  (fc): Linear(in_features=128, out_features=3, bias=True)
  (dropout): Dropout(p=0.0, inplace=False)
)

'train_and_validate' function started. 

Existing folder has been removed : Class_Incremental_CL\Classif_Bi_Dir_GRU_Model\Trained_models\MSE_Distillation_CIL\Period_3\5th_try

y_train:
<class 'torch.Tensor'>
torch.int64
torch.Size([3634, 1000])
X_train:
<class 'torch.Tensor'>
torch.float32
torch.Size([3634, 1000, 7])

y_val:
<class 'torch.Tensor'>
torch.int64
torch.Size([454, 1000])
X_val:
<class 'torch.Tensor'>
torch.float32
torch.Size([454, 1000, 7])

Dataset Lengths:
Train Dataset Length: 3634
Validation Dataset Length: 454

DataLoader Batch Sizes:
Number of Ba

In [19]:
# === Post-training Evaluation: Update .pth with val accuracy and class-wise accuracy ===

# Rebuild model template for evaluation
model_template = BiGRUWithAttention(
    input_size=Number_features,
    hidden_size=64,
    output_size=num_classes,
    num_layers=4,
    dropout=0.0
)

# Prepare validation data
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
X_val_tensor = torch.tensor(X_val, dtype=torch.float32).to(device)
y_val_tensor = torch.tensor(y_val, dtype=torch.long).to(device)
criterion = nn.CrossEntropyLoss()

# Folder where all checkpoints are saved
model_folder = 'Class_Incremental_CL/Classif_Bi_Dir_GRU_Model/Trained_models/MSE_Distillation_CIL/Period_3/5th_try'
ckpt_paths = sorted(glob.glob(os.path.join(model_folder, "*.pth")))

# Loop through and update each model
for file_path in ckpt_paths:
    model_copy = copy.deepcopy(model_template).to(device)
    checkpoint = torch.load(file_path, map_location=device)
    model_copy.load_state_dict(checkpoint['model_state_dict'])

    add_val_class_acc(
        model_copy=model_copy,
        output_size=num_classes,
        criterion=criterion,
        X_val=X_val_tensor,
        y_val=y_val_tensor,
        file_path=file_path,
        model_checkpoint=checkpoint,
        batch_size=64
    )

# Display summary
show_val_accuracy_from_checkpoints(model_folder)


C:\Users\james\AppData\Local\Temp\ipykernel_81344\3612971756.py:25: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(file_path, map_location=device)
C:\

✅ Updated: BiGRUWithAttention_epoch_1272.pth | Acc: 0.9827 | Saved to: Class_Incremental_CL/Classif_Bi_Dir_GRU_Model/Trained_models/MSE_Distillation_CIL/Period_3/5th_try\BiGRUWithAttention_epoch_1272.pth
✅ Updated: BiGRUWithAttention_epoch_1324.pth | Acc: 0.9824 | Saved to: Class_Incremental_CL/Classif_Bi_Dir_GRU_Model/Trained_models/MSE_Distillation_CIL/Period_3/5th_try\BiGRUWithAttention_epoch_1324.pth
✅ Updated: BiGRUWithAttention_epoch_1381.pth | Acc: 0.9828 | Saved to: Class_Incremental_CL/Classif_Bi_Dir_GRU_Model/Trained_models/MSE_Distillation_CIL/Period_3/5th_try\BiGRUWithAttention_epoch_1381.pth
✅ Updated: BiGRUWithAttention_epoch_1413.pth | Acc: 0.9831 | Saved to: Class_Incremental_CL/Classif_Bi_Dir_GRU_Model/Trained_models/MSE_Distillation_CIL/Period_3/5th_try\BiGRUWithAttention_epoch_1413.pth
✅ Updated: BiGRUWithAttention_epoch_599.pth | Acc: 0.9825 | Saved to: Class_Incremental_CL/Classif_Bi_Dir_GRU_Model/Trained_models/MSE_Distillation_CIL/Period_3/5th_try\BiGRUWithAttent

C:\Users\james\AppData\Local\Temp\ipykernel_81344\1933887850.py:66: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(path, map_location="cpu")


### Period 3 --> Training and saving in __*'6th_try'*__ (BiGRUWithAttention, num_layers = 4, alpha = 0.2) ---> Val acc = 97.81 %
### __Val-Class-Acc = {0: '87.98%', 1: '98.63%', 2: '98.14%', 3: '98.90%'} %__

In [18]:
"""
- 'trend': Categorized trend values based on the detected phases:
    - 0: No trend
    - 1: Moderate negative trend
    - 2: Very strong negative trend
    - 3: Moderate positive trend
    - 4: Very strong positive trend
"""
with contextlib.redirect_stdout(open(os.devnull, 'w')):
    X_train, y_train, X_val, y_val, X_test, y_test, Number_features = process_and_return_splits(
        with_indicators_file_path = list_period_files_full_path[2], # Change 
        downsampled_data_minutes = downsampled_data_minutes,
        exclude_columns = exclude_columns,
        lower_threshold = lower_threshold,
        upper_threshold = upper_threshold,
        reverse_steps = reverse_steps,
        sequence_length = sequence_length,
        sliding_interval = sliding_interval,
        trends_to_keep = {0, 1, 2, 3}  # Default keeps all trends : {0, 1, 2, 3, 4}
        # trends_to_keep = {0, 1, 2, 3, 4}  # Default keeps all trends
    )

print(f"\nNumber_features = {Number_features}")

unique_classes = np.unique(y_val)
num_classes = len(unique_classes)
print(f"unique_classes = {unique_classes}")
print(f"num_classes = {num_classes}")



Number_features = 7
unique_classes = [0 1 2 3]
num_classes = 4


In [19]:
# Model parameters
stable_classes = [1, 2] # <----------------------------<<< From period 2: I exclude class 0 because it will change in period 3
input_size = Number_features  # Number of features
hidden_size = 64  # Number of GRU units
output_size = num_classes # Must be dynamic, up to 5  # Number of trend classes (0, 15, 25, -15, -25)
num_layers = 4  # Number of GRU layers
dropout = 0.0
learning_rate = 0.0001
alpha = 0.2          # Weight for distillation loss
num_epochs= 2000 # Number of epochs/ go through entire data
batch_size= 64 # How many sequences passed at once to the model
model_name = 'BiGRUWithAttention' # Name of the model to use for saving
best_results = [] # Initialize this outside the training function or at the beginning of training
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# Define a global stop signal
stop_signal_file = os.path.normpath(os.path.join('Class_Incremental_CL', 'Classif_Bi_Dir_GRU_Model/stop_training.txt'))  # Create this file to stop training
model_saving_folder = os.path.normpath(os.path.join('Class_Incremental_CL', "Classif_Bi_Dir_GRU_Model/Trained_models/MSE_Distillation_CIL/Period_3/6th_try"))
ensure_folder(model_saving_folder)

# Instantiate the student model
student_model = BiGRUWithAttention(input_size, hidden_size, output_size, num_layers, dropout).to(device)

# Instantiate and load the teacher model
teacher_model = BiGRUWithAttention(input_size, hidden_size, output_size - 1, num_layers, dropout).to(device)
#-------------------------------------------------------------------------
best_overall = max(track_across_runs, key=lambda res: res['val_accuracy'])
best_model_path = best_overall['model_path']
#----------------------------------------------------------------------
# Initialize the list to store results across runs
track_across_runs = []
#----------------------------------------------------------------------
#-------------------------------------------------------------------------
teacher_checkpoint_path = os.path.normpath(best_model_path)
teacher_checkpoint = torch.load(teacher_checkpoint_path, map_location=device, weights_only=True)
# print(f"\n{teacher_checkpoint}\n")
teacher_model.load_state_dict(teacher_checkpoint['model_state_dict'])
del teacher_checkpoint
gc.collect()
print(f"Loaded teacher model from: \n\t{teacher_checkpoint_path}")
print(f"\n{teacher_model}\n")

# Define the loss function, optimizer and scheduler
criterion = nn.CrossEntropyLoss()
# Optimizer: only train parameters of the student model (LoRA parameters and any unfrozen ones).
optimizer = optim.Adam(student_model.parameters(), lr=learning_rate) # lr=0.00005
# optimizer = optim.Adam(class_gru_model.parameters(), lr=0.001, weight_decay=1e-5)
# scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', factor=0.9, patience=10)

train_and_validate_MSE(student_model, teacher_model, stable_classes, output_size, criterion, optimizer, 
                       X_train, y_train, X_val, y_val, scheduler, False, num_epochs, batch_size, 
                       alpha, model_saving_folder, model_name, stop_signal_file)

#----------------------------------------------------------------------
# Append only the best result (already at index 0)
track_across_runs.append(best_results[0])
#----------------------------------------------------------------------

for res in best_results:        
    print(f"Epoch {res['epoch']}/{num_epochs}, "
            f"Train Loss: {res['train_loss']:.4f}, " 
            f"Val Loss: {res['val_loss']:.4f}, "
            f"Val Accuracy: {res['val_accuracy'] * 100:.2f}%, "
            f"Model Path: {res['model_path']}")      
print(f"\nclass_gru_model (student_model): \n{student_model}")

print(f"\nunique_classes = {unique_classes}")
print(f"num_classes = {num_classes}")
# del unique_classes, num_classes
for var in ["X_train", "y_train", "X_val", "y_val", "X_test", "y_test", "Number_features", "unique_classes", "num_classes"]:
    if var in locals():
        del locals()[var]


Loaded teacher model from: 
	Class_Incremental_CL\Classif_Bi_Dir_GRU_Model\Trained_models\MSE_Distillation_CIL\Period_2\6th_try\BiGRUWithAttention_epoch_1030.pth

BiGRUWithAttention(
  (gru): GRU(7, 64, num_layers=4, batch_first=True, bidirectional=True)
  (attention_fc): Linear(in_features=128, out_features=128, bias=True)
  (fc): Linear(in_features=128, out_features=3, bias=True)
  (dropout): Dropout(p=0.0, inplace=False)
)

'train_and_validate' function started. 

Existing folder has been removed : Class_Incremental_CL\Classif_Bi_Dir_GRU_Model\Trained_models\MSE_Distillation_CIL\Period_3\6th_try

y_train:
<class 'torch.Tensor'>
torch.int64
torch.Size([3634, 1000])
X_train:
<class 'torch.Tensor'>
torch.float32
torch.Size([3634, 1000, 7])

y_val:
<class 'torch.Tensor'>
torch.int64
torch.Size([454, 1000])
X_val:
<class 'torch.Tensor'>
torch.float32
torch.Size([454, 1000, 7])

Dataset Lengths:
Train Dataset Length: 3634
Validation Dataset Length: 454

DataLoader Batch Sizes:
Number of B

### Period 4 --> Training and saving in __*'1st_try'*__ (BiGRUWithAttention, num_layers = 4, alpha = 0.5) ---> Val acc = 96.05 %
### __Val-Class-Acc = {0: '90.75%', 1: '96.18%', 2: '91.18%', 3: '97.15%', 4: '99.33%'} %__

In [21]:
"""
- 'trend': Categorized trend values based on the detected phases:
    - 0: No trend
    - 1: Moderate negative trend
    - 2: Very strong negative trend
    - 3: Moderate positive trend
    - 4: Very strong positive trend
"""
with contextlib.redirect_stdout(open(os.devnull, 'w')):
    X_train, y_train, X_val, y_val, X_test, y_test, Number_features = process_and_return_splits(
        with_indicators_file_path = list_period_files_full_path[3], # Change 
        downsampled_data_minutes = downsampled_data_minutes,
        exclude_columns = exclude_columns,
        lower_threshold = lower_threshold,
        upper_threshold = upper_threshold,
        reverse_steps = reverse_steps,
        sequence_length = sequence_length,
        sliding_interval = sliding_interval,
        trends_to_keep = {0, 1, 2, 3, 4}  # Default keeps all trends : {0, 1, 2, 3, 4}
        # trends_to_keep = {0, 1, 2, 3, 4}  # Default keeps all trends
    )

print(f"\nNumber_features = {Number_features}")

unique_classes = np.unique(y_val)
num_classes = len(unique_classes)
print(f"unique_classes = {unique_classes}")
print(f"num_classes = {num_classes}")



Number_features = 7
unique_classes = [0 1 2 3 4]
num_classes = 5


In [22]:
# Model parameters
stable_classes = [1, 2, 3] # <----------------------------<<< From period 2: I exclude class 0 because it will change in period 4
input_size = Number_features  # Number of features
hidden_size = 64  # Number of GRU units
output_size = num_classes # Must be dynamic, up to 5  # Number of trend classes (0, 15, 25, -15, -25)
num_layers = 4  # Number of GRU layers
dropout = 0.0
learning_rate = 0.0001
alpha = 0.5          # Weight for distillation loss
num_epochs= 2000 # Number of epochs/ go through entire data
batch_size= 64 # How many sequences passed at once to the model
model_name = 'BiGRUWithAttention' # Name of the model to use for saving
best_results = [] # Initialize this outside the training function or at the beginning of training
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# Define a global stop signal
stop_signal_file = os.path.normpath(os.path.join('Class_Incremental_CL', 'Classif_Bi_Dir_GRU_Model/stop_training.txt'))  # Create this file to stop training
model_saving_folder = os.path.normpath(os.path.join('Class_Incremental_CL', "Classif_Bi_Dir_GRU_Model/Trained_models/MSE_Distillation_CIL/Period_4/1st_try"))
ensure_folder(model_saving_folder)

# Instantiate the student model
student_model = BiGRUWithAttention(input_size, hidden_size, output_size, num_layers, dropout).to(device)

# Instantiate and load the teacher model
teacher_model = BiGRUWithAttention(input_size, hidden_size, output_size - 1, num_layers, dropout).to(device)
#-------------------------------------------------------------------------
best_overall = max(track_across_runs, key=lambda res: res['val_accuracy'])
best_model_path = best_overall['model_path']
#----------------------------------------------------------------------
# Initialize the list to store results across runs
track_across_runs = []
#----------------------------------------------------------------------
#-------------------------------------------------------------------------
teacher_checkpoint_path = os.path.normpath(best_model_path)
teacher_checkpoint = torch.load(teacher_checkpoint_path, map_location=device, weights_only=True)
# print(f"\n{teacher_checkpoint}\n")
teacher_model.load_state_dict(teacher_checkpoint['model_state_dict'])
del teacher_checkpoint
gc.collect()
print(f"Loaded teacher model from: \n\t{teacher_checkpoint_path}")
print(f"\n{teacher_model}\n")

# Define the loss function, optimizer and scheduler
criterion = nn.CrossEntropyLoss()
# Optimizer: only train parameters of the student model (LoRA parameters and any unfrozen ones).
optimizer = optim.Adam(student_model.parameters(), lr=learning_rate) # lr=0.00005
# optimizer = optim.Adam(class_gru_model.parameters(), lr=0.001, weight_decay=1e-5)
# scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', factor=0.9, patience=10)

train_and_validate_MSE(student_model, teacher_model, stable_classes, output_size, criterion, optimizer, 
                       X_train, y_train, X_val, y_val, scheduler, False, num_epochs, batch_size, 
                       alpha, model_saving_folder, model_name, stop_signal_file)

#----------------------------------------------------------------------
# Append only the best result (already at index 0)
track_across_runs.append(best_results[0])
#----------------------------------------------------------------------

for res in best_results:        
    print(f"Epoch {res['epoch']}/{num_epochs}, "
            f"Train Loss: {res['train_loss']:.4f}, " 
            f"Val Loss: {res['val_loss']:.4f}, "
            f"Val Accuracy: {res['val_accuracy'] * 100:.2f}%, "
            f"Model Path: {res['model_path']}")      
print(f"\nclass_gru_model (student_model): \n{student_model}")

print(f"\nunique_classes = {unique_classes}")
print(f"num_classes = {num_classes}")
# del unique_classes, num_classes
for var in ["X_train", "y_train", "X_val", "y_val", "X_test", "y_test", "Number_features", "unique_classes", "num_classes"]:
    if var in locals():
        del locals()[var]


Loaded teacher model from: 
	Class_Incremental_CL\Classif_Bi_Dir_GRU_Model\Trained_models\MSE_Distillation_CIL\Period_3\1st_try\BiGRUWithAttention_epoch_1626.pth

BiGRUWithAttention(
  (gru): GRU(7, 64, num_layers=4, batch_first=True, bidirectional=True)
  (attention_fc): Linear(in_features=128, out_features=128, bias=True)
  (fc): Linear(in_features=128, out_features=4, bias=True)
  (dropout): Dropout(p=0.0, inplace=False)
)

'train_and_validate' function started. 

Existing folder has been removed : Class_Incremental_CL\Classif_Bi_Dir_GRU_Model\Trained_models\MSE_Distillation_CIL\Period_4\1st_try

y_train:
<class 'torch.Tensor'>
torch.int64
torch.Size([3634, 1000])
X_train:
<class 'torch.Tensor'>
torch.float32
torch.Size([3634, 1000, 7])

y_val:
<class 'torch.Tensor'>
torch.int64
torch.Size([454, 1000])
X_val:
<class 'torch.Tensor'>
torch.float32
torch.Size([454, 1000, 7])

Dataset Lengths:
Train Dataset Length: 3634
Validation Dataset Length: 454

DataLoader Batch Sizes:
Number of B

### Period 4 --> Training and saving in __*'2nd_try'*__ (BiGRUWithAttention, num_layers = 4, alpha = 0.4) ---> Val acc = 95.33 %
### __Val-Class-Acc = {0: '88.03%', 1: '94.74%', 2: '84.69%', 3: '97.90%', 4: '98.72%'} %__

In [29]:
"""
- 'trend': Categorized trend values based on the detected phases:
    - 0: No trend
    - 1: Moderate negative trend
    - 2: Very strong negative trend
    - 3: Moderate positive trend
    - 4: Very strong positive trend
"""
with contextlib.redirect_stdout(open(os.devnull, 'w')):
    X_train, y_train, X_val, y_val, X_test, y_test, Number_features = process_and_return_splits(
        with_indicators_file_path = list_period_files_full_path[3], # Change 
        downsampled_data_minutes = downsampled_data_minutes,
        exclude_columns = exclude_columns,
        lower_threshold = lower_threshold,
        upper_threshold = upper_threshold,
        reverse_steps = reverse_steps,
        sequence_length = sequence_length,
        sliding_interval = sliding_interval,
        trends_to_keep = {0, 1, 2, 3, 4}  # Default keeps all trends : {0, 1, 2, 3, 4}
        # trends_to_keep = {0, 1, 2, 3, 4}  # Default keeps all trends
    )

print(f"\nNumber_features = {Number_features}")

unique_classes = np.unique(y_val)
num_classes = len(unique_classes)
print(f"unique_classes = {unique_classes}")
print(f"num_classes = {num_classes}")



Number_features = 7
unique_classes = [0 1 2 3 4]
num_classes = 5


In [30]:
# Model parameters
stable_classes = [1, 2, 3] # <----------------------------<<< From period 2: I exclude class 0 because it will change in period 4
input_size = Number_features  # Number of features
hidden_size = 64  # Number of GRU units
output_size = num_classes # Must be dynamic, up to 5  # Number of trend classes (0, 15, 25, -15, -25)
num_layers = 4  # Number of GRU layers
dropout = 0.0
learning_rate = 0.0001
alpha = 0.4          # Weight for distillation loss
num_epochs= 2000 # Number of epochs/ go through entire data
batch_size= 64 # How many sequences passed at once to the model
model_name = 'BiGRUWithAttention' # Name of the model to use for saving
best_results = [] # Initialize this outside the training function or at the beginning of training
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# Define a global stop signal
stop_signal_file = os.path.normpath(os.path.join('Class_Incremental_CL', 'Classif_Bi_Dir_GRU_Model/stop_training.txt'))  # Create this file to stop training
model_saving_folder = os.path.normpath(os.path.join('Class_Incremental_CL', "Classif_Bi_Dir_GRU_Model/Trained_models/MSE_Distillation_CIL/Period_4/2nd_try"))
ensure_folder(model_saving_folder)

# Instantiate the student model
student_model = BiGRUWithAttention(input_size, hidden_size, output_size, num_layers, dropout).to(device)

# Instantiate and load the teacher model
teacher_model = BiGRUWithAttention(input_size, hidden_size, output_size - 1, num_layers, dropout).to(device)
#-------------------------------------------------------------------------
best_overall = max(track_across_runs, key=lambda res: res['val_accuracy'])
best_model_path = best_overall['model_path']
#----------------------------------------------------------------------
# Initialize the list to store results across runs
track_across_runs = []
#----------------------------------------------------------------------
#-------------------------------------------------------------------------
teacher_checkpoint_path = os.path.normpath(best_model_path)
teacher_checkpoint = torch.load(teacher_checkpoint_path, map_location=device, weights_only=True)
# print(f"\n{teacher_checkpoint}\n")
teacher_model.load_state_dict(teacher_checkpoint['model_state_dict'])
del teacher_checkpoint
gc.collect()
print(f"Loaded teacher model from: \n\t{teacher_checkpoint_path}")
print(f"\n{teacher_model}\n")

# Define the loss function, optimizer and scheduler
criterion = nn.CrossEntropyLoss()
# Optimizer: only train parameters of the student model (LoRA parameters and any unfrozen ones).
optimizer = optim.Adam(student_model.parameters(), lr=learning_rate) # lr=0.00005
# optimizer = optim.Adam(class_gru_model.parameters(), lr=0.001, weight_decay=1e-5)
# scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', factor=0.9, patience=10)

train_and_validate_MSE(student_model, teacher_model, stable_classes, output_size, criterion, optimizer, 
                       X_train, y_train, X_val, y_val, scheduler, False, num_epochs, batch_size, 
                       alpha, model_saving_folder, model_name, stop_signal_file)

#----------------------------------------------------------------------
# Append only the best result (already at index 0)
track_across_runs.append(best_results[0])
#----------------------------------------------------------------------

for res in best_results:        
    print(f"Epoch {res['epoch']}/{num_epochs}, "
            f"Train Loss: {res['train_loss']:.4f}, " 
            f"Val Loss: {res['val_loss']:.4f}, "
            f"Val Accuracy: {res['val_accuracy'] * 100:.2f}%, "
            f"Model Path: {res['model_path']}")      
print(f"\nclass_gru_model (student_model): \n{student_model}")

print(f"\nunique_classes = {unique_classes}")
print(f"num_classes = {num_classes}")
# del unique_classes, num_classes
for var in ["X_train", "y_train", "X_val", "y_val", "X_test", "y_test", "Number_features", "unique_classes", "num_classes"]:
    if var in locals():
        del locals()[var]


Loaded teacher model from: 
	Class_Incremental_CL\Classif_Bi_Dir_GRU_Model\Trained_models\MSE_Distillation_CIL\Period_3\2nd_try\BiGRUWithAttention_epoch_1955.pth

BiGRUWithAttention(
  (gru): GRU(7, 64, num_layers=4, batch_first=True, bidirectional=True)
  (attention_fc): Linear(in_features=128, out_features=128, bias=True)
  (fc): Linear(in_features=128, out_features=4, bias=True)
  (dropout): Dropout(p=0.0, inplace=False)
)

'train_and_validate' function started. 

Existing folder has been removed : Class_Incremental_CL\Classif_Bi_Dir_GRU_Model\Trained_models\MSE_Distillation_CIL\Period_4\2nd_try

y_train:
<class 'torch.Tensor'>
torch.int64
torch.Size([3634, 1000])
X_train:
<class 'torch.Tensor'>
torch.float32
torch.Size([3634, 1000, 7])

y_val:
<class 'torch.Tensor'>
torch.int64
torch.Size([454, 1000])
X_val:
<class 'torch.Tensor'>
torch.float32
torch.Size([454, 1000, 7])

Dataset Lengths:
Train Dataset Length: 3634
Validation Dataset Length: 454

DataLoader Batch Sizes:
Number of B

### Period 4 --> Training and saving in __*'3rd_try'*__ (BiGRUWithAttention, num_layers = 4, alpha = 0.3) ---> Val acc = 96.68 %
### Val-Class-Acc = {0: '94.53%', 1: '96.85%', 2: '92.63%', 3: '98.14%', 4: '98.28%'}

In [20]:
"""
- 'trend': Categorized trend values based on the detected phases:
    - 0: No trend
    - 1: Moderate negative trend
    - 2: Very strong negative trend
    - 3: Moderate positive trend
    - 4: Very strong positive trend
"""
with contextlib.redirect_stdout(open(os.devnull, 'w')):
    X_train, y_train, X_val, y_val, X_test, y_test, Number_features = process_and_return_splits(
        with_indicators_file_path = list_period_files_full_path[3], # Change 
        downsampled_data_minutes = downsampled_data_minutes,
        exclude_columns = exclude_columns,
        lower_threshold = lower_threshold,
        upper_threshold = upper_threshold,
        reverse_steps = reverse_steps,
        sequence_length = sequence_length,
        sliding_interval = sliding_interval,
        trends_to_keep = {0, 1, 2, 3, 4}  # Default keeps all trends : {0, 1, 2, 3, 4}
        # trends_to_keep = {0, 1, 2, 3, 4}  # Default keeps all trends
    )

print(f"\nNumber_features = {Number_features}")

unique_classes = np.unique(y_val)
num_classes = len(unique_classes)
print(f"unique_classes = {unique_classes}")
print(f"num_classes = {num_classes}")



Number_features = 7
unique_classes = [0 1 2 3 4]
num_classes = 5


In [36]:
# Model parameters
stable_classes = [1, 2, 3] # <----------------------------<<< From period 2: I exclude class 0 because it will change in period 4
input_size = Number_features  # Number of features
hidden_size = 64  # Number of GRU units
output_size = num_classes # Must be dynamic, up to 5  # Number of trend classes (0, 15, 25, -15, -25)
num_layers = 4  # Number of GRU layers
dropout = 0.0
learning_rate = 0.0001
alpha = 0.3          # Weight for distillation loss
num_epochs= 2000 # Number of epochs/ go through entire data
batch_size= 64 # How many sequences passed at once to the model
model_name = 'BiGRUWithAttention' # Name of the model to use for saving
best_results = [] # Initialize this outside the training function or at the beginning of training
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# Define a global stop signal
stop_signal_file = os.path.normpath(os.path.join('Class_Incremental_CL', 'Classif_Bi_Dir_GRU_Model/stop_training.txt'))  # Create this file to stop training
model_saving_folder = os.path.normpath(os.path.join('Class_Incremental_CL', "Classif_Bi_Dir_GRU_Model/Trained_models/MSE_Distillation_CIL/Period_4/3rd_try"))
ensure_folder(model_saving_folder)

# Instantiate the student model
student_model = BiGRUWithAttention(input_size, hidden_size, output_size, num_layers, dropout).to(device)

# Instantiate and load the teacher model
teacher_model = BiGRUWithAttention(input_size, hidden_size, output_size - 1, num_layers, dropout).to(device)
#-------------------------------------------------------------------------
best_overall = max(track_across_runs, key=lambda res: res['val_accuracy'])
best_model_path = best_overall['model_path']
#----------------------------------------------------------------------
# Initialize the list to store results across runs
track_across_runs = []
#----------------------------------------------------------------------
#-------------------------------------------------------------------------
teacher_checkpoint_path = os.path.normpath(best_model_path)
teacher_checkpoint = torch.load(teacher_checkpoint_path, map_location=device, weights_only=True)
# print(f"\n{teacher_checkpoint}\n")
teacher_model.load_state_dict(teacher_checkpoint['model_state_dict'])
del teacher_checkpoint
gc.collect()
print(f"Loaded teacher model from: \n\t{teacher_checkpoint_path}")
print(f"\n{teacher_model}\n")

# Define the loss function, optimizer and scheduler
criterion = nn.CrossEntropyLoss()
# Optimizer: only train parameters of the student model (LoRA parameters and any unfrozen ones).
optimizer = optim.Adam(student_model.parameters(), lr=learning_rate) # lr=0.00005
# optimizer = optim.Adam(class_gru_model.parameters(), lr=0.001, weight_decay=1e-5)
# scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', factor=0.9, patience=10)

train_and_validate_MSE(student_model, teacher_model, stable_classes, output_size, criterion, optimizer, 
                       X_train, y_train, X_val, y_val, scheduler, False, num_epochs, batch_size, 
                       alpha, model_saving_folder, model_name, stop_signal_file)

#----------------------------------------------------------------------
# Append only the best result (already at index 0)
track_across_runs.append(best_results[0])
#----------------------------------------------------------------------

for res in best_results:        
    print(f"Epoch {res['epoch']}/{num_epochs}, "
            f"Train Loss: {res['train_loss']:.4f}, " 
            f"Val Loss: {res['val_loss']:.4f}, "
            f"Val Accuracy: {res['val_accuracy'] * 100:.2f}%, "
            f"Model Path: {res['model_path']}")      
print(f"\nclass_gru_model (student_model): \n{student_model}")

print(f"\nunique_classes = {unique_classes}")
print(f"num_classes = {num_classes}")
# del unique_classes, num_classes
for var in ["X_train", "y_train", "X_val", "y_val", "X_test", "y_test", "Number_features", "unique_classes", "num_classes"]:
    if var in locals():
        del locals()[var]


Loaded teacher model from: 
	Class_Incremental_CL\Classif_Bi_Dir_GRU_Model\Trained_models\MSE_Distillation_CIL\Period_3\3rd_try\BiGRUWithAttention_epoch_1902.pth

BiGRUWithAttention(
  (gru): GRU(7, 64, num_layers=4, batch_first=True, bidirectional=True)
  (attention_fc): Linear(in_features=128, out_features=128, bias=True)
  (fc): Linear(in_features=128, out_features=4, bias=True)
  (dropout): Dropout(p=0.0, inplace=False)
)

'train_and_validate' function started. 

Existing folder has been removed : Class_Incremental_CL\Classif_Bi_Dir_GRU_Model\Trained_models\MSE_Distillation_CIL\Period_4\3rd_try

y_train:
<class 'torch.Tensor'>
torch.int64
torch.Size([3634, 1000])
X_train:
<class 'torch.Tensor'>
torch.float32
torch.Size([3634, 1000, 7])

y_val:
<class 'torch.Tensor'>
torch.int64
torch.Size([454, 1000])
X_val:
<class 'torch.Tensor'>
torch.float32
torch.Size([454, 1000, 7])

Dataset Lengths:
Train Dataset Length: 3634
Validation Dataset Length: 454

DataLoader Batch Sizes:
Number of B

In [21]:
# === Post-training Evaluation: Update .pth with val accuracy and class-wise accuracy ===

# Rebuild model template for evaluation
model_template = BiGRUWithAttention(
    input_size=Number_features,
    hidden_size=64,
    output_size=num_classes,
    num_layers=4,
    dropout=0.0
)

# Prepare validation data
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
X_val_tensor = torch.tensor(X_val, dtype=torch.float32).to(device)
y_val_tensor = torch.tensor(y_val, dtype=torch.long).to(device)
criterion = nn.CrossEntropyLoss()

# Folder where all checkpoints are saved
model_folder = 'Class_Incremental_CL/Classif_Bi_Dir_GRU_Model/Trained_models/MSE_Distillation_CIL/Period_4/3rd_try'
ckpt_paths = sorted(glob.glob(os.path.join(model_folder, "*.pth")))

# Loop through and update each model
for file_path in ckpt_paths:
    model_copy = copy.deepcopy(model_template).to(device)
    checkpoint = torch.load(file_path, map_location=device)
    model_copy.load_state_dict(checkpoint['model_state_dict'])

    add_val_class_acc(
        model_copy=model_copy,
        output_size=num_classes,
        criterion=criterion,
        X_val=X_val_tensor,
        y_val=y_val_tensor,
        file_path=file_path,
        model_checkpoint=checkpoint,
        batch_size=64
    )

# Display summary
show_val_accuracy_from_checkpoints(model_folder)


C:\Users\james\AppData\Local\Temp\ipykernel_81344\2987948974.py:25: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(file_path, map_location=device)
C:\

✅ Updated: BiGRUWithAttention_epoch_720.pth | Acc: 0.9665 | Saved to: Class_Incremental_CL/Classif_Bi_Dir_GRU_Model/Trained_models/MSE_Distillation_CIL/Period_4/3rd_try\BiGRUWithAttention_epoch_720.pth
✅ Updated: BiGRUWithAttention_epoch_818.pth | Acc: 0.9664 | Saved to: Class_Incremental_CL/Classif_Bi_Dir_GRU_Model/Trained_models/MSE_Distillation_CIL/Period_4/3rd_try\BiGRUWithAttention_epoch_818.pth
✅ Updated: BiGRUWithAttention_epoch_844.pth | Acc: 0.9665 | Saved to: Class_Incremental_CL/Classif_Bi_Dir_GRU_Model/Trained_models/MSE_Distillation_CIL/Period_4/3rd_try\BiGRUWithAttention_epoch_844.pth
✅ Updated: BiGRUWithAttention_epoch_877.pth | Acc: 0.9668 | Saved to: Class_Incremental_CL/Classif_Bi_Dir_GRU_Model/Trained_models/MSE_Distillation_CIL/Period_4/3rd_try\BiGRUWithAttention_epoch_877.pth
✅ Updated: BiGRUWithAttention_epoch_913.pth | Acc: 0.9665 | Saved to: Class_Incremental_CL/Classif_Bi_Dir_GRU_Model/Trained_models/MSE_Distillation_CIL/Period_4/3rd_try\BiGRUWithAttention_epoc

C:\Users\james\AppData\Local\Temp\ipykernel_81344\1933887850.py:66: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(path, map_location="cpu")


### Period 4 --> Training and saving in __*'4th_try'*__ (BiGRUWithAttention, num_layers = 4, alpha = 0.6) ---> Val acc = 95.28 %
### __Val-Class-Acc = {0: '90.13%', 1: '95.83%', 2: '88.59%', 3: '97.21%', 4: '98.88%'} %

In [41]:
"""
- 'trend': Categorized trend values based on the detected phases:
    - 0: No trend
    - 1: Moderate negative trend
    - 2: Very strong negative trend
    - 3: Moderate positive trend
    - 4: Very strong positive trend
"""
with contextlib.redirect_stdout(open(os.devnull, 'w')):
    X_train, y_train, X_val, y_val, X_test, y_test, Number_features = process_and_return_splits(
        with_indicators_file_path = list_period_files_full_path[3], # Change 
        downsampled_data_minutes = downsampled_data_minutes,
        exclude_columns = exclude_columns,
        lower_threshold = lower_threshold,
        upper_threshold = upper_threshold,
        reverse_steps = reverse_steps,
        sequence_length = sequence_length,
        sliding_interval = sliding_interval,
        trends_to_keep = {0, 1, 2, 3, 4}  # Default keeps all trends : {0, 1, 2, 3, 4}
        # trends_to_keep = {0, 1, 2, 3, 4}  # Default keeps all trends
    )

print(f"\nNumber_features = {Number_features}")

unique_classes = np.unique(y_val)
num_classes = len(unique_classes)
print(f"unique_classes = {unique_classes}")
print(f"num_classes = {num_classes}")



Number_features = 7
unique_classes = [0 1 2 3 4]
num_classes = 5


In [42]:
# Model parameters
stable_classes = [1, 2, 3] # <----------------------------<<< From period 2: I exclude class 0 because it will change in period 4
input_size = Number_features  # Number of features
hidden_size = 64  # Number of GRU units
output_size = num_classes # Must be dynamic, up to 5  # Number of trend classes (0, 15, 25, -15, -25)
num_layers = 4  # Number of GRU layers
dropout = 0.0
learning_rate = 0.0001
alpha = 0.6          # Weight for distillation loss
num_epochs= 2000 # Number of epochs/ go through entire data
batch_size= 64 # How many sequences passed at once to the model
model_name = 'BiGRUWithAttention' # Name of the model to use for saving
best_results = [] # Initialize this outside the training function or at the beginning of training
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# Define a global stop signal
stop_signal_file = os.path.normpath(os.path.join('Class_Incremental_CL', 'Classif_Bi_Dir_GRU_Model/stop_training.txt'))  # Create this file to stop training
model_saving_folder = os.path.normpath(os.path.join('Class_Incremental_CL', "Classif_Bi_Dir_GRU_Model/Trained_models/MSE_Distillation_CIL/Period_4/4th_try"))
ensure_folder(model_saving_folder)

# Instantiate the student model
student_model = BiGRUWithAttention(input_size, hidden_size, output_size, num_layers, dropout).to(device)

# Instantiate and load the teacher model
teacher_model = BiGRUWithAttention(input_size, hidden_size, output_size - 1, num_layers, dropout).to(device)
#-------------------------------------------------------------------------
best_overall = max(track_across_runs, key=lambda res: res['val_accuracy'])
best_model_path = best_overall['model_path']
#----------------------------------------------------------------------
# Initialize the list to store results across runs
track_across_runs = []
#----------------------------------------------------------------------
#-------------------------------------------------------------------------
teacher_checkpoint_path = os.path.normpath(best_model_path)
teacher_checkpoint = torch.load(teacher_checkpoint_path, map_location=device, weights_only=True)
# print(f"\n{teacher_checkpoint}\n")
teacher_model.load_state_dict(teacher_checkpoint['model_state_dict'])
del teacher_checkpoint
gc.collect()
print(f"Loaded teacher model from: \n\t{teacher_checkpoint_path}")
print(f"\n{teacher_model}\n")

# Define the loss function, optimizer and scheduler
criterion = nn.CrossEntropyLoss()
# Optimizer: only train parameters of the student model (LoRA parameters and any unfrozen ones).
optimizer = optim.Adam(student_model.parameters(), lr=learning_rate) # lr=0.00005
# optimizer = optim.Adam(class_gru_model.parameters(), lr=0.001, weight_decay=1e-5)
# scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', factor=0.9, patience=10)

train_and_validate_MSE(student_model, teacher_model, stable_classes, output_size, criterion, optimizer, 
                       X_train, y_train, X_val, y_val, scheduler, False, num_epochs, batch_size, 
                       alpha, model_saving_folder, model_name, stop_signal_file)

#----------------------------------------------------------------------
# Append only the best result (already at index 0)
track_across_runs.append(best_results[0])
#----------------------------------------------------------------------

for res in best_results:        
    print(f"Epoch {res['epoch']}/{num_epochs}, "
            f"Train Loss: {res['train_loss']:.4f}, " 
            f"Val Loss: {res['val_loss']:.4f}, "
            f"Val Accuracy: {res['val_accuracy'] * 100:.2f}%, "
            f"Model Path: {res['model_path']}")      
print(f"\nclass_gru_model (student_model): \n{student_model}")

print(f"\nunique_classes = {unique_classes}")
print(f"num_classes = {num_classes}")
# del unique_classes, num_classes
for var in ["X_train", "y_train", "X_val", "y_val", "X_test", "y_test", "Number_features", "unique_classes", "num_classes"]:
    if var in locals():
        del locals()[var]


Loaded teacher model from: 
	Class_Incremental_CL\Classif_Bi_Dir_GRU_Model\Trained_models\MSE_Distillation_CIL\Period_3\4th_try\BiGRUWithAttention_epoch_1705.pth

BiGRUWithAttention(
  (gru): GRU(7, 64, num_layers=4, batch_first=True, bidirectional=True)
  (attention_fc): Linear(in_features=128, out_features=128, bias=True)
  (fc): Linear(in_features=128, out_features=4, bias=True)
  (dropout): Dropout(p=0.0, inplace=False)
)

'train_and_validate' function started. 

Existing folder has been removed : Class_Incremental_CL\Classif_Bi_Dir_GRU_Model\Trained_models\MSE_Distillation_CIL\Period_4\4th_try

y_train:
<class 'torch.Tensor'>
torch.int64
torch.Size([3634, 1000])
X_train:
<class 'torch.Tensor'>
torch.float32
torch.Size([3634, 1000, 7])

y_val:
<class 'torch.Tensor'>
torch.int64
torch.Size([454, 1000])
X_val:
<class 'torch.Tensor'>
torch.float32
torch.Size([454, 1000, 7])

Dataset Lengths:
Train Dataset Length: 3634
Validation Dataset Length: 454

DataLoader Batch Sizes:
Number of B

### Period 4 --> Training and saving in __*'5th_try'*__ (BiGRUWithAttention, num_layers = 4, alpha = 0.1) ---> Val acc = 97.26 %
### __Val-Class-Acc = {0: '93.73%', 1: '97.08%', 2: '92.44%', 3: '98.25%', 4: '99.66%'} %__

In [47]:
"""
- 'trend': Categorized trend values based on the detected phases:
    - 0: No trend
    - 1: Moderate negative trend
    - 2: Very strong negative trend
    - 3: Moderate positive trend
    - 4: Very strong positive trend
"""
with contextlib.redirect_stdout(open(os.devnull, 'w')):
    X_train, y_train, X_val, y_val, X_test, y_test, Number_features = process_and_return_splits(
        with_indicators_file_path = list_period_files_full_path[3], # Change 
        downsampled_data_minutes = downsampled_data_minutes,
        exclude_columns = exclude_columns,
        lower_threshold = lower_threshold,
        upper_threshold = upper_threshold,
        reverse_steps = reverse_steps,
        sequence_length = sequence_length,
        sliding_interval = sliding_interval,
        trends_to_keep = {0, 1, 2, 3, 4}  # Default keeps all trends : {0, 1, 2, 3, 4}
        # trends_to_keep = {0, 1, 2, 3, 4}  # Default keeps all trends
    )

print(f"\nNumber_features = {Number_features}")

unique_classes = np.unique(y_val)
num_classes = len(unique_classes)
print(f"unique_classes = {unique_classes}")
print(f"num_classes = {num_classes}")



Number_features = 7
unique_classes = [0 1 2 3 4]
num_classes = 5


In [48]:
# Model parameters
stable_classes = [1, 2, 3] # <----------------------------<<< From period 2: I exclude class 0 because it will change in period 4
input_size = Number_features  # Number of features
hidden_size = 64  # Number of GRU units
output_size = num_classes # Must be dynamic, up to 5  # Number of trend classes (0, 15, 25, -15, -25)
num_layers = 4  # Number of GRU layers
dropout = 0.0
learning_rate = 0.0001
alpha = 0.1          # Weight for distillation loss
num_epochs= 2000 # Number of epochs/ go through entire data
batch_size= 64 # How many sequences passed at once to the model
model_name = 'BiGRUWithAttention' # Name of the model to use for saving
best_results = [] # Initialize this outside the training function or at the beginning of training
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# Define a global stop signal
stop_signal_file = os.path.normpath(os.path.join('Class_Incremental_CL', 'Classif_Bi_Dir_GRU_Model/stop_training.txt'))  # Create this file to stop training
model_saving_folder = os.path.normpath(os.path.join('Class_Incremental_CL', "Classif_Bi_Dir_GRU_Model/Trained_models/MSE_Distillation_CIL/Period_4/5th_try"))
ensure_folder(model_saving_folder)

# Instantiate the student model
student_model = BiGRUWithAttention(input_size, hidden_size, output_size, num_layers, dropout).to(device)

# Instantiate and load the teacher model
teacher_model = BiGRUWithAttention(input_size, hidden_size, output_size - 1, num_layers, dropout).to(device)
#-------------------------------------------------------------------------
best_overall = max(track_across_runs, key=lambda res: res['val_accuracy'])
best_model_path = best_overall['model_path']
#----------------------------------------------------------------------
# Initialize the list to store results across runs
track_across_runs = []
#----------------------------------------------------------------------
#-------------------------------------------------------------------------
teacher_checkpoint_path = os.path.normpath(best_model_path)
teacher_checkpoint = torch.load(teacher_checkpoint_path, map_location=device, weights_only=True)
# print(f"\n{teacher_checkpoint}\n")
teacher_model.load_state_dict(teacher_checkpoint['model_state_dict'])
del teacher_checkpoint
gc.collect()
print(f"Loaded teacher model from: \n\t{teacher_checkpoint_path}")
print(f"\n{teacher_model}\n")

# Define the loss function, optimizer and scheduler
criterion = nn.CrossEntropyLoss()
# Optimizer: only train parameters of the student model (LoRA parameters and any unfrozen ones).
optimizer = optim.Adam(student_model.parameters(), lr=learning_rate) # lr=0.00005
# optimizer = optim.Adam(class_gru_model.parameters(), lr=0.001, weight_decay=1e-5)
# scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', factor=0.9, patience=10)

train_and_validate_MSE(student_model, teacher_model, stable_classes, output_size, criterion, optimizer, 
                       X_train, y_train, X_val, y_val, scheduler, False, num_epochs, batch_size, 
                       alpha, model_saving_folder, model_name, stop_signal_file)

#----------------------------------------------------------------------
# Append only the best result (already at index 0)
track_across_runs.append(best_results[0])
#----------------------------------------------------------------------

for res in best_results:        
    print(f"Epoch {res['epoch']}/{num_epochs}, "
            f"Train Loss: {res['train_loss']:.4f}, " 
            f"Val Loss: {res['val_loss']:.4f}, "
            f"Val Accuracy: {res['val_accuracy'] * 100:.2f}%, "
            f"Model Path: {res['model_path']}")      
print(f"\nclass_gru_model (student_model): \n{student_model}")

print(f"\nunique_classes = {unique_classes}")
print(f"num_classes = {num_classes}")
# del unique_classes, num_classes
for var in ["X_train", "y_train", "X_val", "y_val", "X_test", "y_test", "Number_features", "unique_classes", "num_classes"]:
    if var in locals():
        del locals()[var]


Loaded teacher model from: 
	Class_Incremental_CL\Classif_Bi_Dir_GRU_Model\Trained_models\MSE_Distillation_CIL\Period_3\5th_try\BiGRUWithAttention_epoch_1413.pth

BiGRUWithAttention(
  (gru): GRU(7, 64, num_layers=4, batch_first=True, bidirectional=True)
  (attention_fc): Linear(in_features=128, out_features=128, bias=True)
  (fc): Linear(in_features=128, out_features=4, bias=True)
  (dropout): Dropout(p=0.0, inplace=False)
)

'train_and_validate' function started. 

Existing folder has been removed : Class_Incremental_CL\Classif_Bi_Dir_GRU_Model\Trained_models\MSE_Distillation_CIL\Period_4\5th_try

y_train:
<class 'torch.Tensor'>
torch.int64
torch.Size([3634, 1000])
X_train:
<class 'torch.Tensor'>
torch.float32
torch.Size([3634, 1000, 7])

y_val:
<class 'torch.Tensor'>
torch.int64
torch.Size([454, 1000])
X_val:
<class 'torch.Tensor'>
torch.float32
torch.Size([454, 1000, 7])

Dataset Lengths:
Train Dataset Length: 3634
Validation Dataset Length: 454

DataLoader Batch Sizes:
Number of B

### Period 4 --> Training and saving in __*'6th_try'*__ (BiGRUWithAttention, num_layers = 4, alpha = 0.2) ---> Val acc = 96.06 %
### __Val-Class-Acc = {0: '93.19%', 1: '96.31%', 2: '90.58%', 3: '97.08%', 4: '99.28%'} %__

In [20]:
"""
- 'trend': Categorized trend values based on the detected phases:
    - 0: No trend
    - 1: Moderate negative trend
    - 2: Very strong negative trend
    - 3: Moderate positive trend
    - 4: Very strong positive trend
"""
with contextlib.redirect_stdout(open(os.devnull, 'w')):
    X_train, y_train, X_val, y_val, X_test, y_test, Number_features = process_and_return_splits(
        with_indicators_file_path = list_period_files_full_path[3], # Change 
        downsampled_data_minutes = downsampled_data_minutes,
        exclude_columns = exclude_columns,
        lower_threshold = lower_threshold,
        upper_threshold = upper_threshold,
        reverse_steps = reverse_steps,
        sequence_length = sequence_length,
        sliding_interval = sliding_interval,
        trends_to_keep = {0, 1, 2, 3, 4}  # Default keeps all trends : {0, 1, 2, 3, 4}
        # trends_to_keep = {0, 1, 2, 3, 4}  # Default keeps all trends
    )

print(f"\nNumber_features = {Number_features}")

unique_classes = np.unique(y_val)
num_classes = len(unique_classes)
print(f"unique_classes = {unique_classes}")
print(f"num_classes = {num_classes}")



Number_features = 7
unique_classes = [0 1 2 3 4]
num_classes = 5


In [21]:
# Model parameters
stable_classes = [1, 2, 3] # <----------------------------<<< From period 2: I exclude class 0 because it will change in period 4
input_size = Number_features  # Number of features
hidden_size = 64  # Number of GRU units
output_size = num_classes # Must be dynamic, up to 5  # Number of trend classes (0, 15, 25, -15, -25)
num_layers = 4  # Number of GRU layers
dropout = 0.0
learning_rate = 0.0001
alpha = 0.2          # Weight for distillation loss
num_epochs= 2000 # Number of epochs/ go through entire data
batch_size= 64 # How many sequences passed at once to the model
model_name = 'BiGRUWithAttention' # Name of the model to use for saving
best_results = [] # Initialize this outside the training function or at the beginning of training
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# Define a global stop signal
stop_signal_file = os.path.normpath(os.path.join('Class_Incremental_CL', 'Classif_Bi_Dir_GRU_Model/stop_training.txt'))  # Create this file to stop training
model_saving_folder = os.path.normpath(os.path.join('Class_Incremental_CL', "Classif_Bi_Dir_GRU_Model/Trained_models/MSE_Distillation_CIL/Period_4/6th_try"))
ensure_folder(model_saving_folder)

# Instantiate the student model
student_model = BiGRUWithAttention(input_size, hidden_size, output_size, num_layers, dropout).to(device)

# Instantiate and load the teacher model
teacher_model = BiGRUWithAttention(input_size, hidden_size, output_size - 1, num_layers, dropout).to(device)
#-------------------------------------------------------------------------
best_overall = max(track_across_runs, key=lambda res: res['val_accuracy'])
best_model_path = best_overall['model_path']
#----------------------------------------------------------------------
# Initialize the list to store results across runs
track_across_runs = []
#----------------------------------------------------------------------
#-------------------------------------------------------------------------
teacher_checkpoint_path = os.path.normpath(best_model_path)
teacher_checkpoint = torch.load(teacher_checkpoint_path, map_location=device, weights_only=True)
# print(f"\n{teacher_checkpoint}\n")
teacher_model.load_state_dict(teacher_checkpoint['model_state_dict'])
del teacher_checkpoint
gc.collect()
print(f"Loaded teacher model from: \n\t{teacher_checkpoint_path}")
print(f"\n{teacher_model}\n")

# Define the loss function, optimizer and scheduler
criterion = nn.CrossEntropyLoss()
# Optimizer: only train parameters of the student model (LoRA parameters and any unfrozen ones).
optimizer = optim.Adam(student_model.parameters(), lr=learning_rate) # lr=0.00005
# optimizer = optim.Adam(class_gru_model.parameters(), lr=0.001, weight_decay=1e-5)
# scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', factor=0.9, patience=10)

train_and_validate_MSE(student_model, teacher_model, stable_classes, output_size, criterion, optimizer, 
                       X_train, y_train, X_val, y_val, scheduler, False, num_epochs, batch_size, 
                       alpha, model_saving_folder, model_name, stop_signal_file)

#----------------------------------------------------------------------
# Append only the best result (already at index 0)
track_across_runs.append(best_results[0])
#----------------------------------------------------------------------

for res in best_results:        
    print(f"Epoch {res['epoch']}/{num_epochs}, "
            f"Train Loss: {res['train_loss']:.4f}, " 
            f"Val Loss: {res['val_loss']:.4f}, "
            f"Val Accuracy: {res['val_accuracy'] * 100:.2f}%, "
            f"Model Path: {res['model_path']}")      
print(f"\nclass_gru_model (student_model): \n{student_model}")

print(f"\nunique_classes = {unique_classes}")
print(f"num_classes = {num_classes}")
# del unique_classes, num_classes
for var in ["X_train", "y_train", "X_val", "y_val", "X_test", "y_test", "Number_features", "unique_classes", "num_classes"]:
    if var in locals():
        del locals()[var]


Loaded teacher model from: 
	Class_Incremental_CL\Classif_Bi_Dir_GRU_Model\Trained_models\MSE_Distillation_CIL\Period_3\6th_try\BiGRUWithAttention_epoch_1884.pth

BiGRUWithAttention(
  (gru): GRU(7, 64, num_layers=4, batch_first=True, bidirectional=True)
  (attention_fc): Linear(in_features=128, out_features=128, bias=True)
  (fc): Linear(in_features=128, out_features=4, bias=True)
  (dropout): Dropout(p=0.0, inplace=False)
)

'train_and_validate' function started. 

Existing folder has been removed : Class_Incremental_CL\Classif_Bi_Dir_GRU_Model\Trained_models\MSE_Distillation_CIL\Period_4\6th_try

y_train:
<class 'torch.Tensor'>
torch.int64
torch.Size([3634, 1000])
X_train:
<class 'torch.Tensor'>
torch.float32
torch.Size([3634, 1000, 7])

y_val:
<class 'torch.Tensor'>
torch.int64
torch.Size([454, 1000])
X_val:
<class 'torch.Tensor'>
torch.float32
torch.Size([454, 1000, 7])

Dataset Lengths:
Train Dataset Length: 3634
Validation Dataset Length: 454

DataLoader Batch Sizes:
Number of B

In [ ]:
#-------------------------------------------------------------------------
best_overall = max(track_across_runs, key=lambda res: res['val_accuracy'])
best_model_path = best_overall['model_path']
#----------------------------------------------------------------------
# # Initialize the list to store results across runs
# track_across_runs = []
#----------------------------------------------------------------------
#-------------------------------------------------------------------------

print(f"best_model_path = {best_model_path}")

print(f"\ntrack_across_runs = \n{track_across_runs}")

---

## 📊 Summary: MSE Distillation CIL with BiGRUWithAttention  
### (`MSE_Distillation_CIL_Classif_Bi_Dir_GRU_Model.ipynb`)

| Period | Model & Config                                                                | Validation Accuracy | Class-wise Accuracy                                               |
|--------|-------------------------------------------------------------------------------|---------------------|-------------------------------------------------------------------|
| 1      | `BiGRUWithAttention`<br>(num_layers=4)<br>saved in `'1st_try'`               | **98.35%**          | {0: 98.63%, 1: 97.92%}                                            |
| 2      | `BiGRUWithAttention`<br>(num_layers=4, alpha=0.2)<br>saved in `'6th_try'`     | **98.04%**          | {0: 99.33%, 1: 97.66%, 2: 93.01%}                                |
| 3      | `BiGRUWithAttention`<br>(num_layers=4, alpha=0.1)<br>saved in `'5th_try'`     | **98.31%**          | {0: 95.21%, 1: 98.55%, 2: 97.44%, 3: 98.92%}                     |
| 4      | `BiGRUWithAttention`<br>(num_layers=4, alpha=0.3)<br>saved in `'3rd_try'`     | **96.68%**          | {0: 94.53%, 1: 96.85%, 2: 92.63%, 3: 98.14%, 4: 98.28%}          |

---
